Este script resulta en datsets crudos pero limpios, sin duplicados ni datos faltantes ni en la etiqueta de carbono ni en ninguna variable relevante como puede ser la ubicación o el año. No se criba por ningun criterio más allá de la complettitud de los datos. 

Vamos a extender la base de datos de IFN a toda españa
1. Descomprimir los archivos
2. Pasar a excel y unificarlos en un archivo por cada provincia (o dos: campo y sig)
3. Unificar en un archivo por cada inventario añadiendo la variable Provincia

In [1]:
import matplotlib
matplotlib.use("Agg")  # headless backend
# extrae recursivamente todos los archivos comprimidos presentes en los directorios IFN indicados
# admite .zip, .tar(.gz/.tgz), .gz suelto y .7z (py7zr o 7-Zip CLI)

from __future__ import annotations
import logging
import subprocess
import sys
from pathlib import Path
from typing import Iterable, List

import gzip
import shutil
import tarfile
import zipfile

try:
    import py7zr            # pip install py7zr
    _PY7ZR_AVAILABLE = True
except ImportError:
    _PY7ZR_AVAILABLE = False

###############################################################################
# configuración de logging
###############################################################################
LOG_FILE = (
    Path(__file__).with_suffix(".log")           # ejecución como script
    if "__file__" in globals()
    else Path.cwd() / "descompresion_ifn.log"    # ejecución interactiva (notebook)
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(LOG_FILE, encoding="utf-8")],
)
logger = logging.getLogger(__name__)

###############################################################################
# parámetros de entrada
###############################################################################
ROOT_DIRS: List[Path] = [
    Path("IFN_España/IFN4_sig"),
    Path("IFN_España/IFN4_campo"),
    Path("IFN_España/IFN3_sig"),
    Path("IFN_España/IFN3_campo"),
    Path("IFN_España/IFN2"),
]

SUPPORTED_EXTENSIONS = {".zip", ".tar", ".gz", ".tgz", ".tar.gz", ".7z"}

###############################################################################
# utilidades
###############################################################################
def find_archives(directories: Iterable[Path]) -> Iterable[Path]:
    """localiza archivos comprimidos en los directorios dados"""
    for directory in directories:
        if not directory.exists():
            logger.warning("la ruta %s no existe; se omite", directory)
            continue
        yield from (
            path
            for path in directory.rglob("*")
            if path.is_file() and any(path.name.lower().endswith(ext) for ext in SUPPORTED_EXTENSIONS)
        )


def extract_zip(src: Path, dst: Path) -> None:
    with zipfile.ZipFile(src) as zf:
        zf.extractall(dst)


def extract_tar(src: Path, dst: Path) -> None:
    mode = "r:gz" if src.suffixes[-2:] in [[".tar", ".gz"], [".tgz"]] else "r"
    with tarfile.open(src, mode) as tf:
        tf.extractall(dst)


def extract_gz(src: Path, dst: Path) -> None:
    with gzip.open(src, "rb") as f_in, open(dst / src.stem, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)


def extract_7z(src: Path, dst: Path) -> None:
    if _PY7ZR_AVAILABLE:
        with py7zr.SevenZipFile(src, mode="r") as archive:
            archive.extractall(path=dst)
    else:
        cmd = ["7z", "x", str(src), f"-o{dst}"]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError(f"7-Zip CLI error: {result.stderr.strip()}")


def decompress_file(archive_path: Path, overwrite: bool = False) -> None:
    destination = archive_path.parent / archive_path.stem
    if destination.exists() and not overwrite:
        logger.info("destino %s existente; extracción omitida", destination)
        return
    destination.mkdir(exist_ok=True)

    try:
        suffixes = "".join(archive_path.suffixes).lower()
        logger.info("extrayendo %s → %s", archive_path.name, destination)
        if suffixes.endswith(".zip"):
            extract_zip(archive_path, destination)
        elif suffixes.endswith((".tar.gz", ".tgz", ".tar")):
            extract_tar(archive_path, destination)
        elif suffixes.endswith(".gz"):
            extract_gz(archive_path, destination)
        elif suffixes.endswith(".7z"):
            extract_7z(archive_path, destination)
        else:
            logger.warning("formato no soportado: %s", archive_path)
            return
        logger.info("extracción completa de %s", archive_path.name)
    except Exception as exc:
        logger.error("error extrayendo %s: %s", archive_path, exc, exc_info=True)


###############################################################################
# ejecución
###############################################################################
def main() -> None:
    archives = list(find_archives(ROOT_DIRS))
    if not archives:
        logger.warning("no se encontraron archivos comprimidos en las rutas indicadas")
        return

    logger.info("se encontraron %d archivos para extraer", len(archives))
    for archive in archives:
        decompress_file(archive, overwrite=False)


if __name__ == "__main__":
    main()


2026-03-05 18:41:18,018 [INFO] se encontraron 228 archivos para extraer


2026-03-05 18:41:18,019 [INFO] destino IFN_España/IFN4_sig/sig_Málaga.accdb existente; extracción omitida


2026-03-05 18:41:18,020 [INFO] destino IFN_España/IFN4_sig/sig_Cádiz.accdb existente; extracción omitida


2026-03-05 18:41:18,020 [INFO] destino IFN_España/IFN4_sig/sig_valladolid_tcm30-536620 existente; extracción omitida


2026-03-05 18:41:18,020 [INFO] destino IFN_España/IFN4_sig/Sig_22_Huesca existente; extracción omitida


2026-03-05 18:41:18,020 [INFO] destino IFN_España/IFN4_sig/Sig_50_Zaragoza existente; extracción omitida


... (219 outputs left out)

2026-03-05 18:41:18,071 [INFO] destino IFN_España/IFN2/090471228013cc72_tcm30-278543 existente; extracción omitida


2026-03-05 18:41:18,072 [INFO] destino IFN_España/IFN2/090471228013cc54_tcm30-278528 existente; extracción omitida


2026-03-05 18:41:18,072 [INFO] destino IFN_España/IFN2/090471228013cbc5_tcm30-278515 existente; extracción omitida


2026-03-05 18:41:18,072 [INFO] destino IFN_España/IFN2/090471228013cbbd_tcm30-278511 existente; extracción omitida


IFN2

In [2]:
# fusiona los .dbf de cada provincia en IFN2 en un Excel ifn2_<código>.xlsx, quitando el sufijo numérico de los nombres de hoja

from __future__ import annotations
import re
from pathlib import Path
from collections import Counter
import pandas as pd
from dbfread import DBF

ROOT = Path("IFN_España/IFN2")  # carpeta IFN2

###########################################################################
# utilidades
###########################################################################
def detectar_codigo(dir_prov: Path) -> str | None:
    """busca DATEST##.dbf para obtener el código de provincia."""
    for dbf in dir_prov.rglob("DATEST*.[Dd][Bb][Ff]"):
        m = re.match(r"DATEST(\d{2})\.dbf$", dbf.name, flags=re.I)
        if m:
            return m.group(1)
    return None

def leer_dbf(ruta: Path) -> pd.DataFrame | None:
    """carga un .dbf a DataFrame; devuelve None si falla."""
    try:
        return pd.DataFrame(iter(DBF(ruta, load=True, encoding="latin1")))
    except Exception as exc:
        print(f"⚠️  error leyendo {ruta.name}: {exc}")
        return None

def hoja_desde_nombre(stem: str) -> str:
    """elimina un sufijo de hasta dos dígitos finales; limita a 31 car."""
    hoja = re.sub(r"\d{1,2}$", "", stem.upper())
    return hoja[:31] or stem[:31]

###########################################################################
# proceso por provincia
###########################################################################
for dir_prov in [d for d in ROOT.iterdir() if d.is_dir()]:
    codigo = detectar_codigo(dir_prov)
    if not codigo:
        print(f"⚠️  no se encontró DATEST##.dbf en {dir_prov}")
        continue

    destino = ROOT / f"ifn2_{codigo}.xlsx"
    if destino.exists():
        print(f"⏭  {destino.name} ya existe, saltando"); continue
    escritor = pd.ExcelWriter(destino, engine="xlsxwriter")

    nombres_duplicados: Counter[str] = Counter()
    for dbf_path in dir_prov.rglob("*.[Dd][Bb][Ff]"):
        df = leer_dbf(dbf_path)
        if df is None or df.empty:
            continue

        base = hoja_desde_nombre(dbf_path.stem)
        nombres_duplicados[base] += 1
        hoja = f"{base}_{nombres_duplicados[base]}" if nombres_duplicados[base] > 1 else base
        hoja = hoja[:31]  # seguridad extra

        df.to_excel(escritor, sheet_name=hoja, index=False)

    escritor.close()
    print(f"✅ creado {destino.name} en {destino.parent}")


2026-03-05 18:41:18,134 [INFO] NumExpr defaulting to 16 threads.


⏭  ifn2_25.xlsx ya existe, saltando
⏭  ifn2_50.xlsx ya existe, saltando
⏭  ifn2_49.xlsx ya existe, saltando
⏭  ifn2_48.xlsx ya existe, saltando
⏭  ifn2_07.xlsx ya existe, saltando
⏭  ifn2_19.xlsx ya existe, saltando
⏭  ifn2_34.xlsx ya existe, saltando
⏭  ifn2_10.xlsx ya existe, saltando
⏭  ifn2_12.xlsx ya existe, saltando
⏭  ifn2_13.xlsx ya existe, saltando
⏭  ifn2_30.xlsx ya existe, saltando
⏭  ifn2_06.xlsx ya existe, saltando
⏭  ifn2_36.xlsx ya existe, saltando
⏭  ifn2_14.xlsx ya existe, saltando
⏭  ifn2_43.xlsx ya existe, saltando
⏭  ifn2_24.xlsx ya existe, saltando
⏭  ifn2_23.xlsx ya existe, saltando
⏭  ifn2_21.xlsx ya existe, saltando
⏭  ifn2_27.xlsx ya existe, saltando
⏭  ifn2_02.xlsx ya existe, saltando
⏭  ifn2_41.xlsx ya existe, saltando
⏭  ifn2_03.xlsx ya existe, saltando
⏭  ifn2_01.xlsx ya existe, saltando
⏭  ifn2_28.xlsx ya existe, saltando
⏭  ifn2_42.xlsx ya existe, saltando
⏭  ifn2_46.xlsx ya existe, saltando
⏭  ifn2_45.xlsx ya existe, saltando
⏭  ifn2_09.xlsx ya existe, s

IFN3

In [3]:
# exporta cada ACCDB de IFN-3 (campo y sig) a Excel ifn3_<tipo>_<cod>.xlsx
import re, pandas as pd
from pathlib import Path

BASES = [
    (Path("IFN_España/IFN3_campo"), "campo",  r"IFN3P(\d{2})\.accdb$"),
    (Path("IFN_España/IFN3_sig"),   "sig",    r"SIG_(\d{2})\.accdb$"),
]

def tablas(accdb: Path):
    import subprocess
    from io import StringIO
    names = subprocess.run(
        ['mdb-tables', '-1', str(accdb)], capture_output=True, text=True
    ).stdout.strip().split('\n')
    result = {}
    for name in names:
        name = name.strip()
        if not name or name.upper().startswith('MSYS'):
            continue
        out = subprocess.run(
            ['mdb-export', str(accdb), name], capture_output=True, text=True
        ).stdout
        if out:
            try:
                result[name] = pd.read_csv(StringIO(out))
            except Exception:
                pass
    return result

for root, tipo, rx in BASES:
    rgx = re.compile(rx, re.I)
    for acc in root.rglob("*.accdb"):
        m = rgx.search(acc.name)
        if not m: continue
        cod, xlsx = m.group(1), root / f"ifn3_{tipo}_{m.group(1)}.xlsx"
        if xlsx.exists():
            print(f"⏭  {xlsx.name} ya existe, saltando"); continue
        with pd.ExcelWriter(xlsx, engine="xlsxwriter") as w:
            for n, df in tablas(acc).items():
                hoja = re.sub(r"\d{1,2}$", "", n)[:31] or n[:31]
                df.to_excel(w, sheet_name=hoja, index=False)
        print("✔", xlsx)


⏭  ifn3_campo_39.xlsx ya existe, saltando
⏭  ifn3_campo_10.xlsx ya existe, saltando
⏭  ifn3_campo_36.xlsx ya existe, saltando
⏭  ifn3_campo_43.xlsx ya existe, saltando
⏭  ifn3_campo_28.xlsx ya existe, saltando
⏭  ifn3_campo_21.xlsx ya existe, saltando
⏭  ifn3_campo_09.xlsx ya existe, saltando
⏭  ifn3_campo_30.xlsx ya existe, saltando
⏭  ifn3_campo_04.xlsx ya existe, saltando
⏭  ifn3_campo_48.xlsx ya existe, saltando
⏭  ifn3_campo_08.xlsx ya existe, saltando
⏭  ifn3_campo_42.xlsx ya existe, saltando
⏭  ifn3_campo_12.xlsx ya existe, saltando
⏭  ifn3_campo_15.xlsx ya existe, saltando
⏭  ifn3_campo_22.xlsx ya existe, saltando
⏭  ifn3_campo_06.xlsx ya existe, saltando
⏭  ifn3_campo_17.xlsx ya existe, saltando
⏭  ifn3_campo_33.xlsx ya existe, saltando
⏭  ifn3_campo_07.xlsx ya existe, saltando
⏭  ifn3_campo_27.xlsx ya existe, saltando
⏭  ifn3_campo_03.xlsx ya existe, saltando
⏭  ifn3_campo_35.xlsx ya existe, saltando
⏭  ifn3_campo_25.xlsx ya existe, saltando
⏭  ifn3_campo_40.xlsx ya existe, s

IFN4

In [4]:
# convierte cada base .accdb de IFN4 (campo y sig) en un Excel, con una hoja por cada tabla
# Lógica robusta: extrae código de provincia desde el nombre del fichero o directorio

from pathlib import Path
import re
import pandas as pd

# Mapeo nombre → código (se intenta primero con p{NN} en el nombre del fichero)
NOMBRE_A_COD = {
    "albacete": "02", "alicante": "03", "almeria": "04", "avila": "05",
    "badajoz": "06", "extremadura": "06", "baleares": "07", "barcelona": "08",
    "cataluna": "08", "burgos": "09", "caceres": "10", "cadiz": "11",
    "castellon": "12", "ciudadreal": "13", "cordoba": "14", "acoruna": "15",
    "coruna": "15", "cuenca": "16", "girona": "17", "granada": "18",
    "guadalajara": "19", "huelva": "21", "huesca": "22", "jaen": "23",
    "leon": "24", "lleida": "25", "rioja": "26", "lugo": "27", "madrid": "28",
    "malaga": "29", "murcia": "30", "navarra": "31", "ourense": "32",
    "orense": "32", "asturias": "33", "palencia": "34", "canarias": "35",
    "pontevedra": "36", "salamanca": "37", "cantabria": "39", "segovia": "40",
    "sevilla": "41", "soria": "42", "tarragona": "43", "teruel": "44",
    "toledo": "45", "valencia": "46", "valladolid": "47", "zamora": "49",
    "zaragoza": "50", "paisvasco": "01",
}

CARPETAS = [
    (Path("IFN_España/IFN4_campo"), "campo"),
    (Path("IFN_España/IFN4_sig"),   "sig"),
]

def codigo_provincia(accdb: Path) -> str | None:
    """Extrae código de provincia: busca p{NN} en el path, luego por nombre."""
    # Buscar patrón p{2 dígitos} en nombre del fichero o directorio padre
    for parte in [accdb.name, accdb.parent.name]:
        m = re.search(r"[Pp]([0-9]{2})(?:[^0-9]|$)", parte)
        if m:
            return m.group(1).zfill(2)
    # Fallback: buscar nombre de provincia en path completo
    path_lower = re.sub(r"[^a-z]", "", str(accdb).lower())
    for nombre, cod in sorted(NOMBRE_A_COD.items(), key=lambda x: -len(x[0])):
        if nombre in path_lower:
            return cod
    return None

def tablas_access(ruta: Path) -> dict:
    """Lee todas las tablas de una .accdb con mdb-export."""
    import subprocess
    from io import StringIO
    names = subprocess.run(
        ["mdb-tables", "-1", str(ruta)], capture_output=True, text=True
    ).stdout.strip().split("\n")
    result = {}
    for name in names:
        name = name.strip()
        if not name or name.upper().startswith("MSYS"):
            continue
        out = subprocess.run(
            ["mdb-export", str(ruta), name], capture_output=True, text=True
        ).stdout
        if out:
            try:
                result[name] = pd.read_csv(StringIO(out))
            except Exception:
                pass
    return result

def exportar(accdb: Path, destino: Path) -> None:
    tablas = tablas_access(accdb)
    if not tablas:
        print(f"⚠️  {accdb.name}: sin tablas útiles")
        return
    with pd.ExcelWriter(destino, engine="xlsxwriter") as xl:
        for nombre, df in tablas.items():
            hoja = re.sub(r"\d{1,2}$", "", nombre)[:31] or nombre[:31]
            df.to_excel(xl, sheet_name=hoja, index=False)
    print(f"✅ {destino.name}")

for carpeta, tipo in CARPETAS:
    for accdb in sorted(carpeta.rglob("*.accdb")):
        if accdb.is_dir():
            continue
        cod = codigo_provincia(accdb)
        if not cod:
            print(f"⚠️  Sin código: {accdb.relative_to(carpeta)}")
            continue
        destino = carpeta / f"ifn4_{tipo}_{cod}.xlsx"
        if destino.exists():
            print(f"⏭  {destino.name} ya existe, saltando")
            continue
        try:
            exportar(accdb, destino)
        except Exception as e:
            print(f"❌ {accdb.name}: {e}")


⏭  ifn4_campo_11.xlsx ya existe, saltando
⏭  ifn4_campo_14.xlsx ya existe, saltando
⏭  ifn4_campo_29.xlsx ya existe, saltando
⏭  ifn4_campo_23.xlsx ya existe, saltando
⏭  ifn4_campo_01.xlsx ya existe, saltando
⏭  ifn4_campo_05.xlsx ya existe, saltando
⏭  ifn4_campo_07.xlsx ya existe, saltando
⏭  ifn4_campo_09.xlsx ya existe, saltando
⏭  ifn4_campo_15.xlsx ya existe, saltando
⏭  ifn4_campo_24.xlsx ya existe, saltando
⏭  ifn4_campo_26.xlsx ya existe, saltando
⏭  ifn4_campo_27.xlsx ya existe, saltando
⏭  ifn4_campo_28.xlsx ya existe, saltando
⏭  ifn4_campo_30.xlsx ya existe, saltando
⏭  ifn4_campo_31.xlsx ya existe, saltando
⏭  ifn4_campo_32.xlsx ya existe, saltando
⏭  ifn4_campo_33.xlsx ya existe, saltando
⏭  ifn4_campo_34.xlsx ya existe, saltando
⏭  ifn4_campo_36.xlsx ya existe, saltando
⏭  ifn4_campo_37.xlsx ya existe, saltando
⏭  ifn4_campo_39.xlsx ya existe, saltando
⏭  ifn4_campo_40.xlsx ya existe, saltando
⏭  ifn4_campo_42.xlsx ya existe, saltando
⏭  ifn4_campo_47.xlsx ya existe, s

In [5]:
# Celda reemplazada: la lógica de IFN4 está completa en la celda anterior

arreglo las agrupaciones por CCAA

OJO ifn4 agrupa Pais Vasco (01 por 01,48,20), Canarias(35 por 35 y 38), Cataluna (8 por 8,17,25,43) y extremadura (6 por 6 y 10). Andalucia (almeria, cadiz, cordoba, granada, huelva, jaen, malaga, sevilla// 04,11,14,18,21,23,29,41) no aparece directamente. 

In [6]:
# Celda desactivada: los xlsx por provincia ya se generan en celda 8
print("celda 12 saltada")

celda 12 saltada


Ahora unificamos los archivos en dos para cada inventario: sig y campo 

IFN2

In [7]:
import pandas as pd
from pathlib import Path

# Directorio donde están los archivos .xlsx de IFN2
DIRECTORIO_IFN2 = Path("IFN_España/IFN2")

# Obtener todas las rutas de archivos .xlsx en el directorio
archivos_ifn2 = list(DIRECTORIO_IFN2.glob("ifn2_*.xlsx"))

# Definir las columnas esperadas para cada tipo de tabla/hoja
columnas_esperadas = {
    'DATEST': ['FOTO', 'DANOS1', 'ESTADA31', 'FRACCION2', 'ESPECIA21', 'PROVINCIA', 'PENDIEN1', 'USO2', 'COORDEX', 'ESPECIE1', 'EDADB3', 'ORIGEB3', 'CODVE', 'ESPECIB1', 'ESPECIE3', 'ESPECIA11', 'MEROSIVA1', 'ESPESOR', 'MASAB2', 'DISESPAC', 'MAXPEND2', 'HOJA', 'MAXPEND1', 'CODGR', 'ESPECIB2', 'OCUPA32', 'ESPECIE5', 'ESPECIA22', 'ESPECIE4', 'PENDIEN2', 'ESTADA11', 'FECHAVE', 'USO1', 'FRACCION1', 'ESTRATO', 'NUMPMA', 'ESPECIA12', 'MEROSIVA4', 'ESTADA22', 'VERIFICA', 'PROPIED2', 'EDADB2', 'ALTITUD1', 'ESPECIE6', 'ALTITUD2', 'DANOS2', 'NUMMAT', 'ANO', 'ESPECIB3', 'DANOS3', 'ORIGEB2', 'PREPSUEL2', 'VUELO', 'ESPECIE7', 'CORTAS', 'AREAPRO2', 'OCUPA12', 'MEROSIVA3', 'MASAB1', 'OCUPA22', 'ESTADA12', 'NUMTIP', 'AREAPRO1', 'OCUPA11', 'MUNICI', 'TRAVUELO2', 'CLASUELO', 'ESPECIE2', 'ORIENTA2', 'MEROSIVA2', 'ORIGEB1', 'TRAVUELO1', 'OCUPA31', 'ESPECIA31', 'ESTADA32', 'CLACOBER', 'COORDEY', 'PROPIED1', 'FECHGR', 'OCUPA21', 'COMPESPE', 'NUMPME', 'ESPECIA32', 'ESTADA21', 'EDADB1', 'ORIENTA1', 'PREPSUEL1', 'ESTADILLO', 'CUBIERTA', 'MASAB3'],
    'IIFL00BD': ['VSC', 'PAE1', 'AREAB', 'PAE2', 'DN1', 'DISTANCIA', 'PROVINCIA', 'ESPECIE', 'CALIDAD', 'DN2', 'VCC', 'PAE3', 'TIPO', 'ESTADILLO', 'ARBOL', 'IAVC', 'FORMA', 'RUMBO', 'VLE', 'HTOTAL'],
    'IIFL03BD': ['NVSC', 'NIAVC', 'CCDIAMETR', 'CESTADILLO', 'NARBOLES', 'NVLE', 'NAREAB', 'CPROVINCIA', 'NVCC', 'CESPECIE'],
    'IIFL04BD': ['VSC', 'AREAB', 'ESPECIE', 'ESTRATO', 'IAVC', 'VCC', 'COD_LINEA', 'CANT_P_MA', 'VLE', 'LIT_LINEA'],
    'MATORR': ['PROVINCIA', 'ALTUMED', 'ESTADILLO', 'ESPECIE', 'NUMORDEN', 'FRACCAB'],
    'PIESMA': ['PARAMESP', 'PROVINCIA', 'ESTADILLO', 'ARBOL', 'ESPECIE', 'NUMORDEN', 'DIAMETRO1', 'FORMA', 'DISTANCI', 'ALTURA', 'RUMBO', 'DIAMETRO2', 'CALIDAD', 'TIPO'],
    'PIESME': ['PROVINCIA', 'ALTUMED', 'REGENA', 'ESTADILLO', 'ESPECIE', 'NUMORDEN', 'NUMERO'],
    'TIPOSX': ['PROVINCIA', 'CRECIMI2', 'ESTADILLO', 'CORTEZA1', 'ALTURFF', 'CORTEZA2', 'NUMORDEN', 'DIAMCOPA2', 'CRECIMI1', 'ALTUPV', 'DIAMFF', 'DIAMETR4', 'TIPO', 'DIAMCOPA1']
}

# Función para unificar todas las hojas de un archivo en un único archivo CSV por tipo
def unificar_tablas(archivos: list[Path], columnas_esperadas: dict) -> None:
    dataframes_unificados = {key: [] for key in columnas_esperadas}

    for archivo in archivos:
        xls = pd.ExcelFile(archivo)
        provincia = archivo.stem.split('_')[1]  # Extrae el código de provincia (por ejemplo '01' de 'ifn2_01')

        for hoja in columnas_esperadas.keys():
            if hoja in xls.sheet_names:
                df = pd.read_excel(xls, sheet_name=hoja)

                # Añadir columna con el código de provincia
                df['Provincia'] = provincia

                # Añadir columnas faltantes como NaN
                for col in columnas_esperadas[hoja]:
                    if col not in df.columns:
                        df[col] = pd.NA

                # Reordenar columnas esperadas + Provincia al final
                columnas_finales = columnas_esperadas[hoja] + ['Provincia']
                df = df[[col for col in columnas_finales if col in df.columns]]

                # Añadir al diccionario de DataFrames
                dataframes_unificados[hoja].append(df)

    # Guardar los DataFrames unificados como CSV
    for hoja, dataframes in dataframes_unificados.items():
        if dataframes:
            df_unificado = pd.concat(dataframes, ignore_index=True)
            archivo_csv = Path(DIRECTORIO_IFN2) / f'ifn2_{hoja}.csv'
            df_unificado.to_csv(archivo_csv, index=False)
            print(f"Guardado {archivo_csv}")

# Ejecutar la unificación
unificar_tablas(archivos_ifn2, columnas_esperadas)


Guardado IFN_España/IFN2/ifn2_DATEST.csv


Guardado IFN_España/IFN2/ifn2_IIFL00BD.csv


Guardado IFN_España/IFN2/ifn2_IIFL03BD.csv


Guardado IFN_España/IFN2/ifn2_IIFL04BD.csv


Guardado IFN_España/IFN2/ifn2_MATORR.csv


Guardado IFN_España/IFN2/ifn2_PIESMA.csv


Guardado IFN_España/IFN2/ifn2_PIESME.csv


Guardado IFN_España/IFN2/ifn2_TIPOSX.csv


IFN3

In [8]:
import pandas as pd
from pathlib import Path

# Directorio donde están los archivos .xlsx de IFN3_campo
DIRECTORIO_IFN3_CAMPO = Path("IFN_España/IFN3_campo")

# Obtener todas las rutas de archivos .xlsx en el directorio
archivos_ifn3 = list(DIRECTORIO_IFN3_CAMPO.glob("ifn3_campo_*.xlsx"))

# Diccionario con las columnas esperadas para cada tabla
columnas_esperadas = {
    'Errores de conversión': ['Descripción del error', 'Nombre de objeto', 'Tipo de objeto'],
    'PCDatosMap': ['JefeEq', 'CoorX', 'CoorY', 'Hoja50', 'Nivel5', 'Nivel4', 'FccArb', 'Nivel6', 'Nivel2', 'DisEsp', 'Subclase', 'Estadillo', 'Clase', 'ComEsp', 'Nivel1', 'Provincia', 'FccTot', 'INE', 'Estado', 'Nivel3'],
    'PCDetTabla': ['CodTabla', 'DenValor', 'Valor'],
    'PCEspMapa': ['Estado', 'Ocupa', 'Especie', 'Estadillo', 'PosEsp'],
    'PCEspParc': ['AltPer', 'FPMasa', 'Edad', 'OrgMasa2', 'Ocupa', 'Barrena1', 'Subclase', 'Fiabil', 'Estadillo', 'PosEsp', 'FInfor', 'Barrena2', 'Barrena3', 'TratMasa', 'Estado', 'OrgMasa1', 'Especie', 'Cla'],
    'PCMatorral': ['Subclase', 'Especie', 'Estadillo', 'Fcc', 'Cla', 'Hm'],
    'PCMayores': ['Calidad.1', 'Dn2', 'Agente', 'Diametro2', 'Ht', 'Forma.1', 'ParamEsp', 'Estadillo', 'Diametro1', 'Calidad', 'NumOrden', 'Compara', 'Import', 'Rumbo',
                   'Especie', 'ParEsp', 'Distanci', 'Elemento', 'Forma', 'Altura'],
    'PCNueEsp': ['Especie', 'Estadillo', 'Cla', 'Subclase'],
    'PCParcelas': ['CarFoto2', 'JefeEq', 'MaxPend2', 'MejVue1', 'FccArb', 'EspCMue', 'Tecnico', 'RumboF2', 'Rocosid', 'Provincia', 'Orienta1', 'TipSuelo2', 'Obser', 'Vuelo2', 'ConFoto2', 'MErosiva', 'MejSue2', 'PresReg', 'Localiza', 'Tiempo', 'NumFoto1', 'EfecReg', 'Foto2', 'Estadillo', 'Nivel1', 'ConFoto1', 'FechaPh', 'FechaFin', 'INE', 'Acceso', 'PhSuelo', 'Estado', 'Vuelo1', 'Pasada2', 'HoraPh', 'Levanta', 'TipSuelo3', 'Textura', 'Pasada1', 'Coory', 'Foto1', 'MatOrg', 'CarFoto1', 'MejVue2', 'DisEsp', 'Tipo', 'Orienta2', 'DistFoto', 'CortaReg', 'Nivel3', 'Equipo', 'Cla', 'CoorX', 'Ano', 'TipSuelo1', 'Nivel2', 'Subclase', 'ComEsp', 'MejSue1', 'FccTot', 'FechaIni', 'HoraFin', 'ModComb', 'MaxPend1', 'HoraIni', 'Resid', 'RumboF1', 'NumFoto2'],
    'PCRegenera': ['Tipo', 'CatDes', 'Densidad', 'NumPies', 'Subclase', 'Especie', 'Estadillo', 'Cla', 'Hm'],
    'PCTablaEsp': ['HtMin', 'DnHtMin', 'DifDnMax', 'HmRegMax', 'DnMin', 'DnMax', 'DnHtMax', 'HtMax', 'Especie', 'LForma']
}

# Función para unificar las tablas de todos los archivos
def unificar_tablas(archivos: list[Path], columnas_esperadas: dict):
    tablas_unificadas = {tabla: [] for tabla in columnas_esperadas.keys()}
    
    for archivo in archivos:
        xls = pd.ExcelFile(archivo)
        provincia = archivo.stem.split('_')[2]  # Extraer el código de provincia (e.g. '01' de 'ifn3_campo_01.xlsx')
        
        for hoja in xls.sheet_names:
            if hoja in columnas_esperadas:
                df = pd.read_excel(xls, sheet_name=hoja)
                
                # Añadir columna de provincia
                df['Provincia'] = provincia
                
                # Añadir columnas faltantes como NaN
                for col in columnas_esperadas[hoja]:
                    if col not in df.columns:
                        df[col] = pd.NA

                # Reordenar columnas: esperadas + 'Provincia' al final (solo si no está ya)
                columnas_finales = columnas_esperadas[hoja] + (['Provincia'] if 'Provincia' not in columnas_esperadas[hoja] else [])
                df = df[[col for col in columnas_finales if col in df.columns]]

                tablas_unificadas[hoja].append(df)
    
    # Guardar como CSV
    for tabla, dataframes in tablas_unificadas.items():
        if dataframes:
            df_unificada = pd.concat(dataframes, ignore_index=True)
            archivo_csv = DIRECTORIO_IFN3_CAMPO / f"ifn3_campo_{tabla}.csv"
            df_unificada.to_csv(archivo_csv, index=False)
            print(f"Guardado {archivo_csv}")

# Ejecutar la unificación
unificar_tablas(archivos_ifn3, columnas_esperadas)


Guardado IFN_España/IFN3_campo/ifn3_campo_Errores de conversión.csv


Guardado IFN_España/IFN3_campo/ifn3_campo_PCDatosMap.csv
Guardado IFN_España/IFN3_campo/ifn3_campo_PCDetTabla.csv
Guardado IFN_España/IFN3_campo/ifn3_campo_PCEspMapa.csv


Guardado IFN_España/IFN3_campo/ifn3_campo_PCEspParc.csv


Guardado IFN_España/IFN3_campo/ifn3_campo_PCMatorral.csv


Guardado IFN_España/IFN3_campo/ifn3_campo_PCMayores.csv


Guardado IFN_España/IFN3_campo/ifn3_campo_PCNueEsp.csv


Guardado IFN_España/IFN3_campo/ifn3_campo_PCParcelas.csv


Guardado IFN_España/IFN3_campo/ifn3_campo_PCRegenera.csv
Guardado IFN_España/IFN3_campo/ifn3_campo_PCTablaEsp.csv


In [9]:
import pandas as pd
from pathlib import Path

# Directorio donde están los archivos .xlsx de IFN3_campo
DIRECTORIO_IFN3_CAMPO = Path("IFN_España/IFN3_sig")

# Obtener todas las rutas de archivos .xlsx en el directorio
archivos_ifn3 = list(DIRECTORIO_IFN3_CAMPO.glob("ifn3_sig_*.xlsx"))

# Diccionario con las columnas esperadas para cada tabla
columnas_esperadas = {
'CambioEspecie':['EspCambiada', 'EspOriginal', 'NumPies'],

'CambioEspecieReg':['EspCambiada', 'EspOriginal', 'NumPies'],

'Errores de conversión':['Nombre de objeto', 'Descripción del error', 'Tipo de objeto'],
'EspDominante':['IdEspDom', 'EspDom'],

'Estratos':['IdEspDom', 'Estrato', 'NPar', 'Superficie'],

'Estratos_exs':['VSC', 'NPies', 'Estrato', 'Especie', 'ABas', 'IAVC', 'VCC', 'CD', 'VLE'],

'Mayores_exs':['Forma', 'Elemento', 'VSC', 'Ht', 'OrdenIf2', 'Fac', 'Especie', 'Calidad', 'Subclase', 'Rumbo', 'VCC', 'Dn1', 'Agente', 'nArbol', 'Estadillo', 'Dn2', 'CD', 
              'OrdenIf3', 'Estrato', 'Import', 'G', 'Cla', 'IAVC', 'EspecieOriginal', 'ParEsp', 'VLE', 'Distanci'],

'Parcelas_exs':['VSC', 'NPies', 'Estrato', 'Especie', 'Cla', 'ABas', 'IAVC', 'VCC', 'Estadillo', 'Subclase', 'CD', 'VLE'],

'Parcpoly':['Desclasada4', 'Desclasada7', 'Desclasada', 'Desclasada5', 'Desclasada8', 'Desclasada3', 'SubClase', 'Orientacion', 'Desclasada9', 'Altitud', 'Desclasada6', 
              'Desclasada1', 'Parcela', 'Estrato', 'Linea', 'Cla', 'Polygon', 'Pendiente', 'Desclasada2'],

'Poligon':['N5', 'TIPOESTR', 'SUPERFICIE', 'POLYGON', 'FccP', 'NVal', 'CLIMA', 'A revisar', 'N2', 'LimProv2', 'ManEstrato', 'UG', 'Esp1', 'LimProv1', 'O3', 'N6', 'E2', 'N1',
               'Esp3', 'Fcc', 'N3', 'E3', 'ManNivel', 'Fcctot', 'Estrato', 'Linea', 'E1', 'O1', 'N4', 'Esp2', 'O2', 'Encontrado'],

'TarifasIFN':['PARAMET', 'RR', 'CPARAM', 'CC', 'ESTR', 'BB', 'APLIC', 'SISTGEOG', 'PROVINCIA', 'AA', 'ESPECIE', 'TT', 'DD', 'NUM', 'CALIDAD', 'PP', 'SS', 'R2', 'KK', 'OBSERV', 
              'FORMA', 'QQ', 'R2A', 'CUADRO', 'MODELO', 'PARESP', 'CLASE'],
              
'ParcPoly':['Estrato', 'Linea', 'Cla', 'Polygon', 'Subclase', 'Altitud', 'Desclasada', 'Pendiente', 'Orientacion', 'Parcela'],
}

# Función para unificar las tablas de todos los archivos
def unificar_tablas(archivos: list[Path], columnas_esperadas: dict):
    tablas_unificadas = {tabla: [] for tabla in columnas_esperadas.keys()}
    
    for archivo in archivos:
        xls = pd.ExcelFile(archivo)
        provincia = archivo.stem.split('_')[2]  # Extraer el código de provincia (e.g. '01' de 'ifn3_campo_01.xlsx')
        
        for hoja in xls.sheet_names:
            if hoja in columnas_esperadas:
                df = pd.read_excel(xls, sheet_name=hoja)
                
                # Añadir columna de provincia
                df['Provincia'] = provincia
                
                # Añadir columnas faltantes como NaN
                for col in columnas_esperadas[hoja]:
                    if col not in df.columns:
                        df[col] = pd.NA

                # Reordenar columnas: esperadas + 'Provincia' al final (solo si no está ya)
                columnas_finales = columnas_esperadas[hoja] + (['Provincia'] if 'Provincia' not in columnas_esperadas[hoja] else [])
                df = df[[col for col in columnas_finales if col in df.columns]]

                tablas_unificadas[hoja].append(df)
    
    # Guardar como CSV
    for tabla, dataframes in tablas_unificadas.items():
        if dataframes:
            df_unificada = pd.concat(dataframes, ignore_index=True)
            archivo_csv = DIRECTORIO_IFN3_CAMPO / f"ifn3_sig_{tabla}.csv"
            df_unificada.to_csv(archivo_csv, index=False)
            print(f"Guardado {archivo_csv}")

# Ejecutar la unificación
unificar_tablas(archivos_ifn3, columnas_esperadas)


Guardado IFN_España/IFN3_sig/ifn3_sig_CambioEspecie.csv
Guardado IFN_España/IFN3_sig/ifn3_sig_CambioEspecieReg.csv
Guardado IFN_España/IFN3_sig/ifn3_sig_Errores de conversión.csv
Guardado IFN_España/IFN3_sig/ifn3_sig_EspDominante.csv
Guardado IFN_España/IFN3_sig/ifn3_sig_Estratos.csv
Guardado IFN_España/IFN3_sig/ifn3_sig_Estratos_exs.csv


Guardado IFN_España/IFN3_sig/ifn3_sig_Mayores_exs.csv


Guardado IFN_España/IFN3_sig/ifn3_sig_Parcelas_exs.csv
Guardado IFN_España/IFN3_sig/ifn3_sig_Parcpoly.csv


Guardado IFN_España/IFN3_sig/ifn3_sig_Poligon.csv


Guardado IFN_España/IFN3_sig/ifn3_sig_TarifasIFN.csv
Guardado IFN_España/IFN3_sig/ifn3_sig_ParcPoly.csv


IFN4

In [10]:
import pandas as pd
from pathlib import Path

# Directorio donde están los archivos .xlsx de IFN4_campo
DIRECTORIO_IFN4_CAMPO = Path("IFN_España/IFN4_campo")

# Obtener todas las rutas de archivos .xlsx en el directorio
archivos_ifn4_campo = list(DIRECTORIO_IFN4_CAMPO.glob("ifn4_campo_*.xlsx"))

# Diccionario con las columnas esperadas para cada tabla
columnas_esperadas = {
'PCDatosMap':[ 'Provincia','Estadillo', 'CoorX', 'CoorY', 'Huso', 'Hoja50', 'Tesela','Poligon','Clase','Subclase','Nivel1', 'Nivel2', 'Nivel3','FccArb',  
              'FccTot','DisEsp', 'ComEsp', 'TipEstr'],

'PCEspMapa':['Provincia', 'Estadillo','Especie', 'Cla', 'Subclase', 'Estado', 'Ocupa',  'PosEsp'],

'PCEspParc':['Provincia', 'Estadillo','Especie', 'Cla', 'Subclase','PosEsp', 'Estado', 'Ocupa', 'TratMasa', 'FPMasa', 'FInfor','Edad', 'Fiabil', 'OrgMasa1', 'OrgMasa2'],

'PCMatorral':['Provincia', 'Estadillo','Especie', 'Cla', 'Subclase','Elemento', 'Import', 'Agente', 'Fcc', 'Hm'],

'PCMayores':['Provincia', 'Estadillo','Especie', 'Cla', 'Subclase','nArbol','OrdenIf3', 'OrdenIf2','Rumbo', 'Distanci',  'Especie','Ht', 'Dn1','Dn2','Forma', 'Elemento', 
             'Calidad', 'Hcopa', 'Agente', 'Import',  'ParEsp'],

'PCNueEsp':['Provincia', 'Estadillo','Especie', 'Cla', 'Subclase'],

'PCParcelas':['Provincia', 'Estadillo','Especie', 'Cla', 'Subclase','Nivel1', 'Nivel2', 'Nivel3','Rocosid', 'MaxPend2', 'Pasada2', 'TipSuelo3', 'Acceso', 'Levanta', 'Foto1', 'HoraFin', 'MatOrg', 'MejSue2', 'EfecReg', 'Vuelo1', 'Pasada1', 'ComEsp', 
              'ModComb', 'HoraIni', 'MejVue1', 'TipSuelo2', 'Foto2', 'FechaFin', 'MejVue2', 'Vuelo2', 'PresReg', 'FccArb', 'Obser', 'Orienta1', 
              'FechaIni', 'Ano', 'MejSue1', 'CortaReg', 'MaxPend1', 'Orienta2', 'EspCMue', 'MErosiva', 'PhSuelo', 'TipSuelo1', 'Textura', 'DisEsp',
              'Localiza', 'FccTot', 'Tipo'],

'PCRegenera':['Especie', 'Cla', 'Estadillo', 'Subclase', 'Provincia', 'NumPies', 'CatDes', 'Tipo', 'Densidad', 'Hm'],

'PCAlcornoque':['CantRamas', 'Cla', 'Muestra', 'LongRamas', 'LongTronco', 'Calibre2', 'Dn1', 'nArbol', 'Estadillo', 'Subclase', 'Dn2', 'Provincia', 'Calibre1', 'OrdenIf3', 'DRed', 
                'Rumbo', 'OrdenIf4']

    
}

# Función para unificar las tablas de todos los archivos
def unificar_tablas(archivos: list[Path], columnas_esperadas: dict):
    tablas_unificadas = {tabla: [] for tabla in columnas_esperadas.keys()}
    
    for archivo in archivos:
        xls = pd.ExcelFile(archivo)
        provincia = archivo.stem.split('_')[2]  # Extraer el código de provincia (e.g. '01' de 'ifn4_campo_01.xlsx')
        
        for hoja in xls.sheet_names:
            if hoja in columnas_esperadas:
                df = pd.read_excel(xls, sheet_name=hoja)
                
                # Añadir columna de provincia
                df['Provincia'] = provincia
                
                # Añadir columnas faltantes como NaN
                for col in columnas_esperadas[hoja]:
                    if col not in df.columns:
                        df[col] = pd.NA

                # Reordenar columnas: esperadas + 'Provincia' al final (solo si no está ya)
                columnas_finales = columnas_esperadas[hoja] + (['Provincia'] if 'Provincia' not in columnas_esperadas[hoja] else [])
                df = df[[col for col in columnas_finales if col in df.columns]]

                tablas_unificadas[hoja].append(df)
    
    # Guardar como CSV
    for tabla, dataframes in tablas_unificadas.items():
        if dataframes:
            df_unificada = pd.concat(dataframes, ignore_index=True)
            archivo_csv = DIRECTORIO_IFN4_CAMPO / f"ifn4_campo_{tabla}.csv"
            df_unificada.to_csv(archivo_csv, index=False)
            print(f"Guardado {archivo_csv}")

# Ejecutar la unificación
unificar_tablas(archivos_ifn4_campo, columnas_esperadas)

Guardado IFN_España/IFN4_campo/ifn4_campo_PCDatosMap.csv
Guardado IFN_España/IFN4_campo/ifn4_campo_PCEspMapa.csv


Guardado IFN_España/IFN4_campo/ifn4_campo_PCEspParc.csv


Guardado IFN_España/IFN4_campo/ifn4_campo_PCMatorral.csv


Guardado IFN_España/IFN4_campo/ifn4_campo_PCMayores.csv
Guardado IFN_España/IFN4_campo/ifn4_campo_PCNueEsp.csv


Guardado IFN_España/IFN4_campo/ifn4_campo_PCParcelas.csv


Guardado IFN_España/IFN4_campo/ifn4_campo_PCRegenera.csv
Guardado IFN_España/IFN4_campo/ifn4_campo_PCAlcornoque.csv


In [11]:
import pandas as pd
from pathlib import Path

# Directorio donde están los archivos .xlsx de IFN3_campo
DIRECTORIO_IFN4_sig = Path("IFN_España/IFN4_sig")

# Obtener todas las rutas de archivos .xlsx en el directorio
archivos_ifn4_sig = list(DIRECTORIO_IFN4_sig.glob("ifn4_sig_*.xlsx"))

# Diccionario con las columnas esperadas para cada tabla
columnas_esperadas = {
   'CambioEspecie': ['Proceso', 'EspCambiada', 'EspOriginal', 'NumPies'],

'EspDominante': ['Proceso', 'EspDom', 'IdEspDom'],

'Estratos': ['NParBu', 'Estrato', 'E', 'Superficie', 'Proceso', 'FCC (%)', 'Cód FAN', 'IdEspDom', 'NParEn', 'NPar'],

'Estratos_exs': ['Proceso','Estrato','Especie', 'CD','NPies', 'ABas', 'IAVC', 'VCC', 'VSC', 'VLE'],

'Mayores_exs': ['Provincia', 'Estadillo','Especie','CD', 'Cla', 'Subclase','nArbol','OrdenIf3', 'OrdenIf4', 
                'Forma', 'Elemento', 'VSC', 'Proceso', 'Ht', 'Fac',  'Calidad',  'Rumbo',  'VCC', 'Dn1', 'Agente', 
               'Dn2',  'Estrato', 'Import', 'G', 'IAVC', 'EspecieOriginal', 'ParEsp', 'VLE', 'Distanci', 'CA', 'CR'],

'Parcelas_exs': ['Provincia', 'Estadillo','Especie','CD', 'Cla', 'Subclase','Estrato','Proceso', 'VSC', 'NPies',   'ABas', 'IAVC', 'VCC',  'VLE', 'CA', 'CR'],

'Parcpoly': ['Estrato', 'Cla', 'ForArb_Polygon', 'Altitud Absoluta', 'SubClase', 'Polygon', 'Exposicion', 'Proceso', 'Provincia', 'Pendiente Rango', 'Pendiente Absoluta', 
              'Altitud Rango', 'ForArb', 'Orientacion', 'Parcela'],

'Poligon': ['N5', 'TIPOESTR', 'TipoBosque', 'ForMat', 'ForSubpiso', 'HmMat', 'SUPERFICIE', 'Proceso', 'POLYGON', 'Contenido', 'ModComb', 'Provincia', 'N2', 'RegBio', 'Esp1', 
              'FccHer', 'O3', 'FccArb', 'N6', 'E2', 'N1', 'Esp3', 'N3', 'E3', 'CompEsp', 'ForHer', 'Estrato', 'E1', 'ETes', 'O1', 'ForMat2', 'ForArb', 'Esp2', 'FccMat', 'O2'],

'TarifasIFN': ['PARAMET', 'RR', 'CPARAM', 'CC', 'ESTR', 'BB', 'APLIC', 'SISTGEOG', 'PROVINCIA', 'AA', 'ESPECIE', 'TT', 'DD', 'NUM', 'CALIDAD', 'PP', 'SS', 'R2', 'KK', 'FORMA', 'QQ',
               'R2A', 'MODELO', 'PARESP', 'PROCESO', 'CLASE'],

'CambioEspecieReg': ['Proceso', 'EspCambiada', 'EspOriginal', 'NumPies'],

'Tarifas_IFN': ['PARAMET', 'RR', 'CPARAM', 'CC', 'ESTR', 'BB', 'APLIC', 'SISTGEOG', 'PROVINCIA', 'AA', 'ESPECIE', 'TT', 'DD', 'NUM', 'CALIDAD', 'PP', 'SS', 'R2', 'KK', 'FORMA', 'QQ',
                 'R2A', 'MODELO', 'PARESP', 'PROCESO', 'CLASE']

}

# Función para unificar las tablas de todos los archivos
def unificar_tablas(archivos: list[Path], columnas_esperadas: dict):
    tablas_unificadas = {tabla: [] for tabla in columnas_esperadas.keys()}
    
    for archivo in archivos:
        xls = pd.ExcelFile(archivo)
        provincia = archivo.stem.split('_')[2]  # Extraer el código de provincia (e.g. '01' de 'ifn4_sig_01.xlsx')
        
        for hoja in xls.sheet_names:
            if hoja in columnas_esperadas:
                df = pd.read_excel(xls, sheet_name=hoja)
                
                # Añadir columna de provincia
                df['Provincia'] = provincia
                
                # Añadir columnas faltantes como NaN
                for col in columnas_esperadas[hoja]:
                    if col not in df.columns:
                        df[col] = pd.NA

                # Reordenar columnas: esperadas + 'Provincia' al final (solo si no está ya)
                columnas_finales = columnas_esperadas[hoja] + (['Provincia'] if 'Provincia' not in columnas_esperadas[hoja] else [])
                df = df[[col for col in columnas_finales if col in df.columns]]

                tablas_unificadas[hoja].append(df)
    
    # Guardar como CSV
    for tabla, dataframes in tablas_unificadas.items():
        if dataframes:
            df_unificada = pd.concat(dataframes, ignore_index=True)
            archivo_csv = DIRECTORIO_IFN4_sig / f"ifn4_sig_{tabla}.csv"
            df_unificada.to_csv(archivo_csv, index=False)
            print(f"Guardado {archivo_csv}")

# Ejecutar la unificación
unificar_tablas(archivos_ifn4_sig, columnas_esperadas)

Guardado IFN_España/IFN4_sig/ifn4_sig_CambioEspecie.csv
Guardado IFN_España/IFN4_sig/ifn4_sig_EspDominante.csv
Guardado IFN_España/IFN4_sig/ifn4_sig_Estratos.csv
Guardado IFN_España/IFN4_sig/ifn4_sig_Estratos_exs.csv


Guardado IFN_España/IFN4_sig/ifn4_sig_Mayores_exs.csv


Guardado IFN_España/IFN4_sig/ifn4_sig_Parcelas_exs.csv
Guardado IFN_España/IFN4_sig/ifn4_sig_Parcpoly.csv


Guardado IFN_España/IFN4_sig/ifn4_sig_Poligon.csv
Guardado IFN_España/IFN4_sig/ifn4_sig_TarifasIFN.csv
Guardado IFN_España/IFN4_sig/ifn4_sig_CambioEspecieReg.csv
Guardado IFN_España/IFN4_sig/ifn4_sig_Tarifas_IFN.csv


retoco porque se me ha desdoblado tarifas

In [12]:
import pandas as pd
from pathlib import Path

# Definir rutas
ruta_base = Path("IFN_España/IFN4_sig")
archivo_origen = ruta_base / "ifn4_sig_Tarifas_IFN.csv"
archivo_destino = ruta_base / "ifn4_sig_TarifasIFN.csv"

# Leer los dos CSV
df_origen = pd.read_csv(archivo_origen)
df_destino = pd.read_csv(archivo_destino)

# Unir las entradas, asumiendo que no hay solapamiento (si las hay y se quiere eliminar duplicados, se puede hacer)
df_combinado = pd.concat([df_destino, df_origen], ignore_index=True)

# Guardar el archivo combinado sobrescribiendo el archivo de destino
df_combinado.to_csv(archivo_destino, index=False)

print(f"Archivo actualizado y guardado en: {archivo_destino}")


Archivo actualizado y guardado en: IFN_España/IFN4_sig/ifn4_sig_TarifasIFN.csv


In [13]:
import pandas as pd
from pathlib import Path

# Definir rutas
ruta_base = Path("IFN_España/IFN3_sig")
archivo_origen = ruta_base / "ifn3_sig_CambioEspecieReg.csv"
archivo_destino = ruta_base / "ifn3_sig_CambioEspecie.csv"

# Leer los dos CSV
df_origen = pd.read_csv(archivo_origen)
df_destino = pd.read_csv(archivo_destino)

# Unir las entradas, asumiendo que no hay solapamiento (si las hay y se quiere eliminar duplicados, se puede hacer)
df_combinado = pd.concat([df_destino, df_origen], ignore_index=True)

# Guardar el archivo combinado sobrescribiendo el archivo de destino
df_combinado.to_csv(archivo_destino, index=False)

print(f"Archivo actualizado y guardado en: {archivo_destino}")

Archivo actualizado y guardado en: IFN_España/IFN3_sig/ifn3_sig_CambioEspecie.csv


In [14]:
import pandas as pd
from pathlib import Path

# Definir rutas
ruta_base = Path("IFN_España/IFN4_sig")
archivo_origen = ruta_base / "ifn4_sig_CambioEspecieReg.csv"
archivo_destino = ruta_base / "ifn4_sig_CambioEspecie.csv"

# Leer los dos CSV
df_origen = pd.read_csv(archivo_origen)
df_destino = pd.read_csv(archivo_destino)

# Unir las entradas, asumiendo que no hay solapamiento (si las hay y se quiere eliminar duplicados, se puede hacer)
df_combinado = pd.concat([df_destino, df_origen], ignore_index=True)

# Guardar el archivo combinado sobrescribiendo el archivo de destino
df_combinado.to_csv(archivo_destino, index=False)

print(f"Archivo actualizado y guardado en: {archivo_destino}")

Archivo actualizado y guardado en: IFN_España/IFN4_sig/ifn4_sig_CambioEspecie.csv


Ahora comenzamos a limpiar las tablas. Quiero unas mismas variables en cada inventario y una misma estructura parcela, parcela_especie, parcela_especie_cd y pies_mayores. Voy a empezar por eliminar lo que no voy a usar y renombrar las variables. 

IFN3

In [15]:
#para leer las opciones usamos este codigo 
import pandas as pd
from pathlib import Path

# Ruta al directorio de los CSV
directorio = Path("IFN_España/IFN4_sig")

# Crear diccionario de columnas por archivo
columnas_por_csv = {}

# Recorrer todos los archivos CSV del directorio
for archivo in directorio.glob("*.csv"):
    try:
        df = pd.read_csv(archivo, nrows=0)  # Solo leer cabecera
        columnas_por_csv[archivo.name] = df.columns.tolist()
    except Exception as e:
        print(f"Error al procesar {archivo.name}: {e}")

# Imprimir el diccionario resultante
print(columnas_por_csv)


{'ifn4_sig_Mayores_exs.csv': ['Provincia', 'Estadillo', 'Especie', 'CD', 'Cla', 'Subclase', 'nArbol', 'OrdenIf3', 'OrdenIf4', 'Forma', 'Elemento', 'VSC', 'Proceso', 'Ht', 'Fac', 'Calidad', 'Rumbo', 'VCC', 'Dn1', 'Agente', 'Dn2', 'Estrato', 'Import', 'G', 'IAVC', 'EspecieOriginal', 'ParEsp', 'VLE', 'Distanci', 'CA', 'CR'], 'ifn4_sig_EspDominante.csv': ['Proceso', 'EspDom', 'IdEspDom', 'Provincia'], 'ifn4_sig_Parcelas_exs.csv': ['Provincia', 'Estadillo', 'Especie', 'CD', 'Cla', 'Subclase', 'Estrato', 'Proceso', 'VSC', 'NPies', 'ABas', 'IAVC', 'VCC', 'VLE', 'CA', 'CR'], 'ifn4_sig_CambioEspecieReg.csv': ['Proceso', 'EspCambiada', 'EspOriginal', 'NumPies', 'Provincia'], 'ifn4_sig_Estratos_exs.csv': ['Proceso', 'Estrato', 'Especie', 'CD', 'NPies', 'ABas', 'IAVC', 'VCC', 'VSC', 'VLE', 'Provincia'], 'ifn4_sig_Parcpoly.csv': ['Estrato', 'Cla', 'ForArb_Polygon', 'Altitud Absoluta', 'SubClase', 'Polygon', 'Exposicion', 'Proceso', 'Provincia', 'Pendiente Rango', 'Pendiente Absoluta', 'Altitud Rang

In [16]:
import pandas as pd
from pathlib import Path

# Definir las rutas de los directorios
directorio_ifn3_campo = Path("IFN_España/IFN3_campo")
directorio_ifn3_sig = Path("IFN_España/IFN3_sig")

# Diccionarios con las columnas que queremos conservar
columnas_ifn3_campo = { 
    'ifn3_campo_PCDatosMap.csv': ['Provincia','Estadillo', 'INE', 'CoorX', 'CoorY', 'Hoja50'],
    'ifn3_campo_PCEspParc.csv': ['Provincia','Estadillo', 'Especie','FPMasa', 'Edad', 'Ocupa', 'PosEsp', 'TratMasa', 'Estado', 'OrgMasa1'], 
    'ifn3_campo_PCParcelas.csv': ['Provincia','Estadillo','Clase', 'Subclase','Ano', 'TipSuelo1','TipSuelo2', 'TipSuelo3', 'Nivel1','Nivel2',
                                  'Rocosid','Orienta1','Orienta2','MaxPend1','MaxPend2','MErosiva', 'PhSuelo', 'Textura', 'MatOrg','PresReg',
                                  'ModComb','EspCMue','DisEsp', 'ComEsp', 'FccArb', 'FccTot', 'Estado'],
    'ifn3_campo_PCRegenera.csv': ['Provincia','Estadillo', 'Especie','Tipo', 'CatDes', 'Densidad', 'NumPies', 'Hm'],
    'ifn3_campo_PCTablaEsp.csv': ['Provincia', 'Especie','HtMin', 'DnHtMin', 'DifDnMax', 'HmRegMax', 'DnMin', 'DnMax', 'DnHtMax', 'HtMax','LForma']
}

columnas_ifn3_sig = {
    'ifn3_sig_CambioEspecie.csv': ['Provincia','EspCambiada', 'EspOriginal', 'NumPies'], 
    'ifn3_sig_Mayores_exs.csv': ['Provincia','Estadillo', 'Especie','EspecieOriginal','Estrato','nArbol','OrdenIf3','OrdenIf2', 'Rumbo', 'Distanci',   
                                 'CD', 'Ht', 'Dn1','Dn2','G', 'IAVC', 'VCC','VSC',  'VLE',], 
    'ifn3_sig_Parcelas_exs.csv': ['Provincia','Estadillo', 'Especie', 'Estrato', 'CD', 'VSC', 'NPies', 'ABas', 'IAVC', 'VCC', 'VLE',], 
}

# Función para procesar y guardar archivos CSV con solo las columnas deseadas
def conservar_columnas(directorio, columnas_dict):
    for archivo, columnas in columnas_dict.items():
        archivo_path = directorio / archivo
        try:
            # Leer el archivo CSV
            df = pd.read_csv(archivo_path)
            
            # Conservar solo las columnas deseadas
            columnas_a_conservar = [col for col in columnas if col in df.columns]
            df_conservado = df[columnas_a_conservar]
            
            # Guardar el archivo con las columnas seleccionadas (sobrescribiendo el archivo original)
            df_conservado.to_csv(archivo_path, index=False)
            print(f"Archivo guardado: {archivo_path}")
        except Exception as e:
            print(f"Error al procesar {archivo}: {e}")

# Procesar los archivos de IFN3_campo
conservar_columnas(directorio_ifn3_campo, columnas_ifn3_campo)

# Procesar los archivos de IFN3_sig
conservar_columnas(directorio_ifn3_sig, columnas_ifn3_sig)


Archivo guardado: IFN_España/IFN3_campo/ifn3_campo_PCDatosMap.csv


/tmp/ipykernel_2004326/616527105.py:32: DtypeWarning: Columns (0: Hoja50, 1: Nivel5, 2: Nivel4, 3: FccArb, 4: Nivel6, 5: Nivel2, 6: Subclase, 7: FccTot, 8: Nivel3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo_path)
/tmp/ipykernel_2004326/616527105.py:32: DtypeWarning: Columns (0: OrgMasa2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo_path)


Archivo guardado: IFN_España/IFN3_campo/ifn3_campo_PCEspParc.csv


/tmp/ipykernel_2004326/616527105.py:32: DtypeWarning: Columns (0: MejVue1, 1: RumboF2, 2: Vuelo2, 3: MejSue2, 4: INE, 5: Acceso, 6: MejVue2, 7: DistFoto, 8: CortaReg, 9: Nivel3, 10: Ano, 11: Nivel2, 12: MejSue1, 13: Resid, 14: RumboF1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo_path)


Archivo guardado: IFN_España/IFN3_campo/ifn3_campo_PCParcelas.csv


/tmp/ipykernel_2004326/616527105.py:32: DtypeWarning: Columns (0: Especie) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo_path)


Archivo guardado: IFN_España/IFN3_campo/ifn3_campo_PCRegenera.csv
Archivo guardado: IFN_España/IFN3_campo/ifn3_campo_PCTablaEsp.csv
Archivo guardado: IFN_España/IFN3_sig/ifn3_sig_CambioEspecie.csv


Archivo guardado: IFN_España/IFN3_sig/ifn3_sig_Mayores_exs.csv


Archivo guardado: IFN_España/IFN3_sig/ifn3_sig_Parcelas_exs.csv


IFN4

In [17]:
import pandas as pd
from pathlib import Path

# Definir las rutas de los directorios
directorio_ifn4_campo = Path("IFN_España/IFN4_campo")
directorio_ifn4_sig = Path("IFN_España/IFN4_sig")

# Diccionarios con las columnas que queremos conservar
columnas_ifn4_campo={

    'ifn4_campo_PCDatosMap.csv': ['Provincia', 'Estadillo', 'CoorX', 'CoorY', 'Huso', 'Hoja50', 'Tesela'],                   
    'ifn4_campo_PCEspParc.csv': ['Provincia','Estadillo', 'Especie','FPMasa', 'Edad', 'Ocupa', 'PosEsp', 'TratMasa', 'Estado', 'OrgMasa1'], 
    'ifn4_campo_PCParcelas.csv': ['Provincia','Estadillo','Clase', 'Subclase','Ano', 'TipSuelo1','TipSuelo2', 'TipSuelo3', 'Nivel1','Nivel2',
                                  'Rocosid','Orienta1','Orienta2','MaxPend1','MaxPend2','MErosiva', 'PhSuelo', 'Textura', 'MatOrg','PresReg',
                                  'ModComb','EspCMue','DisEsp', 'ComEsp', 'FccArb', 'FccTot', 'Estado'],                              
    'ifn4_campo_PCRegenera.csv': ['Provincia','Estadillo', 'Especie','Tipo', 'NumPies','CatDes', 'Densidad', 'Hm']
    }

columnas_ifn4_sig= {

    'ifn4_sig_CambioEspecie.csv': ['Provincia','EspCambiada', 'EspOriginal', 'NumPies'], 
    'ifn4_sig_Mayores_exs.csv': ['Provincia','Estadillo', 'Especie','EspecieOriginal','Estrato','nArbol','OrdenIf3','OrdenIf2', 'Rumbo', 'Distanci',   
                                 'CD', 'Ht', 'Dn1','Dn2','G', 'IAVC', 'VCC','VSC',  'VLE', 'CA', 'CR'],                                 
    'ifn4_sig_Parcelas_exs.csv': ['Provincia','Estadillo', 'Especie', 'Estrato', 'CD', 'VSC', 'NPies', 'ABas', 'IAVC', 'VCC', 'VLE', 'CA', 'CR'],     
    }

# Función para procesar y guardar archivos CSV con solo las columnas deseadas
def conservar_columnas(directorio, columnas_dict):
    for archivo, columnas in columnas_dict.items():
        archivo_path = directorio / archivo
        try:
            # Leer el archivo CSV
            df = pd.read_csv(archivo_path)
            
            # Conservar solo las columnas deseadas
            columnas_a_conservar = [col for col in columnas if col in df.columns]
            df_conservado = df[columnas_a_conservar]
            
            # Guardar el archivo con las columnas seleccionadas (sobrescribiendo el archivo original)
            df_conservado.to_csv(archivo_path, index=False)
            print(f"Archivo guardado: {archivo_path}")
        except Exception as e:
            print(f"Error al procesar {archivo}: {e}")

# Procesar los archivos de IFN3_campo
conservar_columnas(directorio_ifn4_campo, columnas_ifn4_campo)

# Procesar los archivos de IFN3_sig
conservar_columnas(directorio_ifn4_sig, columnas_ifn4_sig)

Archivo guardado: IFN_España/IFN4_campo/ifn4_campo_PCDatosMap.csv


/tmp/ipykernel_2004326/3391485664.py:33: DtypeWarning: Columns (0: Hoja50) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo_path)


Archivo guardado: IFN_España/IFN4_campo/ifn4_campo_PCEspParc.csv


/tmp/ipykernel_2004326/3391485664.py:33: DtypeWarning: Columns (0: Vuelo2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo_path)


Archivo guardado: IFN_España/IFN4_campo/ifn4_campo_PCParcelas.csv


Archivo guardado: IFN_España/IFN4_campo/ifn4_campo_PCRegenera.csv
Archivo guardado: IFN_España/IFN4_sig/ifn4_sig_CambioEspecie.csv


Archivo guardado: IFN_España/IFN4_sig/ifn4_sig_Mayores_exs.csv


Archivo guardado: IFN_España/IFN4_sig/ifn4_sig_Parcelas_exs.csv


IFN2

ifn2 va a ser mas dificil porque cambian los nombres de las etiquetas, debemos repasar con mucho cuidado y estraer la informacion que queremos. 

In [18]:
import pandas as pd
from pathlib import Path

# Definir las rutas de los directorios
directorio_ifn2 = Path("IFN_España/IFN2")

# Diccionarios con las columnas que queremos conservar
columnas_ifn2={

    'ifn2_DATEST.csv': ['PROVINCIA',	'ESTADILLO',	'ESTRATO', 'COORDEX',	'COORDEY',	'ANO',	'MUNICI',	'FRACCION1',	'FRACCION2', 
                   'ESPECIA12',	'OCUPA12',	'ESTADA12',
                   'ESPECIA22',	'OCUPA22',	'ESTADA22',
                   'ESPECIA32',	'OCUPA32',	'ESTADA32',
                   'ESPECIB1',	'MASAB1',	'ORIGEB1',	'EDADB1',
                   'ESPECIB2',	'MASAB2',	'ORIGEB2',	'EDADB2',
                   'ESPECIB3',	'MASAB3',	'ORIGEB3',	'EDADB3',
                   'CLASUELO',	'ESPESOR',	'CLACOBER',	'CUBIERTA',	'MEROSIVA1', 'DISESPAC',	'COMPESPE', 'ORIENTA1',	'ORIENTA2',	'MAXPEND1',	'MAXPEND2'
                   ],

    'ifn2_IIFL00BD.csv': ['PROVINCIA',	'ESTADILLO', 'ARBOL',	'RUMBO',	'DISTANCIA',	'ESPECIE', 'DN1',	'DN2',	'HTOTAL',	'AREAB', 'VCC',	'VSC',	'IAVC',	'VLE'
                     ], 

    'ifn2_IIFL03BD.csv': ['CPROVINCIA',	'CESTADILLO',	'CESPECIE',	'CCDIAMETR',	'NARBOLES',	'NVCC',	'NVSC', 'NVLE',	'NIAVC',	'NAREAB'],             

    'ifn2_PIESME.csv': ['PROVINCIA',	'ESTADILLO',	'NUMORDEN',	'ESPECIE',	'NUMERO',	'ALTUMED',	'REGENA'
                   ]
    }


# Función para procesar y guardar archivos CSV con solo las columnas deseadas
def conservar_columnas(directorio, columnas_dict):
    for archivo, columnas in columnas_dict.items():
        archivo_path = directorio / archivo
        try:
            # Leer el archivo CSV
            df = pd.read_csv(archivo_path)
            
            # Conservar solo las columnas deseadas
            columnas_a_conservar = [col for col in columnas if col in df.columns]
            df_conservado = df[columnas_a_conservar]
            
            # Guardar el archivo con las columnas seleccionadas (sobrescribiendo el archivo original)
            df_conservado.to_csv(archivo_path, index=False)
            print(f"Archivo guardado: {archivo_path}")
        except Exception as e:
            print(f"Error al procesar {archivo}: {e}")

# Procesar los archivos de IFN2
conservar_columnas(directorio_ifn2, columnas_ifn2)


/tmp/ipykernel_2004326/345528305.py:36: DtypeWarning: Columns (0: COORDEX, 1: ESPECIA11, 2: MAXPEND2, 3: MAXPEND1, 4: FRACCION1, 5: MEROSIVA4, 6: PROPIED2, 7: ALTITUD1, 8: AREAPRO2, 9: AREAPRO1, 10: OCUPA31, 11: COORDEY, 12: PROPIED1, 13: CUBIERTA) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo_path)


Archivo guardado: IFN_España/IFN2/ifn2_DATEST.csv


/tmp/ipykernel_2004326/345528305.py:36: DtypeWarning: Columns (0: PAE3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo_path)


Archivo guardado: IFN_España/IFN2/ifn2_IIFL00BD.csv


Archivo guardado: IFN_España/IFN2/ifn2_IIFL03BD.csv


Archivo guardado: IFN_España/IFN2/ifn2_PIESME.csv


renombro algunas columnas aunque no es definitivo 

In [19]:
import pandas as pd
from pathlib import Path

# Mapeo de columnas
columnas_mapeo = {
    'PROVINCIA': 'Provincia',
    'ESTADILLO': 'Estadillo',
    'ESTRATO': 'Estrato',
    'COORDEX': 'CoorX',
    'COORDEY': 'CoorY',
    'ANO': 'Ano',
    'ESPECIE':'Especie',
    'MUNICI':'INE',
    'ESPECIA12': 'Especie1',
    'OCUPA12': 'Ocupa1',
    'ESTADA12': 'Estado1',
    'ESPECIA22': 'Especie2',
    'OCUPA22': 'Ocupa2',
    'ESTADA22': 'Estado2',
    'ESPECIA32': 'Especie3',
    'OCUPA32': 'Ocupa3',
    'ESTADA32': 'Estado3',
    'MEROSIVA1': 'MErosiva',
    'DISESPAC': 'DisEsp',
    'COMPESPE': 'ComEsp',
    'ARBOL': 'OrdenIf2',
    'RUMBO': 'Rumbo',
    'DISTANCIA': 'Distanci',
    'DN1': 'Dn1',
    'DN2': 'Dn2',
    'HTOTAL': 'Ht',
    'AREAB': 'G',
    'CPROVINCIA': 'Provincia',
    'CESTADILLO': 'Estadillo',
    'CESPECIE': 'Especie',
    'CCDIAMETR': 'CD',
    'NARBOLES': 'NPies',
    'NAREAB': 'ABas',
    'NVCC': 'VCC',
    'NVSC': 'VSC',
    'NVLE': 'VLE',
    'NIAVC': 'IAVC'
}

# Directorio donde están los archivos .csv de IFN2
directorio_ifn2 = Path('IFN_España/IFN2')

# Lista de archivos en el directorio
archivos_ifn2 = list(directorio_ifn2.glob('*.csv'))

# Procesar cada archivo
for archivo in archivos_ifn2:
    # Leer el archivo CSV
    df = pd.read_csv(archivo)
    
    # Renombrar las columnas según el mapeo
    df.rename(columns=columnas_mapeo, inplace=True)
    
    # Guardar el DataFrame con las columnas renombradas
    df.to_csv(archivo, index=False)
    
    print(f"Archivo procesado y guardado: {archivo.name}")


Archivo procesado y guardado: ifn2_IIFL04BD.csv


Archivo procesado y guardado: ifn2_PIESME.csv


Archivo procesado y guardado: ifn2_IIFL03BD.csv


Archivo procesado y guardado: ifn2_IIFL00BD.csv


/tmp/ipykernel_2004326/3312655958.py:54: DtypeWarning: Columns (0: PARAMESP, 1: DISTANCI, 2: ALTURA) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo)


Archivo procesado y guardado: ifn2_PIESMA.csv


/tmp/ipykernel_2004326/3312655958.py:54: DtypeWarning: Columns (0: ALTURFF, 1: DIAMCOPA2, 2: ALTUPV, 3: DIAMCOPA1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo)


Archivo procesado y guardado: ifn2_TIPOSX.csv
Archivo procesado y guardado: ifn2_parcela_y_especie.csv


Archivo procesado y guardado: ifn2_MATORR.csv


/tmp/ipykernel_2004326/3312655958.py:54: DtypeWarning: Columns (0: COORDEX, 1: COORDEY, 2: FRACCION1, 3: CUBIERTA, 4: MAXPEND1, 5: MAXPEND2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo)


Archivo procesado y guardado: ifn2_DATEST.csv


Arreglos en el Huso

In [20]:
import pandas as pd

# Cargar el archivo CSV
archivo_path = 'IFN_España/IFN4_campo/ifn4_campo_PCDatosMap.csv'
df = pd.read_csv(archivo_path)

# Filtrar las parcelas de la provincia de León (Provincia == 24)
df_leon = df[df['Provincia'] == 24]

# Cambiar el valor de 'Huso' a 29 para las parcelas de León
df_leon['Huso'] = 30

# Reemplazar las filas de León con las modificadas en el DataFrame original
df.update(df_leon)

# Guardar el archivo editado
df.to_csv(archivo_path, index=False)

print(f"Archivo guardado en: {archivo_path}")

Archivo guardado en: IFN_España/IFN4_campo/ifn4_campo_PCDatosMap.csv


In [21]:
import pandas as pd
import numpy as np

# Cargar el archivo CSV
archivo = 'IFN_España/IFN4_campo/ifn4_campo_PCDatosMap.csv'
df = pd.read_csv(archivo)

# Ver el estado inicial
print("=== ESTADO INICIAL ===")
print(f"Número total de filas: {len(df)}")
print("Valores únicos de 'Huso':")
print(df['Huso'].unique())
print(f"\nConteo de valores de 'Huso':")
print(df['Huso'].value_counts(dropna=False))

# Identificar filas con problemas en coordenadas
filas_coorx_problemas = (df['CoorX'] == 0) | df['CoorX'].isna()
filas_coory_problemas = (df['CoorY'] == 0) | df['CoorY'].isna()
filas_coordenadas_problemas = filas_coorx_problemas | filas_coory_problemas

print(f"\n=== ANÁLISIS DE COORDENADAS PROBLEMÁTICAS ===")
print(f"Filas con CoorX = 0 o NaN: {filas_coorx_problemas.sum()}")
print(f"Filas con CoorY = 0 o NaN: {filas_coory_problemas.sum()}")
print(f"Filas con problemas en coordenadas (CoorX o CoorY): {filas_coordenadas_problemas.sum()}")

if filas_coordenadas_problemas.sum() > 0 and filas_coordenadas_problemas.sum() <= 10:
    print("\nEjemplos de filas que serán eliminadas:")
    print(df[filas_coordenadas_problemas][['Provincia', 'Estadillo', 'CoorX', 'CoorY', 'Huso']])
elif filas_coordenadas_problemas.sum() > 10:
    print("\nPrimeros 10 ejemplos de filas que serán eliminadas:")
    print(df[filas_coordenadas_problemas][['Provincia', 'Estadillo', 'CoorX', 'CoorY', 'Huso']].head(10))

# Eliminar las filas con coordenadas problemáticas
df_limpio = df[~filas_coordenadas_problemas].copy()
print(f"\nFilas eliminadas: {filas_coordenadas_problemas.sum()}")
print(f"Número de filas después de eliminar: {len(df_limpio)}")

# Ahora proceder con el relleno de valores faltantes para Huso
# Solo convertir 0 a NaN en Huso (las coordenadas ya están limpias)
df_limpio['Huso'] = df_limpio['Huso'].replace(0, np.nan)

print(f"\n=== ANÁLISIS DE HUSO DESPUÉS DE LIMPIAR COORDENADAS ===")
print(f"Valores faltantes en Huso (0 y NaN): {df_limpio['Huso'].isna().sum()}")
print("Valores únicos de 'Huso' después de eliminar coordenadas problemáticas:")
print(df_limpio['Huso'].unique())

# Función para obtener la moda (valor más frecuente) excluyendo NaN
def obtener_moda(serie):
    serie_limpia = serie.dropna()
    if len(serie_limpia) > 0:
        return serie_limpia.mode().iloc[0]
    else:
        return np.nan

# Rellenar valores faltantes específicos de Huso
df_limpio.loc[df_limpio['Provincia'] == 15, 'Huso'] = 29  # Rellenar Huso=28 para Provincia=15
df_limpio.loc[df_limpio['Provincia'] == 31, 'Huso'] = 30  # Rellenar Huso=30 para Provincia=31

# Rellenar el resto de los valores faltantes con la moda de la provincia
df_limpio['Huso'] = df_limpio.groupby('Provincia')['Huso'].transform(
    lambda x: x.fillna(obtener_moda(x))
)

# Verificar el resultado final
print(f"\n=== RESULTADO FINAL ===")
print("Valores únicos de 'Huso' después de la limpieza completa:")
print(df_limpio['Huso'].unique())
print(f"\nConteo final de valores de 'Huso':")
print(df_limpio['Huso'].value_counts(dropna=False))
print(f"Valores faltantes restantes en Huso: {df_limpio['Huso'].isna().sum()}")

# Verificar calidad de las coordenadas finales
print(f"\n=== VERIFICACIÓN DE COORDENADAS FINALES ===")
print(f"Rango CoorX: {df_limpio['CoorX'].min():.1f} a {df_limpio['CoorX'].max():.1f}")
print(f"Rango CoorY: {df_limpio['CoorY'].min():.1f} a {df_limpio['CoorY'].max():.1f}")
print(f"Coordenadas con valor 0: {((df_limpio['CoorX'] == 0) | (df_limpio['CoorY'] == 0)).sum()}")

# Resumen del proceso de limpieza
print(f"\n=== RESUMEN DEL PROCESO ===")
print(f"Filas originales: {len(df)}")
print(f"Filas eliminadas (coordenadas 0 o NaN): {filas_coordenadas_problemas.sum()}")
print(f"Filas finales: {len(df_limpio)}")
print(f"Valores de Huso rellenados: {df_limpio['Huso'].isna().sum()}")

# Guardar el dataset limpio (opcional)
df_limpio.to_csv( 'IFN_España/IFN4_campo/ifn4_campo_PCDatosMap.csv', index=False)

print(f"\nDataset limpio guardado como 'ifn4_campo_PCDatosMap.csv'")

=== ESTADO INICIAL ===
Número total de filas: 57918
Valores únicos de 'Huso':
[30.  0. 29. 28. nan 31.]

Conteo de valores de 'Huso':
Huso
30.0    34458
29.0     8470
31.0     6710
0.0      6297
28.0     1971
NaN        12
Name: count, dtype: int64

=== ANÁLISIS DE COORDENADAS PROBLEMÁTICAS ===
Filas con CoorX = 0 o NaN: 16
Filas con CoorY = 0 o NaN: 16
Filas con problemas en coordenadas (CoorX o CoorY): 16

Primeros 10 ejemplos de filas que serán eliminadas:
       Provincia  Estadillo  CoorX  CoorY  Huso
17007         36       2460    NaN    NaN   0.0
20583         32       3461    NaN    NaN   0.0
20584         32       3463    NaN    NaN   0.0
23662         15       4600    NaN    NaN   0.0
23663         15       4601    NaN    NaN   0.0
23664         15       4603    NaN    NaN   0.0
23665         15       4604    NaN    NaN   0.0
23666         15       4607    NaN    NaN   0.0
23667         15       4608    NaN    NaN   0.0
23668         15       4610    NaN    NaN   0.0

Filas e


Dataset limpio guardado como 'ifn4_campo_PCDatosMap.csv'


In [22]:
import pandas as pd
import numpy as np

archivo = 'IFN_España/IFN4_campo/ifn4_campo_PCDatosMap.csv'
df = pd.read_csv(archivo)

df_sin_outliers = df.copy()

for prov_cod, prov_nombre in [(47, "Valladolid"), (50, "Teruel")]:
    df_prov = df[df['Provincia'] == prov_cod].copy()
    if df_prov.empty:
        print(f"⚠️  Provincia {prov_cod} ({prov_nombre}) sin datos, saltando.")
        continue
    media_x, media_y = df_prov['CoorX'].mean(), df_prov['CoorY'].mean()
    df_prov['distancia'] = np.sqrt((df_prov['CoorX'] - media_x)**2 + (df_prov['CoorY'] - media_y)**2)
    outlier = df_prov.loc[df_prov['distancia'].idxmax()]
    print(f"Outlier Provincia {prov_cod} ({prov_nombre}):")
    print(outlier[['Provincia', 'Estadillo', 'CoorX', 'CoorY', 'Huso']])
    df_sin_outliers = df_sin_outliers.drop(outlier.name)

if 'distancia' in df_sin_outliers.columns:
    df_sin_outliers = df_sin_outliers.drop(columns=['distancia'])

print(f"\nFilas tras eliminar outliers: {len(df_sin_outliers)}")
df_sin_outliers.to_csv('IFN_España/IFN4_campo/ifn4_campo_PCDatosMap.csv', index=False)
print("Guardado ifn4_campo_PCDatosMap.csv")


Outlier Provincia 47 (Valladolid):
Provincia          47
Estadillo         242
CoorX        352891.0
CoorY        352891.0
Huso             30.0
Name: 31242, dtype: object
⚠️  Provincia 50 (Teruel) sin datos, saltando.

Filas tras eliminar outliers: 57901


Guardado ifn4_campo_PCDatosMap.csv


corrijo CoorX y CoorY para que sea útil

In [23]:
import pandas as pd

# Ruta del archivo a modificar
archivo_ifn2 = 'IFN_España/IFN2/ifn2_DATEST.csv'

# Cargar el archivo CSV
df = pd.read_csv(archivo_ifn2)

# Realizar las operaciones para sustituir CoorX y CoorY
df['CoorX'] = df['CoorX'] * 1000
df['CoorY'] = df['CoorY'] * 100

# Guardar el archivo con las nuevas columnas
df.to_csv('IFN_España/IFN2/ifn2_DATEST.csv', index=False)

print("Las columnas CoorX y CoorY han sido actualizadas correctamente y el archivo se ha guardado.")


/tmp/ipykernel_2004326/4186403846.py:7: DtypeWarning: Columns (0: CoorX, 1: CoorY, 2: FRACCION1, 3: CUBIERTA, 4: MAXPEND1, 5: MAXPEND2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo_ifn2)


Las columnas CoorX y CoorY han sido actualizadas correctamente y el archivo se ha guardado.


Recupero el Ano completo para IFN2 (81->1981)

In [24]:
import pandas as pd

# Ruta del archivo
ifn2_datedst = 'IFN_España/IFN2/ifn2_DATEST.csv'

# Leer el archivo CSV
df_ifn2 = pd.read_csv(ifn2_datedst)

# Convertir la columna 'Ano' añadiendo '19' delante
df_ifn2['Ano'] = df_ifn2['Ano'].apply(lambda x: f"19{int(x)}" if pd.notnull(x) else x)

# Guardar el archivo modificado
df_ifn2.to_csv(ifn2_datedst, index=False)

print(f"Archivo '{ifn2_datedst}' modificado correctamente.")


/tmp/ipykernel_2004326/1674573.py:7: DtypeWarning: Columns (0: CoorX, 1: CoorY, 2: FRACCION1, 3: CUBIERTA, 4: MAXPEND1, 5: MAXPEND2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ifn2 = pd.read_csv(ifn2_datedst)


Archivo 'IFN_España/IFN2/ifn2_DATEST.csv' modificado correctamente.


In [25]:
import pandas as pd

# Ruta del archivo
ifn2_datedst = 'IFN_España/IFN2/ifn2_DATEST.csv'

# Leer el archivo CSV
df_ifn2 = pd.read_csv(ifn2_datedst)

# Contar la frecuencia de cada valor único en la columna 'Ano'
conteo_ano = df_ifn2['Ano'].value_counts()

# Mostrar el conteo
print("Conteo de cada valor en la columna 'Ano':")
print(conteo_ano)


Conteo de cada valor en la columna 'Ano':
Ano
1992.0    13013
1990.0    11180
1991.0    10763
1993.0     8621
1994.0     8447
1987.0     8007
1995.0     7949
1989.0     7727
1996.0     7706
1988.0     5042
1986.0     4740
1985.0        9
1980.0        7
1982.0        6
1950.0        5
1981.0        4
1913.0        4
194.0         2
1998.0        2
1918.0        2
1984.0        2
1983.0        2
198.0         2
1919.0        2
1930.0        2
1925.0        1
1922.0        1
1933.0        1
1960.0        1
1916.0        1
197.0         1
1949.0        1
199.0         1
1976.0        1
1999.0        1
1946.0        1
1978.0        1
1939.0        1
1956.0        1
1941.0        1
Name: count, dtype: int64


/tmp/ipykernel_2004326/250916979.py:7: DtypeWarning: Columns (0: CoorX, 1: CoorY, 2: FRACCION1, 3: CUBIERTA, 4: MAXPEND1, 5: MAXPEND2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ifn2 = pd.read_csv(ifn2_datedst)


In [26]:
import pandas as pd

# Ruta del archivo
ifn2_datedst = 'IFN_España/IFN2/ifn2_DATEST.csv'

# Leer el archivo CSV
df_ifn2 = pd.read_csv(ifn2_datedst)

# Asegurarse de que 'Provincia' y 'Ano' son columnas válidas
# Rellenar los valores inferiores a 1950 con la moda de su respectiva provincia
moda_por_provincia = df_ifn2.groupby('Provincia')['Ano'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)

# Aplicar la moda de la provincia a los valores de 'Ano' inferiores a 1950
df_ifn2['Ano'] = df_ifn2.apply(
    lambda row: moda_por_provincia[row['Provincia']] if row['Ano'] < 1980 else row['Ano'], axis=1
)

# Guardar el archivo modificado
df_ifn2.to_csv(ifn2_datedst, index=False)

print("Valores inferiores a 1950 en la columna 'Ano' han sido reasignados correctamente.")


/tmp/ipykernel_2004326/811246485.py:7: DtypeWarning: Columns (0: CoorX, 1: CoorY, 2: FRACCION1, 3: CUBIERTA, 4: MAXPEND1, 5: MAXPEND2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ifn2 = pd.read_csv(ifn2_datedst)


Valores inferiores a 1950 en la columna 'Ano' han sido reasignados correctamente.


Añado IDParcela

In [27]:
import os
import pandas as pd

# Función para agregar la columna 'IDParcela'
def agregar_idparcela(df):
    # Verificar si las columnas 'Provincia' y 'Estadillo' están presentes
    if 'Provincia' in df.columns and 'Estadillo' in df.columns:
        # Estandarizar las columnas 'Provincia' y 'Estadillo'
        df['Provincia'] = df['Provincia'].astype(str).str.strip().str.zfill(2)  # Asegura 2 dígitos para Provincia
        df['Estadillo'] = df['Estadillo'].astype(str).str.strip().str.zfill(2)  # Asegura 2 dígitos para Estadillo
        
        # Crear la columna IDParcela combinando 'Provincia' y 'Estadillo'
        df['IDParcela'] = df['Provincia'] + '_' + df['Estadillo']
    
    return df

# Rutas de las carpetas donde están los archivos CSV
carpetas = [
    'IFN_España/IFN4_campo',
    'IFN_España/IFN3_sig',
    'IFN_España/IFN4_sig',
    'IFN_España/IFN3_campo',
    'IFN_España/IFN2'
]

# Recorrer las carpetas
for carpeta in carpetas:
    # Obtener todos los archivos CSV en la carpeta
    archivos_csv = [f for f in os.listdir(carpeta) if f.endswith('.csv')]
    
    # Procesar cada archivo CSV
    for archivo_csv in archivos_csv:
        archivo_path = os.path.join(carpeta, archivo_csv)
        
        # Cargar el archivo CSV
        df = pd.read_csv(archivo_path)
        
        # Verificar si contiene las columnas 'Provincia' y 'Estadillo'
        if 'Provincia' in df.columns and 'Estadillo' in df.columns:
            
            # Agregar la columna 'IDParcela'
            df = agregar_idparcela(df)
            
            # Guardar el archivo con la nueva columna 'IDParcela'
            df.to_csv(archivo_path, index=False)
            print(f"Columna 'IDParcela' añadida a: {archivo_csv}")

print("Proceso completado para todos los archivos CSV.")


Columna 'IDParcela' añadida a: ifn4_campo_PCNueEsp.csv


Columna 'IDParcela' añadida a: ifn4_campo_PCParcelas.csv


Columna 'IDParcela' añadida a: ifn4_campo_PCRegenera.csv


/tmp/ipykernel_2004326/4158543129.py:36: DtypeWarning: Columns (0: Subclase) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo_path)


Columna 'IDParcela' añadida a: ifn4_campo_PCEspMapa.csv
Columna 'IDParcela' añadida a: ifn4_campo_PCEspParc.csv


/tmp/ipykernel_2004326/4158543129.py:36: DtypeWarning: Columns (0: Distanci) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo_path)


... (30 outputs left out)

Columna 'IDParcela' añadida a: ifn2_TIPOSX.csv
Columna 'IDParcela' añadida a: ifn2_parcela_y_especie.csv


Columna 'IDParcela' añadida a: ifn2_MATORR.csv


/tmp/ipykernel_2004326/4158543129.py:36: DtypeWarning: Columns (0: CoorX, 1: CoorY, 2: FRACCION1, 3: CUBIERTA, 4: MAXPEND1, 5: MAXPEND2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo_path)


Columna 'IDParcela' añadida a: ifn2_DATEST.csv
Proceso completado para todos los archivos CSV.


retoque a idparcela de ifn2

In [28]:
import pandas as pd

# Ruta del archivo que contiene 'IDParcela' errónea
archivo = 'IFN_España/IFN2/ifn2_IIFL00BD.csv'

# Cargar el archivo CSV
df = pd.read_csv(archivo)

# Mostrar las primeras filas para ver el estado antes de la corrección
print("Antes de la corrección:")
print(df[['Provincia', 'Estadillo', 'IDParcela']].head())

# Eliminar la columna 'IDParcela'
df = df.drop(columns=['IDParcela'], errors='ignore')

# Reemplazar NaN en 'Provincia' y 'Estadillo' por un valor predeterminado (ej. 0)
df['Provincia'] = df['Provincia'].fillna(0)
df['Estadillo'] = df['Estadillo'].fillna(0)

# Asegurar que 'Provincia' y 'Estadillo' sean cadenas con ceros a la izquierda
df['Provincia'] = df['Provincia'].astype(int).astype(str).str.zfill(2)  # Añadir 0 a la izquierda si es necesario
df['Estadillo'] = df['Estadillo'].astype(int).astype(str).str.zfill(2)  # Añadir 0 a la izquierda si es necesario

# Crear nuevamente la columna 'IDParcela' combinando 'Provincia' y 'Estadillo'
df['IDParcela'] = df['Provincia'] + '_' + df['Estadillo']

# Mostrar las primeras filas después de la corrección
print("\nDespués de la corrección:")
print(df[['Provincia', 'Estadillo', 'IDParcela']].head())

# Guardar el archivo con la columna 'IDParcela' corregida
df.to_csv(archivo, index=False)
print(f"Archivo corregido guardado como: {archivo}")


Antes de la corrección:
   Provincia  Estadillo IDParcela
0        7.0        1.0   7.0_1.0
1        7.0        1.0   7.0_1.0
2        7.0        1.0   7.0_1.0
3        7.0        1.0   7.0_1.0
4        7.0        1.0   7.0_1.0



Después de la corrección:
  Provincia Estadillo IDParcela
0        07        01     07_01
1        07        01     07_01
2        07        01     07_01
3        07        01     07_01
4        07        01     07_01


Archivo corregido guardado como: IFN_España/IFN2/ifn2_IIFL00BD.csv


In [29]:
import pandas as pd

# Ruta del archivo que deseas procesar
archivo = 'IFN_España/IFN2/ifn2_IIFL00BD.csv'

# Cargar el archivo CSV
df = pd.read_csv(archivo)

# Mostrar las primeras filas antes de la eliminación para verificar el estado
print("Antes de la eliminación:")
print(df[['Provincia', 'Estadillo']].head())

# Eliminar las filas donde 'Provincia' o 'Estadillo' sean igual a 0
df_limpio = df[(df['Provincia'] != 0) & (df['Estadillo'] != 0)]

# Mostrar las primeras filas después de la eliminación para verificar el resultado
print("\nDespués de la eliminación:")
print(df_limpio[['Provincia', 'Estadillo']].head())

# Guardar el archivo con las filas eliminadas
df_limpio.to_csv(archivo, index=False)
print(f"Archivo procesado y guardado como: {archivo}")


Antes de la eliminación:
   Provincia  Estadillo
0          7          1
1          7          1
2          7          1
3          7          1
4          7          1

Después de la eliminación:
   Provincia  Estadillo
0          7          1
1          7          1
2          7          1
3          7          1
4          7          1


Archivo procesado y guardado como: IFN_España/IFN2/ifn2_IIFL00BD.csv


de cara a comprobar la unicidad de idparcela eliminamos duplicados

In [30]:
import os
import pandas as pd

# Función para comprobar y eliminar duplicados
def comprobar_duplicados(df):
    # Comprobar duplicados basados en todas las columnas
    duplicados = df[df.duplicated()]
    
    # Si hay duplicados, mostrar las filas duplicadas
    if not duplicados.empty:
        print(f"\nDuplicados encontrados en el archivo:")
        print(duplicados)
    
    # Eliminar los duplicados y retornar el DataFrame sin duplicados
    df_sin_duplicados = df.drop_duplicates()
    
    return df_sin_duplicados, len(duplicados)

# Rutas de las carpetas donde están los archivos CSV
carpetas = [
    'IFN_España/IFN4_campo',
    'IFN_España/IFN3_sig',
    'IFN_España/IFN4_sig',
    'IFN_España/IFN3_campo',
    'IFN_España/IFN2'
]

# Recorrer las carpetas
for carpeta in carpetas:
    # Obtener todos los archivos CSV en la carpeta
    archivos_csv = [f for f in os.listdir(carpeta) if f.endswith('.csv')]
    
    # Procesar cada archivo CSV
    for archivo_csv in archivos_csv:
        archivo_path = os.path.join(carpeta, archivo_csv)
        
        # Cargar el archivo CSV
        df = pd.read_csv(archivo_path)
        
        # Comprobar duplicados
        df_sin_duplicados, num_duplicados = comprobar_duplicados(df)
        
        # Si hay duplicados, informar y guardar el archivo limpio
        if num_duplicados > 0:
            print(f"\nArchivo: {archivo_csv}")
            print(f"Se eliminaron {num_duplicados} duplicados.")
            
            # Guardar el archivo sin duplicados
            df_sin_duplicados.to_csv(archivo_path, index=False)
            print(f"Archivo sin duplicados guardado: {archivo_csv}")
        else:
            print(f"No se encontraron duplicados en el archivo: {archivo_csv}")

print("Proceso completado para todos los archivos CSV.")



Duplicados encontrados en el archivo:
        Provincia  Estadillo  Especie Cla Subclase IDParcela
33142           1          3       28   A        1     01_03
33143           1          3       41   A        1     01_03
33144           1          3       65   A        1     01_03
33145           1          3       72   A        1     01_03
33148           1          3      357   A        1     01_03
...           ...        ...      ...  ..      ...       ...
226675          8       2107      237   A        1   08_2107
226690          8       2115       25   A        1   08_2115
226691          8       2115       39   A        1   08_2115
226692          8       2115       45   A        1   08_2115
226693          8       2115      237   A        1   08_2115

[5690 rows x 6 columns]

Archivo: ifn4_campo_PCNueEsp.csv
Se eliminaron 5690 duplicados.


Archivo sin duplicados guardado: ifn4_campo_PCNueEsp.csv



Duplicados encontrados en el archivo:
       Provincia  Estadillo Subclase   Ano  TipSuelo1  TipSuelo2  TipSuelo3  \
7220          31       2989       6C  2008        1.0        1.0        1.0   
13775         39        182       4C  2009        NaN        NaN        NaN   
48648         40         68        4  2019        NaN        NaN        NaN   

       Nivel1  Nivel2  Rocosid  ...  Textura  MatOrg  PresReg  ModComb  \
7220      1.0     1.0      1.0  ...      3.0     2.0      NaN      1.0   
13775     3.0     NaN      NaN  ...      NaN     NaN      NaN      NaN   
48648     NaN     NaN      NaN  ...      NaN     NaN      NaN      NaN   

       EspCMue  DisEsp  ComEsp  FccArb  FccTot  IDParcela  
7220       0.0     1.0     1.0    35.0    40.0    31_2989  
13775      NaN     NaN     NaN     NaN     NaN     39_182  
48648      NaN     NaN     NaN     NaN     NaN      40_68  

[3 rows x 26 columns]

Archivo: ifn4_campo_PCParcelas.csv
Se eliminaron 3 duplicados.


Archivo sin duplicados guardado: ifn4_campo_PCParcelas.csv

Duplicados encontrados en el archivo:
        Provincia  Estadillo  Especie  Tipo  NumPies  CatDes  Densidad  Hm  \
12973           5       2641       54     6      NaN       3       1.0 NaN   
12976           5       2641       55     6      NaN       1       1.0 NaN   
13154          31          2       74     6      NaN       2       1.0 NaN   
13992          31        234      215     1      NaN       2       1.0 NaN   
15649          31        596       71     6      NaN       1       1.0 NaN   
...           ...        ...      ...   ...      ...     ...       ...  ..   
275666         26        843      215     6      NaN       1       1.0 NaN   
275667         26        843      215     6      NaN       2       1.0 NaN   
277796         26       2248      215     6      NaN       2       2.0 NaN   
278065         26         42       76     6      NaN       2       1.0 NaN   
278068         26         42      215     6 

Archivo sin duplicados guardado: ifn4_campo_PCRegenera.csv

Duplicados encontrados en el archivo:
        Provincia  Estadillo  Especie Cla Subclase  Estado  Ocupa  PosEsp  \
18582           1          9       28   A      NaN     4.0   10.0       1   
18596           1         37       28   A      NaN     4.0   10.0       1   
18607           1         57       28   A      NaN     4.0   10.0       1   
18612           1         71       28   A      NaN     4.0   10.0       1   
18626           1         89       99   A      NaN     2.0    5.0       2   
...           ...        ...      ...  ..      ...     ...    ...     ...   
138191          8       2087        0   A      NaN     0.0    0.0       3   
138194          8       2090        0   A      NaN     0.0    0.0       3   
138200          8       2093        0   A      NaN     0.0    0.0       3   
138215          8       2103        0   A      NaN     0.0    0.0       3   
138230          8       2112        0   A      NaN     

/tmp/ipykernel_2004326/2848688878.py:38: DtypeWarning: Columns (0: Subclase) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo_path)


... (47 outputs left out)

Archivo sin duplicados guardado: ifn2_TIPOSX.csv
No se encontraron duplicados en el archivo: ifn2_parcela_y_especie.csv

Duplicados encontrados en el archivo:
        Provincia  ALTUMED  Estadillo  Especie  NUMORDEN  FRACCAB  \
190038         30      8.0       1917    114.0         1     10.0   
190039         30      4.0       1918    114.0         1      5.0   
190040         30      4.0       1918    101.0         2      5.0   
190041         30      4.0          8    114.0         1     10.0   
190042         30      4.0          8    105.0         2      5.0   
...           ...      ...        ...      ...       ...      ...   
191376         30      5.0       1836    114.0         1     10.0   
191377         30      9.0       1836    122.0         2      5.0   
191378         30      5.0       1836    125.0         3      5.0   
191379         30      4.0       1836    101.0         4      5.0   
191380         30      2.0       1836    129.0         5      5.0   

        Prov

Archivo sin duplicados guardado: ifn2_MATORR.csv


No se encontraron duplicados en el archivo: ifn2_DATEST.csv
Proceso completado para todos los archivos CSV.


/tmp/ipykernel_2004326/2848688878.py:38: DtypeWarning: Columns (0: CoorX, 1: CoorY, 2: FRACCION1, 3: CUBIERTA, 4: MAXPEND1, 5: MAXPEND2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo_path)


diagnostico de duplicados

In [31]:
import pandas as pd

# Rutas de los archivos que se van a procesar
archivos = [
    'IFN_España/IFN4_campo/ifn4_campo_PCDatosMap.csv',
    'IFN_España/IFN3_campo/ifn3_campo_PCDatosMap.csv',
    'IFN_España/IFN4_campo/ifn4_campo_PCParcelas.csv',
    'IFN_España/IFN3_campo/ifn3_campo_PCParcelas.csv'
]

# Función para diagnosticar duplicados en 'IDParcela'
def diagnosticar_duplicados(df):
    # Comprobar si 'IDParcela' está en las columnas
    if 'IDParcela' in df.columns:
        # Encontrar los duplicados en la columna 'IDParcela'
        duplicados = df[df.duplicated(subset=['IDParcela'], keep=False)]
        
        # Si existen duplicados, los devolveremos
        if not duplicados.empty:
            return duplicados
        else:
            return None
    else:
        return None

# Procesar cada archivo
for archivo in archivos:
    # Cargar el archivo CSV
    df = pd.read_csv(archivo)
    
    # Diagnosticar duplicados en la columna 'IDParcela'
    duplicados = diagnosticar_duplicados(df)
    
    # Mostrar los resultados
    if duplicados is not None:
        print(f"\nDuplicados encontrados en el archivo {archivo}:")
        print(duplicados)  # Imprimir los duplicados
    else:
        print(f"No se encontraron duplicados en el archivo {archivo}.")

print("\nDiagnóstico completado para todos los archivos.")



Duplicados encontrados en el archivo IFN_España/IFN4_campo/ifn4_campo_PCDatosMap.csv:
       Provincia  Estadillo     CoorX      CoorY  Huso Hoja50  Tesela  \
3065           5         70  324940.0  4533785.0  30.0    480    69.0   
3066           5         70  324891.0  4533793.0  30.0    480    69.0   
3138           5        173  357944.0  4521776.0  30.0    481    69.0   
3139           5        173  357903.0  4521804.0  30.0    481    69.0   
3899           5       2004  358858.0  4532831.0  30.0    481    69.0   
...          ...        ...       ...        ...   ...    ...     ...   
56111         13       2442  386889.0  4253793.0  30.0  860.0     NaN   
56112         13       2445  386869.0  4249747.0  30.0  860.0     NaN   
56113         13       2445  386889.0  4249793.0  30.0  860.0     NaN   
56120         13       2462  415881.0  4250744.0  30.0  861.0     NaN   
56121         13       2462  415889.0  4250793.0  30.0  861.0     NaN   

      IDParcela  
3065      05_70  



Duplicados encontrados en el archivo IFN_España/IFN3_campo/ifn3_campo_PCParcelas.csv:
       Provincia  Estadillo Subclase   Ano  TipSuelo1  TipSuelo2  TipSuelo3  \
26            18         28       3C  2007        NaN        NaN        NaN   
27            18         28       3E  2007        1.0        1.0        1.0   
35            18         37       3C  2007        NaN        NaN        NaN   
36            18         37       3E  2007        1.0        1.0        1.0   
63            18         77       3C  2007        NaN        NaN        NaN   
...          ...        ...      ...   ...        ...        ...        ...   
98827         47        886       3E  2002        NaN        NaN        NaN   
98834         47        908       3C  2002        NaN        NaN        NaN   
98835         47        908       3E  2002        NaN        NaN        NaN   
98878         47        960       3C  2002        NaN        NaN        NaN   
98879         47        960       3E  2002  

/tmp/ipykernel_2004326/980232493.py:29: DtypeWarning: Columns (0: Ano, 1: Nivel2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo)


ahora busco duplicados de IDParcela en cada documento y elimino el que más Nan tenga 

In [32]:
#primero quito los duplicados en idparcela y orienta1 o IDparcela y coorx porque se que esas entradas estan repetidas tal cual
import pandas as pd

# Rutas de los archivos que se van a procesar
archivos = [
    'IFN_España/IFN4_campo/ifn4_campo_PCDatosMap.csv',
    'IFN_España/IFN3_campo/ifn3_campo_PCDatosMap.csv',
    'IFN_España/IFN4_campo/ifn4_campo_PCParcelas.csv',
    'IFN_España/IFN3_campo/ifn3_campo_PCParcelas.csv'
]

# Función para eliminar duplicados basados en la cantidad de NaN
def eliminar_duplicados_por_nan(df, subset_columns):
    # Comprobar si las columnas necesarias están presentes
    if all(col in df.columns for col in subset_columns):
        # Contar duplicados antes de la eliminación
        duplicados_iniciales = df.duplicated(subset=subset_columns, keep=False).sum()

        # Encontrar duplicados en las columnas especificadas
        duplicados = df[df.duplicated(subset=subset_columns, keep=False)]
        
        # Si hay duplicados
        if not duplicados.empty:
            # Ordenar duplicados por la cantidad de NaN en cada fila y eliminar el duplicado con más NaN
            df = df.sort_values(by=subset_columns)  # Ordenar por las columnas seleccionadas para consistencia
            df['NaN_count'] = df.isna().sum(axis=1)  # Contar NaN en cada fila
            df = df.loc[df.groupby(subset_columns)['NaN_count'].idxmin()]  # Mantener el registro con menos NaN

            # Eliminar la columna 'NaN_count' después de usarla
            df = df.drop(columns=['NaN_count'])

        # Contar duplicados después de la eliminación
        duplicados_finales = df.duplicated(subset=subset_columns, keep=False).sum()

        return df, duplicados_iniciales, duplicados_finales
    else:
        return df, 0, 0  # Si no están las columnas necesarias, no hay duplicados

# Procesar cada archivo
for archivo in archivos:
    # Cargar el archivo CSV
    df = pd.read_csv(archivo)
    
    # Definir las columnas sobre las que se eliminarán los duplicados
    if 'PCParcelas' in archivo:
        subset_columns = ['IDParcela', 'Orienta1']
    else:
        subset_columns = ['IDParcela', 'CoorX']
    
    # Eliminar duplicados con más NaN y obtener el conteo de duplicados
    df_sin_duplicados, duplicados_iniciales, duplicados_finales = eliminar_duplicados_por_nan(df, subset_columns)
    
    # Mostrar los resultados
    print(f"\nArchivo: {archivo}")
    print(f"Duplicados antes de la criba: {duplicados_iniciales}")
    print(f"Duplicados después de la criba: {duplicados_finales}")
    
    # Guardar el archivo limpio
    df_sin_duplicados.to_csv(archivo, index=False)
    print(f"Archivo procesado y guardado: {archivo}")

print("Proceso completado para todos los archivos.")



Archivo: IFN_España/IFN4_campo/ifn4_campo_PCDatosMap.csv
Duplicados antes de la criba: 20
Duplicados después de la criba: 0
Archivo procesado y guardado: IFN_España/IFN4_campo/ifn4_campo_PCDatosMap.csv



Archivo: IFN_España/IFN3_campo/ifn3_campo_PCDatosMap.csv
Duplicados antes de la criba: 0
Duplicados después de la criba: 0


Archivo procesado y guardado: IFN_España/IFN3_campo/ifn3_campo_PCDatosMap.csv

Archivo: IFN_España/IFN4_campo/ifn4_campo_PCParcelas.csv
Duplicados antes de la criba: 133
Duplicados después de la criba: 0


Archivo procesado y guardado: IFN_España/IFN4_campo/ifn4_campo_PCParcelas.csv

Archivo: IFN_España/IFN3_campo/ifn3_campo_PCParcelas.csv
Duplicados antes de la criba: 630
Duplicados después de la criba: 0


/tmp/ipykernel_2004326/2143338001.py:42: DtypeWarning: Columns (0: Ano, 1: Nivel2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo)


Archivo procesado y guardado: IFN_España/IFN3_campo/ifn3_campo_PCParcelas.csv
Proceso completado para todos los archivos.


In [33]:
import pandas as pd

# Rutas de los archivos que se van a procesar
archivos = [
    'IFN_España/IFN2/ifn2_DATEST.csv',
    'IFN_España/IFN3_campo/ifn3_campo_PCDatosMap.csv',
    'IFN_España/IFN3_campo/ifn3_campo_PCParcelas.csv',
    'IFN_España/IFN4_campo/ifn4_campo_PCDatosMap.csv',
    'IFN_España/IFN4_campo/ifn4_campo_PCParcelas.csv'
]

# Función para eliminar duplicados basados en la cantidad de NaN en 'IDParcela'
def eliminar_duplicados_por_nan(df):
    # Comprobar si 'IDParcela' está en las columnas
    if 'IDParcela' in df.columns:
        # Contar duplicados antes de la eliminación
        duplicados_iniciales = df.duplicated(subset=['IDParcela'], keep=False).sum()

        # Encontrar duplicados en la columna 'IDParcela'
        duplicados = df[df.duplicated(subset=['IDParcela'], keep=False)]
        
        # Si hay duplicados
        if not duplicados.empty:
            # Ordenar duplicados por la cantidad de NaN en cada fila y eliminar el duplicado con más NaN
            df = df.sort_values(by='IDParcela')  # Ordenar por 'IDParcela' para asegurar consistencia
            df['NaN_count'] = df.isna().sum(axis=1)  # Contar NaN en cada fila
            df = df.loc[df.groupby('IDParcela')['NaN_count'].idxmin()]  # Mantener el registro con menos NaN

            # Eliminar la columna 'NaN_count' después de usarla
            df = df.drop(columns=['NaN_count'])

        # Contar duplicados después de la eliminación
        duplicados_finales = df.duplicated(subset=['IDParcela'], keep=False).sum()

        return df, duplicados_iniciales, duplicados_finales
    else:
        return df, 0, 0  # No hay columna 'IDParcela', no hay duplicados

# Procesar cada archivo
for archivo in archivos:
    # Cargar el archivo CSV
    df = pd.read_csv(archivo)
    
    # Eliminar duplicados con más NaN y obtener el conteo de duplicados
    df_sin_duplicados, duplicados_iniciales, duplicados_finales = eliminar_duplicados_por_nan(df)
    
    # Mostrar los resultados
    print(f"\nArchivo: {archivo}")
    print(f"Duplicados antes de la criba: {duplicados_iniciales}")
    print(f"Duplicados después de la criba: {duplicados_finales}")
    
    # Guardar el archivo limpio
    df_sin_duplicados.to_csv(archivo, index=False)
    print(f"Archivo procesado y guardado: {archivo}")

print("Proceso completado para todos los archivos.")


/tmp/ipykernel_2004326/3220659836.py:42: DtypeWarning: Columns (0: CoorX, 1: CoorY, 2: FRACCION1, 3: CUBIERTA, 4: MAXPEND1, 5: MAXPEND2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo)



Archivo: IFN_España/IFN2/ifn2_DATEST.csv
Duplicados antes de la criba: 0
Duplicados después de la criba: 0


Archivo procesado y guardado: IFN_España/IFN2/ifn2_DATEST.csv

Archivo: IFN_España/IFN3_campo/ifn3_campo_PCDatosMap.csv
Duplicados antes de la criba: 0
Duplicados después de la criba: 0
Archivo procesado y guardado: IFN_España/IFN3_campo/ifn3_campo_PCDatosMap.csv


/tmp/ipykernel_2004326/3220659836.py:42: DtypeWarning: Columns (0: Ano) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo)



Archivo: IFN_España/IFN3_campo/ifn3_campo_PCParcelas.csv
Duplicados antes de la criba: 10477
Duplicados después de la criba: 0


Archivo procesado y guardado: IFN_España/IFN3_campo/ifn3_campo_PCParcelas.csv

Archivo: IFN_España/IFN4_campo/ifn4_campo_PCDatosMap.csv
Duplicados antes de la criba: 6704
Duplicados después de la criba: 0
Archivo procesado y guardado: IFN_España/IFN4_campo/ifn4_campo_PCDatosMap.csv



Archivo: IFN_España/IFN4_campo/ifn4_campo_PCParcelas.csv
Duplicados antes de la criba: 7029
Duplicados después de la criba: 0


Archivo procesado y guardado: IFN_España/IFN4_campo/ifn4_campo_PCParcelas.csv
Proceso completado para todos los archivos.


Ahora vemos que las combinaciones de 

IDParcela y Especie de     
data/raw/IFN_España/IFN2/ifn2_IIFL00BD.csv, 
data/raw/IFN_España/IFN3_campo/ifn3_campo_PCEspParc.csv, 
data/raw/IFN_España/IFN4_campo/ifn4_campo_PCEspParc.csv                                  
y de IDParcela, Especie y CD de 
data/raw/IFN_España/IFN2/ifn2_IIFL03BD.csv, 
data/raw/IFN_España/IFN3_sig/ifn3_sig_Parcelas_exs.csv, 
data/raw/IFN_España/IFN4_sig/ifn4_sig_Parcelas_exs.csv                                   
y de IDParcela, Especie y CatDes de 
data/raw/IFN_España/IFN3_campo/ifn3_campo_PCRegenera.csv, 
data/raw/IFN_España/IFN4_campo/ifn4_campo_PCRegenera.csv

In [34]:
import pandas as pd

# Rutas de los archivos a procesar
archivos = [
    'IFN_España/IFN2/ifn2_IIFL00BD.csv',  # Combinación: IDParcela, OrdenIf2
    'IFN_España/IFN3_campo/ifn3_campo_PCEspParc.csv',  # Combinación: IDParcela, Especie
    'IFN_España/IFN4_campo/ifn4_campo_PCEspParc.csv',  # Combinación: IDParcela, Especie
    'IFN_España/IFN2/ifn2_IIFL03BD.csv',  # Combinación: IDParcela, Especie, CD
    'IFN_España/IFN3_sig/ifn3_sig_Parcelas_exs.csv',  # Combinación: IDParcela, Especie, CD
    'IFN_España/IFN4_sig/ifn4_sig_Parcelas_exs.csv',  # Combinación: IDParcela, Especie, CD
    'IFN_España/IFN3_campo/ifn3_campo_PCRegenera.csv',  # Combinación: IDParcela, Especie, CatDes
    'IFN_España/IFN4_campo/ifn4_campo_PCRegenera.csv'  # Combinación: IDParcela, Especie, CatDes
]

# Función para verificar la unicidad de combinaciones
def verificar_unicidad(df, columnas):
    # Contar los duplicados en las combinaciones de columnas
    duplicados_iniciales = df.duplicated(subset=columnas, keep=False).sum()
    
    # Si hay duplicados, los imprimimos
    if duplicados_iniciales > 0:
        print(f"Duplicados encontrados en las combinaciones de columnas {columnas}: {duplicados_iniciales}")
    else:
        print(f"Las combinaciones de columnas {columnas} son únicas.")
    
    return duplicados_iniciales

# Procesar cada archivo
for archivo in archivos:
    # Cargar el archivo CSV
    df = pd.read_csv(archivo)
    
    # Comprobar la unicidad de las combinaciones necesarias
    if archivo == 'IFN_España/IFN2/ifn2_IIFL00BD.csv':
        duplicados = verificar_unicidad(df, ['IDParcela', 'OrdenIf2'])
    elif archivo in ['IFN_España/IFN3_campo/ifn3_campo_PCEspParc.csv', 'IFN_España/IFN4_campo/ifn4_campo_PCEspParc.csv']:
        duplicados = verificar_unicidad(df, ['IDParcela', 'Especie'])
    elif archivo == 'IFN_España/IFN2/ifn2_IIFL03BD.csv':
        duplicados = verificar_unicidad(df, ['IDParcela', 'Especie', 'CD'])
    elif archivo in ['IFN_España/IFN3_sig/ifn3_sig_Parcelas_exs.csv', 'IFN_España/IFN4_sig/ifn4_sig_Parcelas_exs.csv']:
        duplicados = verificar_unicidad(df, ['IDParcela', 'Especie', 'CD'])
    elif archivo in ['IFN_España/IFN3_campo/ifn3_campo_PCRegenera.csv', 'IFN_España/IFN4_campo/ifn4_campo_PCRegenera.csv']:
        duplicados = verificar_unicidad(df, ['IDParcela', 'Especie', 'CatDes'])
    
    print(f"\nEn el archivo {archivo}, duplicados encontrados: {duplicados}")

print("\nProceso completado para todos los archivos.")



Las combinaciones de columnas ['IDParcela', 'OrdenIf2'] son únicas.

En el archivo IFN_España/IFN2/ifn2_IIFL00BD.csv, duplicados encontrados: 0
Duplicados encontrados en las combinaciones de columnas ['IDParcela', 'Especie']: 13785

En el archivo IFN_España/IFN3_campo/ifn3_campo_PCEspParc.csv, duplicados encontrados: 13785
Duplicados encontrados en las combinaciones de columnas ['IDParcela', 'Especie']: 6292

En el archivo IFN_España/IFN4_campo/ifn4_campo_PCEspParc.csv, duplicados encontrados: 6292


Las combinaciones de columnas ['IDParcela', 'Especie', 'CD'] son únicas.

En el archivo IFN_España/IFN2/ifn2_IIFL03BD.csv, duplicados encontrados: 0


Duplicados encontrados en las combinaciones de columnas ['IDParcela', 'Especie', 'CD']: 2899

En el archivo IFN_España/IFN3_sig/ifn3_sig_Parcelas_exs.csv, duplicados encontrados: 2899


Duplicados encontrados en las combinaciones de columnas ['IDParcela', 'Especie', 'CD']: 7779

En el archivo IFN_España/IFN4_sig/ifn4_sig_Parcelas_exs.csv, duplicados encontrados: 7779
Duplicados encontrados en las combinaciones de columnas ['IDParcela', 'Especie', 'CatDes']: 1318

En el archivo IFN_España/IFN3_campo/ifn3_campo_PCRegenera.csv, duplicados encontrados: 1318


Duplicados encontrados en las combinaciones de columnas ['IDParcela', 'Especie', 'CatDes']: 7739

En el archivo IFN_España/IFN4_campo/ifn4_campo_PCRegenera.csv, duplicados encontrados: 7739

Proceso completado para todos los archivos.


/tmp/ipykernel_2004326/1255766246.py:31: DtypeWarning: Columns (0: Especie) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo)


In [35]:
import pandas as pd

# Rutas de los archivos a procesar
archivos = [
    'IFN_España/IFN2/ifn2_IIFL00BD.csv',  # Combinación: IDParcela, Especie, OrdenIf2
    'IFN_España/IFN3_campo/ifn3_campo_PCEspParc.csv',  # Combinación: IDParcela, Especie
    'IFN_España/IFN4_campo/ifn4_campo_PCEspParc.csv',  # Combinación: IDParcela, Especie
    'IFN_España/IFN2/ifn2_IIFL03BD.csv',  # Combinación: IDParcela, Especie, CD
    'IFN_España/IFN3_sig/ifn3_sig_Parcelas_exs.csv',  # Combinación: IDParcela, Especie, CD
    'IFN_España/IFN4_sig/ifn4_sig_Parcelas_exs.csv',  # Combinación: IDParcela, Especie, CD
    'IFN_España/IFN3_campo/ifn3_campo_PCRegenera.csv',  # Combinación: IDParcela, Especie, CatDes
    'IFN_España/IFN4_campo/ifn4_campo_PCRegenera.csv'  # Combinación: IDParcela, Especie, CatDes
]

# Función para eliminar duplicados con más NaN y contar duplicados
def eliminar_duplicados_por_nan(df, columnas):
    # Contar duplicados antes de la eliminación
    duplicados_iniciales = df.duplicated(subset=columnas, keep=False).sum()

    # Encontrar duplicados en las combinaciones de columnas
    duplicados = df[df.duplicated(subset=columnas, keep=False)]

    # Si hay duplicados, ordenarlos por la cantidad de NaN en cada fila
    if not duplicados.empty:
        df['NaN_count'] = df.isna().sum(axis=1)  # Contar NaN en cada fila
        df = df.loc[df.groupby(columnas)['NaN_count'].idxmin()]  # Mantener el registro con menos NaN
        
        # Eliminar la columna 'NaN_count' después de usarla
        df = df.drop(columns=['NaN_count'])

    # Contar duplicados después de la eliminación
    duplicados_finales = df.duplicated(subset=columnas, keep=False).sum()

    return df, duplicados_iniciales, duplicados_finales

# Procesar cada archivo
for archivo in archivos:
    # Cargar el archivo CSV
    df = pd.read_csv(archivo)
    
    # Comprobar la unicidad de las combinaciones necesarias y eliminar duplicados con más NaN
    if archivo == 'IFN_España/IFN2/ifn2_IIFL00BD.csv':
        df_sin_duplicados, duplicados_iniciales, duplicados_finales = eliminar_duplicados_por_nan(df, ['IDParcela', 'Especie', 'OrdenIf2'])
    elif archivo in ['IFN_España/IFN3_campo/ifn3_campo_PCEspParc.csv', 'IFN_España/IFN4_campo/ifn4_campo_PCEspParc.csv']:
        df_sin_duplicados, duplicados_iniciales, duplicados_finales = eliminar_duplicados_por_nan(df, ['IDParcela', 'Especie'])
    elif archivo == 'IFN_España/IFN2/ifn2_IIFL03BD.csv':
        df_sin_duplicados, duplicados_iniciales, duplicados_finales = eliminar_duplicados_por_nan(df, ['IDParcela', 'Especie', 'CD'])
    elif archivo in ['IFN_España/IFN3_sig/ifn3_sig_Parcelas_exs.csv', 'IFN_España/IFN4_sig/ifn4_sig_Parcelas_exs.csv']:
        df_sin_duplicados, duplicados_iniciales, duplicados_finales = eliminar_duplicados_por_nan(df, ['IDParcela', 'Especie', 'CD'])
    elif archivo in ['IFN_España/IFN3_campo/ifn3_campo_PCRegenera.csv', 'IFN_España/IFN4_campo/ifn4_campo_PCRegenera.csv']:
        df_sin_duplicados, duplicados_iniciales, duplicados_finales = eliminar_duplicados_por_nan(df, ['IDParcela', 'Especie', 'CatDes'])

    # Mostrar el número de duplicados antes y después de la criba
    print(f"\nArchivo: {archivo}")
    print(f"Duplicados antes de la criba: {duplicados_iniciales}")
    print(f"Duplicados después de la criba: {duplicados_finales}")
    
    # Guardar el archivo con la eliminación de duplicados
    df_sin_duplicados.to_csv(archivo, index=False)
    print(f"Archivo procesado y guardado: {archivo}")

print("\nProceso completado para todos los archivos.")



Archivo: IFN_España/IFN2/ifn2_IIFL00BD.csv
Duplicados antes de la criba: 0
Duplicados después de la criba: 0


Archivo procesado y guardado: IFN_España/IFN2/ifn2_IIFL00BD.csv

Archivo: IFN_España/IFN3_campo/ifn3_campo_PCEspParc.csv
Duplicados antes de la criba: 13785
Duplicados después de la criba: 0


Archivo procesado y guardado: IFN_España/IFN3_campo/ifn3_campo_PCEspParc.csv

Archivo: IFN_España/IFN4_campo/ifn4_campo_PCEspParc.csv
Duplicados antes de la criba: 6292
Duplicados después de la criba: 0
Archivo procesado y guardado: IFN_España/IFN4_campo/ifn4_campo_PCEspParc.csv



Archivo: IFN_España/IFN2/ifn2_IIFL03BD.csv
Duplicados antes de la criba: 0
Duplicados después de la criba: 0


Archivo procesado y guardado: IFN_España/IFN2/ifn2_IIFL03BD.csv



Archivo: IFN_España/IFN3_sig/ifn3_sig_Parcelas_exs.csv
Duplicados antes de la criba: 2899
Duplicados después de la criba: 0


... (3 outputs left out)

/tmp/ipykernel_2004326/2888455469.py:39: DtypeWarning: Columns (0: Especie) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo)



Archivo: IFN_España/IFN3_campo/ifn3_campo_PCRegenera.csv
Duplicados antes de la criba: 1318
Duplicados después de la criba: 0


Archivo procesado y guardado: IFN_España/IFN3_campo/ifn3_campo_PCRegenera.csv

Archivo: IFN_España/IFN4_campo/ifn4_campo_PCRegenera.csv
Duplicados antes de la criba: 7739
Duplicados después de la criba: 0


Archivo procesado y guardado: IFN_España/IFN4_campo/ifn4_campo_PCRegenera.csv

Proceso completado para todos los archivos.


Ahora ya no hay duplicados en las tablas (salvo en las de pies mayores y pies menores que en principio no vamos a usar)

Comprobacion tontorrona de que los IDParcela coinciden entre IFN4 e IFN3 (Filtrar las filas donde IDParcela coincida.Comprobar si los valores de CoorX son iguales para esas filas.)

In [36]:
import pandas as pd

# Rutas de los archivos a procesar
archivo_ifn3 = 'IFN_España/IFN3_campo/ifn3_campo_PCDatosMap.csv'
archivo_ifn4 = 'IFN_España/IFN4_campo/ifn4_campo_PCDatosMap.csv'

# Cargar los archivos CSV
df_ifn3 = pd.read_csv(archivo_ifn3)
df_ifn4 = pd.read_csv(archivo_ifn4)

# Asegurarse de que no haya valores NaN en las columnas necesarias
df_ifn3 = df_ifn3.dropna(subset=['IDParcela', 'CoorY'])
df_ifn4 = df_ifn4.dropna(subset=['IDParcela', 'CoorY'])

# Hacer una fusión de los DataFrames por 'IDParcela'
merged_df = pd.merge(df_ifn3[['IDParcela', 'CoorY']], df_ifn4[['IDParcela', 'CoorY']], on='IDParcela', suffixes=('_ifn3', '_ifn4'))

# Comprobar si los valores de CoorY coinciden
merged_df['CoorY_coincide'] = merged_df['CoorY_ifn3'] == merged_df['CoorY_ifn4']

# Contar cuántos coincidieron y cuántos no
coincidencias = merged_df['CoorY_coincide'].sum()
no_coincidencias = len(merged_df) - coincidencias

# Mostrar los resultados
print(f"Total de coincidencias en CoorY: {coincidencias}")
print(f"Total de no coincidencias en CoorY: {no_coincidencias}")

# Mostrar las filas donde no coinciden las coordenadas CoorY
print("\nFilas donde las coordenadas CoorY no coinciden:")
print(merged_df[merged_df['CoorY_coincide'] == False])


Total de coincidencias en CoorY: 10014
Total de no coincidencias en CoorY: 34317

Filas donde las coordenadas CoorY no coinciden:
      IDParcela  CoorY_ifn3  CoorY_ifn4  CoorY_coincide
0         26_01     4721000   4721015.0           False
1         26_03     4720000   4720016.0           False
2         26_04     4720000   4720043.0           False
3         26_05     4720000   4720041.0           False
4         26_06     4720000   4720057.0           False
...         ...         ...         ...             ...
44326   47_2448     4558000   4557793.0           False
44327   47_2449     4557000   4556794.0           False
44328   47_2450     4558000   4557793.0           False
44329   47_2451     4558000   4557793.0           False
44330   47_2452     4557000   4556793.0           False

[34317 rows x 4 columns]


Total de coincidencias en CoorY: 10025
Total de no coincidencias en CoorY: 37253
Total de coincidencias en CoorX: 10050
Total de no coincidencias en CoorX: 37228

Total de coincidencias en (CoorX, CoorY): 9714
Total de no coincidencias en (CoorX, CoorY): 37564

Vamos a pensar que se debe a redondeo... estudio la magnitud de la diferencia

In [37]:
import pandas as pd

# Rutas de los archivos a procesar
archivo_ifn3 = 'IFN_España/IFN3_campo/ifn3_campo_PCDatosMap.csv'
archivo_ifn4 = 'IFN_España/IFN4_campo/ifn4_campo_PCDatosMap.csv'

# Cargar los archivos CSV
df_ifn3 = pd.read_csv(archivo_ifn3)
df_ifn4 = pd.read_csv(archivo_ifn4)

# Asegurarse de que no haya valores NaN en las columnas necesarias
df_ifn3 = df_ifn3.dropna(subset=['IDParcela', 'CoorX', 'CoorY'])
df_ifn4 = df_ifn4.dropna(subset=['IDParcela', 'CoorX', 'CoorY'])

# Hacer una fusión de los DataFrames por 'IDParcela'
merged_df = pd.merge(df_ifn3[['IDParcela', 'CoorX', 'CoorY']], df_ifn4[['IDParcela', 'CoorX', 'CoorY']], on='IDParcela', suffixes=('_ifn3', '_ifn4'))

# Calcular la diferencia entre CoorX de IFN3 e IFN4
merged_df['diferencia_CoorX'] = abs(merged_df['CoorX_ifn3'] - merged_df['CoorX_ifn4'])

# Calcular la diferencia entre CoorY de IFN3 e IFN4
merged_df['diferencia_CoorY'] = abs(merged_df['CoorY_ifn3'] - merged_df['CoorY_ifn4'])

# Filtrar las filas que cumplen las condiciones
coorx_diferencia_menor_110 = merged_df[merged_df['diferencia_CoorX'] < 300]
coory_diferencia_menor_210 = merged_df[merged_df['diferencia_CoorY'] < 300]

# Contar cuántos valores cumplen las condiciones
coorx_count = coorx_diferencia_menor_110.shape[0]
coory_count = coory_diferencia_menor_210.shape[0]

# Filtrar las filas que no cumplen las condiciones
coorx_no_diferencia_menor_110 = merged_df[merged_df['diferencia_CoorX'] >= 300]
coory_no_diferencia_menor_210 = merged_df[merged_df['diferencia_CoorY'] >= 300]

# Contar cuántos valores no cumplen las condiciones
coorx_no_count = coorx_no_diferencia_menor_110.shape[0]
coory_no_count = coory_no_diferencia_menor_210.shape[0]

# Mostrar los resultados
print(f"Cantidad de parcelas donde la diferencia en CoorX es menor a 300 unidades: {coorx_count}")
print(f"Cantidad de parcelas donde la diferencia en CoorY es menor a 300 unidades: {coory_count}")
print(f"Cantidad de parcelas donde la diferencia en CoorX NO es menor a 300 unidades: {coorx_no_count}")
print(f"Cantidad de parcelas donde la diferencia en CoorY NO es menor a 300 unidades: {coory_no_count}")



Cantidad de parcelas donde la diferencia en CoorX es menor a 300 unidades: 39577
Cantidad de parcelas donde la diferencia en CoorY es menor a 300 unidades: 39565
Cantidad de parcelas donde la diferencia en CoorX NO es menor a 300 unidades: 4754
Cantidad de parcelas donde la diferencia en CoorY NO es menor a 300 unidades: 4766


Ojo, CoorX y CoorY de IFN2 estan mal registrados, CoorX ha sido sustituido por CoorX*1000 y CoorY por CoorY*10000

In [38]:
import pandas as pd
import matplotlib.pyplot as plt

# Rutas de los archivos a procesar
archivo_ifn2 = 'IFN_España/IFN2/ifn2_DATEST.csv'
archivo_ifn4 = 'IFN_España/IFN4_campo/ifn4_campo_PCDatosMap.csv'

# Cargar los archivos CSV
df_ifn2 = pd.read_csv(archivo_ifn2)
df_ifn4 = pd.read_csv(archivo_ifn4)

# Asegurarse de que no haya valores NaN en las columnas necesarias
df_ifn2 = df_ifn2.dropna(subset=['IDParcela', 'CoorX', 'CoorY'])
df_ifn4 = df_ifn4.dropna(subset=['IDParcela', 'CoorX', 'CoorY'])

# Convertir las columnas 'CoorX' y 'CoorY' a tipo numérico, si no lo están
df_ifn2['CoorX'] = pd.to_numeric(df_ifn2['CoorX'], errors='coerce')
df_ifn2['CoorY'] = pd.to_numeric(df_ifn2['CoorY'], errors='coerce')
df_ifn4['CoorX'] = pd.to_numeric(df_ifn4['CoorX'], errors='coerce')
df_ifn4['CoorY'] = pd.to_numeric(df_ifn4['CoorY'], errors='coerce')

# Hacer una fusión de los DataFrames por 'IDParcela'
merged_df = pd.merge(df_ifn2[['IDParcela', 'CoorX', 'CoorY']], df_ifn4[['IDParcela', 'CoorX', 'CoorY']], on='IDParcela', suffixes=('_ifn2', '_ifn4'))

# Calcular las diferencias entre CoorX y CoorY 
merged_df['diferencia_CoorX'] = abs(merged_df['CoorX_ifn2'] - merged_df['CoorX_ifn4']) 
merged_df['diferencia_CoorY'] = abs(merged_df['CoorY_ifn2'] - merged_df['CoorY_ifn4'])

# Contar las coincidencias de IDParcela
coincidencias_idparcela = len(merged_df)

# Filtrar las diferencias que son inferiores 300
coincidencias_bajas_CoorX = merged_df[merged_df['diferencia_CoorX'] < 300]
coincidencias_bajas_CoorY = merged_df[merged_df['diferencia_CoorY'] < 300]

# Mostrar los resultados
print(f"Total de coincidencias de IDParcelas: {coincidencias_idparcela}")
print(f"Coincidencias con diferencia en CoorX < 300: {len(coincidencias_bajas_CoorX)}")
print(f"Coincidencias con diferencia en CoorY < 300: {len(coincidencias_bajas_CoorY)}")

# Estadísticas para la diferencia en CoorX
print("\nEstadísticas de la diferencia en CoorX:")
print(f"Media: {merged_df['diferencia_CoorX'].mean()}")
print(f"Mediana: {merged_df['diferencia_CoorX'].median()}")
print(f"Desviación Estándar: {merged_df['diferencia_CoorX'].std()}")
print(f"Moda: {merged_df['diferencia_CoorX'].mode()[0]}")

# Estadísticas para la diferencia en CoorY
print("\nEstadísticas de la diferencia en CoorY:")
print(f"Media: {merged_df['diferencia_CoorY'].mean()}")
print(f"Mediana: {merged_df['diferencia_CoorY'].median()}")
print(f"Desviación Estándar: {merged_df['diferencia_CoorY'].std()}")
print(f"Moda: {merged_df['diferencia_CoorY'].mode()[0]}")

# Limpiar inf/nan antes de graficar
merged_df = merged_df.replace([float("inf"), float("-inf")], pd.NA).dropna(subset=["diferencia_CoorX", "diferencia_CoorY"])

# Gráficos de la distribución de las diferencias
plt.figure(figsize=(10, 6))

# Histograma para CoorX
plt.subplot(1, 2, 1)
plt.hist(merged_df['diferencia_CoorX'], bins=50, alpha=0.7, color='blue', label='Diferencia CoorX')
plt.title('Distribución de la Diferencia en CoorX')
plt.xlabel('Diferencia en CoorX')
plt.ylabel('Frecuencia')
plt.legend()

# Histograma para CoorY
plt.subplot(1, 2, 2)
plt.hist(merged_df['diferencia_CoorY'], bins=50, alpha=0.7, color='green', label='Diferencia CoorY')
plt.title('Distribución de la Diferencia en CoorY')
plt.xlabel('Diferencia en CoorY')
plt.ylabel('Frecuencia')
plt.legend()

plt.tight_layout()
plt.show()


/tmp/ipykernel_2004326/3082750769.py:9: DtypeWarning: Columns (0: CoorX, 1: CoorY, 2: FRACCION1, 3: CUBIERTA, 4: MAXPEND1, 5: MAXPEND2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ifn2 = pd.read_csv(archivo_ifn2)


Total de coincidencias de IDParcelas: 39016
Coincidencias con diferencia en CoorX < 300: 6562
Coincidencias con diferencia en CoorY < 300: 0

Estadísticas de la diferencia en CoorX:
Media: inf
Mediana: inf
Desviación Estándar: nan
Moda: inf

Estadísticas de la diferencia en CoorY:
Media: inf
Mediana: 4245092.0
Desviación Estándar: nan
Moda: inf


<venv>/lib/python3.13/site-packages/pandas/core/nanops.py:1020: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
<venv>/lib/python3.13/site-packages/pandas/core/nanops.py:1020: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/tmp/ipykernel_2004326/3082750769.py:78: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Me quedo con las parcelas cuya diferencia en las coordenadas es inferior a 300 unidades.
Categorías de Presencia:
1: El valor de IDParcela está en ifn2, ifn3 e ifn4.

2: El valor de IDParcela solo está en ifn2.

3: El valor de IDParcela solo está en ifn3.

4: El valor de IDParcela solo está en ifn4.

5: El valor de IDParcela está en ifn2 e ifn3, pero no en ifn4.

6: El valor de IDParcela está en ifn2 e ifn4, pero no en ifn3.

7: El valor de IDParcela está en ifn4 e ifn3, pero no en ifn2.

In [39]:
import pandas as pd

# Rutas de los archivos a procesar
archivo_ifn2 = 'IFN_España/IFN2/ifn2_DATEST.csv'
archivo_ifn3 = 'IFN_España/IFN3_campo/ifn3_campo_PCDatosMap.csv'
archivo_ifn4 = 'IFN_España/IFN4_campo/ifn4_campo_PCDatosMap.csv'

# Cargar los archivos CSV
df_ifn2 = pd.read_csv(archivo_ifn2)
df_ifn3 = pd.read_csv(archivo_ifn3)
df_ifn4 = pd.read_csv(archivo_ifn4)

# Asegurarse de que no haya valores NaN en las columnas necesarias
df_ifn2 = df_ifn2.dropna(subset=['IDParcela', 'CoorX', 'CoorY'])
df_ifn3 = df_ifn3.dropna(subset=['IDParcela', 'CoorX', 'CoorY'])
df_ifn4 = df_ifn4.dropna(subset=['IDParcela', 'CoorX', 'CoorY'])

# Convertir las columnas 'CoorX' y 'CoorY' a tipo numérico, si no lo están
df_ifn2['CoorX'] = pd.to_numeric(df_ifn2['CoorX'], errors='coerce')
df_ifn2['CoorY'] = pd.to_numeric(df_ifn2['CoorY'], errors='coerce')
df_ifn3['CoorX'] = pd.to_numeric(df_ifn3['CoorX'], errors='coerce')
df_ifn3['CoorY'] = pd.to_numeric(df_ifn3['CoorY'], errors='coerce')
df_ifn4['CoorX'] = pd.to_numeric(df_ifn4['CoorX'], errors='coerce')
df_ifn4['CoorY'] = pd.to_numeric(df_ifn4['CoorY'], errors='coerce')

# Limpiar valores inf/nan de coordenadas
for d in [df_ifn2, df_ifn3, df_ifn4]:
    d["CoorX"] = d["CoorX"].replace([float("inf"), float("-inf")], pd.NA)
    d["CoorY"] = d["CoorY"].replace([float("inf"), float("-inf")], pd.NA)
    d.dropna(subset=["CoorX", "CoorY"], inplace=True)

# Inicializar la columna 'Presencia'
df_ifn2['Presencia'] = 2
df_ifn3['Presencia'] = 3
df_ifn4['Presencia'] = 4

# Hacer una fusión de los DataFrames por 'IDParcela' para comparar las parcelas
merged_ifn2_ifn3 = pd.merge(df_ifn2[['IDParcela', 'CoorX', 'CoorY']], df_ifn3[['IDParcela', 'CoorX', 'CoorY']], on='IDParcela', suffixes=('_ifn2', '_ifn3'))
merged_ifn3_ifn4 = pd.merge(df_ifn3[['IDParcela', 'CoorX', 'CoorY']], df_ifn4[['IDParcela', 'CoorX', 'CoorY']], on='IDParcela', suffixes=('_ifn3', '_ifn4'))
merged_ifn2_ifn4 = pd.merge(df_ifn2[['IDParcela', 'CoorX', 'CoorY']], df_ifn4[['IDParcela', 'CoorX', 'CoorY']], on='IDParcela', suffixes=('_ifn2', '_ifn4'))

# IFN2 y IFN3 coinciden si las diferencias en CoorX y CoorY son menores a 300 unidades
merged_ifn2_ifn3['diferencia_CoorX'] = abs(merged_ifn2_ifn3['CoorX_ifn2'] - merged_ifn2_ifn3['CoorX_ifn3'])
merged_ifn2_ifn3['diferencia_CoorY'] = abs(merged_ifn2_ifn3['CoorY_ifn2']  - merged_ifn2_ifn3['CoorY_ifn3'])

# Modificar 'Presencia' en ifn2 e ifn3 a 1 si coinciden
df_ifn2.loc[df_ifn2['IDParcela'].isin(merged_ifn2_ifn3[merged_ifn2_ifn3['diferencia_CoorX'] < 300]['IDParcela']), 'Presencia'] = 1
df_ifn3.loc[df_ifn3['IDParcela'].isin(merged_ifn2_ifn3[merged_ifn2_ifn3['diferencia_CoorX'] < 300]['IDParcela']), 'Presencia'] = 1

# IFN2 y IFN4 coinciden si las diferencias en CoorX y CoorY son menores a 300 unidades
merged_ifn2_ifn4['diferencia_CoorX'] = abs(merged_ifn2_ifn4['CoorX_ifn2'] - merged_ifn2_ifn4['CoorX_ifn4'])
merged_ifn2_ifn4['diferencia_CoorY'] = abs(merged_ifn2_ifn4['CoorY_ifn2'] - merged_ifn2_ifn4['CoorY_ifn4'])

# Modificar 'Presencia' en ifn2 e ifn4 a 1 si coinciden
df_ifn2.loc[df_ifn2['IDParcela'].isin(merged_ifn2_ifn4[merged_ifn2_ifn4['diferencia_CoorX'] < 300]['IDParcela']), 'Presencia'] = 1
df_ifn4.loc[df_ifn4['IDParcela'].isin(merged_ifn2_ifn4[merged_ifn2_ifn4['diferencia_CoorX'] < 300]['IDParcela']), 'Presencia'] = 1

# IFN3 y IFN4 coinciden si las diferencias en CoorX y CoorY son menores a 300 unidades
merged_ifn3_ifn4['diferencia_CoorX'] = abs(merged_ifn3_ifn4['CoorX_ifn3'] - merged_ifn3_ifn4['CoorX_ifn4'])
merged_ifn3_ifn4['diferencia_CoorY'] = abs(merged_ifn3_ifn4['CoorY_ifn3'] - merged_ifn3_ifn4['CoorY_ifn4'])

# Modificar 'Presencia' en ifn3 e ifn4 a 1 si coinciden
df_ifn3.loc[df_ifn3['IDParcela'].isin(merged_ifn3_ifn4[merged_ifn3_ifn4['diferencia_CoorX'] < 300]['IDParcela']), 'Presencia'] = 1
df_ifn4.loc[df_ifn4['IDParcela'].isin(merged_ifn3_ifn4[merged_ifn3_ifn4['diferencia_CoorX'] < 300]['IDParcela']), 'Presencia'] = 1

# Para el caso de ifn2 y ifn3 pero no en ifn4
df_ifn2.loc[df_ifn2['IDParcela'].isin(merged_ifn2_ifn3[merged_ifn2_ifn3['diferencia_CoorX'] < 300]['IDParcela']) & 
             ~df_ifn2['IDParcela'].isin(merged_ifn2_ifn4[merged_ifn2_ifn4['diferencia_CoorX'] < 300]['IDParcela']), 'Presencia'] = 5

# Para el caso de ifn2 y ifn4 pero no en ifn3
df_ifn2.loc[df_ifn2['IDParcela'].isin(merged_ifn2_ifn4[merged_ifn2_ifn4['diferencia_CoorX'] < 300]['IDParcela']) & 
             ~df_ifn2['IDParcela'].isin(merged_ifn2_ifn3[merged_ifn2_ifn3['diferencia_CoorX'] < 300]['IDParcela']), 'Presencia'] = 6

# Para el caso de ifn4 y ifn3 pero no en ifn2
df_ifn4.loc[df_ifn4['IDParcela'].isin(merged_ifn3_ifn4[merged_ifn3_ifn4['diferencia_CoorX'] < 300]['IDParcela']) & 
             ~df_ifn4['IDParcela'].isin(merged_ifn2_ifn4[merged_ifn2_ifn4['diferencia_CoorX'] < 300]['IDParcela']), 'Presencia'] = 7

# Guardar los archivos actualizados con la columna 'Presencia' en sus respectivos archivos
df_ifn2.to_csv('IFN_España/IFN2/ifn2_DATEST.csv', index=False)
df_ifn3.to_csv('IFN_España/IFN3_campo/ifn3_campo_PCDatosMap.csv', index=False)
df_ifn4.to_csv('IFN_España/IFN4_campo/ifn4_campo_PCDatosMap.csv', index=False)

print("Proceso completado. Los archivos han sido actualizados con la columna 'Presencia'.")



/tmp/ipykernel_2004326/723971217.py:9: DtypeWarning: Columns (0: CoorX, 1: CoorY, 2: FRACCION1, 3: CUBIERTA, 4: MAXPEND1, 5: MAXPEND2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ifn2 = pd.read_csv(archivo_ifn2)


Proceso completado. Los archivos han sido actualizados con la columna 'Presencia'.


Trato PCRegenera y lo llevo a una tabla llamada Brotes: 

·PCRegenera contiene: Provincia	Estadillo	Especie	Cla	Subclase	CatDes	Tipo	Densidad	NumPies	Hm	IDParcela.
·Brotes tendrá: Provincia	Estadillo	IDParcela Especie CD y NPies

 Si CatDes\in{1,2} CD=1, si CatDes=3 CD=2, si CatDes=4 CD=5 y NPies=NumPies; si  Densidad=1 NPies=2.5, si Densidad=2 NPies=10 y si  Densidad=3 NPies=15.

In [40]:
import pandas as pd

# Rutas de los archivos IFN3 e IFN4 originales
archivo_ifn3_pc_regenera = 'IFN_España/IFN3_campo/ifn3_campo_PCRegenera.csv'
archivo_ifn4_pc_regenera = 'IFN_España/IFN4_campo/ifn4_campo_PCRegenera.csv'

# Rutas para guardar los archivos modificados
archivo_ifn3_brotes = 'IFN_España/IFN3_campo/ifn3_campo_Brotes.csv'
archivo_ifn4_brotes = 'IFN_España/IFN4_campo/ifn4_campo_Brotes.csv'

# Leer los archivos CSV
df_ifn3 = pd.read_csv(archivo_ifn3_pc_regenera)
df_ifn4 = pd.read_csv(archivo_ifn4_pc_regenera)


# Función para crear los "Brotes" a partir de la tabla PCRegenera
def crear_brotes(df_regenera):
    df = df_regenera.copy()
    
    # Mapear CatDes a CD
    def map_cd(catdes):
        if catdes in [1, 2]:
            return 1
        elif catdes == 3:
            return 2
        elif catdes == 4:
            return 5
        else:
            return None
    
    df['CD'] = df['CatDes'].apply(map_cd)

    # Mapear Densidad a NPies, si Densidad está en {1,2,3} sobrescribe NPies
    def map_npies(row):
        if row['Densidad'] == 1:
            return 2.5
        elif row['Densidad'] == 2:
            return 10
        elif row['Densidad'] == 3:
            return 15
        else:
            return row['NumPies']

    df['NPies'] = df.apply(map_npies, axis=1)

    # Seleccionar columnas para Brotes
    cols_brotes = ['Provincia', 'Estadillo', 'IDParcela', 'Especie', 'CD', 'NPies']

    brotes = df[cols_brotes].copy()
    return brotes

# Crear tabla Brotes para IFN3 si existe la columna PCRegenera
brotes_ifn3 = None
if 'CatDes' in df_ifn3.columns and 'Densidad' in df_ifn3.columns:
    brotes_ifn3 = crear_brotes(df_ifn3)
else:
    print("No existe la columna 'PCRegenera' o 'CatDes' en IFN3.")

# Crear tabla Brotes para IFN4 si existe la columna PCRegenera
brotes_ifn4 = None
if 'CatDes' in df_ifn4.columns and 'Densidad' in df_ifn4.columns:
    brotes_ifn4 = crear_brotes(df_ifn4)
else:
    print("No existe la columna 'PCRegenera' o 'CatDes' en IFN4.")

# Guardar los archivos nuevos con la tabla 'Brotes', si fueron creados
if brotes_ifn3 is not None:
    brotes_ifn3.to_csv(archivo_ifn3_brotes, index=False)
    print("El archivo 'ifn3_campo_Brotes.csv' ha sido guardado correctamente.")
else:
    print("No se pudo crear el archivo 'ifn3_campo_Brotes.csv'.")

if brotes_ifn4 is not None:
    brotes_ifn4.to_csv(archivo_ifn4_brotes, index=False)
    print("El archivo 'ifn4_campo_Brotes.csv' ha sido guardado correctamente.")
else:
    print("No se pudo crear el archivo 'ifn4_campo_Brotes.csv'.")


/tmp/ipykernel_2004326/485576311.py:12: DtypeWarning: Columns (0: Especie) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ifn3 = pd.read_csv(archivo_ifn3_pc_regenera)


El archivo 'ifn3_campo_Brotes.csv' ha sido guardado correctamente.


El archivo 'ifn4_campo_Brotes.csv' ha sido guardado correctamente.


Creamos un diccionario para normalizar las especies del IFN3 e IFN4 (no tengo datos del 2) y añado a dichos inventarios una columna EspecieNormalizada

In [41]:

import pandas as pd

# Rutas de los archivos a procesar
archivo_ifn3 = 'IFN_España/IFN3_sig/ifn3_sig_CambioEspecie.csv'
archivo_ifn4 = 'IFN_España/IFN4_sig/ifn4_sig_CambioEspecie.csv'

# Cargar los archivos CSV
df3 = pd.read_csv(archivo_ifn3)
df4 = pd.read_csv(archivo_ifn4)

# Agrupar por EspOriginal y obtener listas únicas de EspCambiada
mapa3 = df3.groupby('EspCambiada')['EspOriginal'].apply(lambda x: sorted(set(x)))
mapa4 = df4.groupby('EspOriginal')['EspCambiada'].apply(lambda x: sorted(set(x)))

# Obtener todas las claves únicas
todas_claves = sorted(set(mapa3.index).union(set(mapa4.index)))

# Construir nuevo DataFrame
resultado = pd.DataFrame({
    'clave': todas_claves,
    'valores3': [mapa3.get(k, []) for k in todas_claves],
    'valores4': [mapa4.get(k, []) for k in todas_claves]
})

# Guardar a nuevo Excel
ruta_salida = 'IFN_España/resumen_cambios_especie.csv'
resultado.to_csv(ruta_salida, index=False)

print(f"Archivo generado: {ruta_salida}")

Archivo generado: IFN_España/resumen_cambios_especie.csv


In [42]:
import pandas as pd
import os

# Rutas del archivo de cambios y directorios
archivo_resumen_cambios = 'IFN_España/resumen_cambios_especie.csv'
directorios = [
    'IFN_España/IFN3_sig',
    'IFN_España/IFN3_campo',
    'IFN_España/IFN4_campo',
    'IFN_España/IFN4_sig'
]

# Leer el archivo resumen de cambios
df_resumen = pd.read_csv(archivo_resumen_cambios)

# Asegurarse de que 'valores3' y 'valores4' sean listas de valores únicos
valores3_set = set(df_resumen['valores3'].dropna())
valores4_set = set(df_resumen['valores4'].dropna())

# Recorrer cada directorio y archivo CSV
for directorio in directorios:
    archivos_csv = [f for f in os.listdir(directorio) if f.endswith('.csv')]
    
    for archivo in archivos_csv:
        archivo_path = os.path.join(directorio, archivo)
        
        # Cargar el archivo CSV
        df = pd.read_csv(archivo_path)
        
        # Verificar si la columna 'Especie' está presente
        if 'Especie' in df.columns:
            # Crear una nueva columna 'EspecieNormalizada'
            def normalizar_especie(row):
                especie = row['Especie']
                
                # Determinar si es un archivo de IFN3 o IFN4
                if 'IFN3' in directorio:
                    # Usar 'valores3' de resumen
                    if especie in valores3_set:
                        return df_resumen[df_resumen['valores3'] == especie]['clave'].values[0]
                    else:
                        return especie  # Si no está en 'valores3', mantener el valor original
                elif 'IFN4' in directorio:
                    # Usar 'valores4' de resumen
                    if especie in valores4_set:
                        return df_resumen[df_resumen['valores4'] == especie]['clave'].values[0]
                    else:
                        return especie  # Si no está en 'valores4', mantener el valor original
                else:
                    return especie  # Si no es IFN3 o IFN4, no hacer cambios

            df['EspecieNormalizada'] = df.apply(normalizar_especie, axis=1)

            # Guardar el archivo con la nueva columna
            df.to_csv(archivo_path, index=False)
            print(f"Columna 'EspecieNormalizada' añadida en: {archivo}")

print("Proceso completado.")


Columna 'EspecieNormalizada' añadida en: ifn3_sig_Mayores_exs.csv


/tmp/ipykernel_2004326/2101612356.py:28: DtypeWarning: Columns (0: LimProv2, 1: UG) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo_path)


Columna 'EspecieNormalizada' añadida en: ifn3_sig_Parcelas_exs.csv


Columna 'EspecieNormalizada' añadida en: ifn3_sig_Estratos_exs.csv


/tmp/ipykernel_2004326/2101612356.py:28: DtypeWarning: Columns (0: Especie) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo_path)


Columna 'EspecieNormalizada' añadida en: ifn3_campo_PCRegenera.csv
Columna 'EspecieNormalizada' añadida en: ifn3_campo_PCTablaEsp.csv


... (20 outputs left out)

Columna 'EspecieNormalizada' añadida en: ifn4_sig_Parcelas_exs.csv


Columna 'EspecieNormalizada' añadida en: ifn4_sig_Estratos_exs.csv


Proceso completado.


/tmp/ipykernel_2004326/2101612356.py:28: DtypeWarning: Columns (0: N5, 1: TipoBosque, 2: CompEsp) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo_path)


Entreno un modelo para rellenar CA y CR de IFN4 y posteriormente crear dichas variables en IFN3

In [43]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

# Cargar el archivo CSV
archivo = 'IFN_España/IFN4_sig/ifn4_sig_Parcelas_exs.csv'
df = pd.read_csv(archivo)

# Verificación de las columnas con valores faltantes para 'CA' y 'CR'
faltantes_ca = df['CA'].isnull().sum()
faltantes_cr = df['CR'].isnull().sum()

# Cálculo del porcentaje de faltantes
total_filas = len(df)
porcentaje_faltantes_ca = (faltantes_ca / total_filas) * 100
porcentaje_faltantes_cr = (faltantes_cr / total_filas) * 100

print(f"Faltantes de CA: {faltantes_ca} valores ({porcentaje_faltantes_ca:.2f}%)")
print(f"Faltantes de CR: {faltantes_cr} valores ({porcentaje_faltantes_cr:.2f}%)")

# Eliminar filas con valores faltantes en 'CA' o 'CR'
df_limpio = df.dropna(subset=['CA', 'CR'])

# Variables predictoras
variables_predictoras = ['EspecieNormalizada', 'Estrato', 'CD', 'VSC', 'NPies', 'ABas', 'IAVC', 'VCC', 'VLE']

# Limpiar datos: eliminar filas con valores faltantes en las variables predictoras
df_limpio = df_limpio.dropna(subset=variables_predictoras)

# Separa los datos para predecir CA y CR
X = df_limpio[variables_predictoras]
y_ca = df_limpio['CA']
y_cr = df_limpio['CR']

# Dividir los datos en conjunto de entrenamiento y prueba
X_train, X_test, y_train_ca, y_test_ca = train_test_split(X, y_ca, test_size=0.2, random_state=42)
X_train, X_test, y_train_cr, y_test_cr = train_test_split(X, y_cr, test_size=0.2, random_state=42)

# Modelo para CA - Usaremos un RandomForestRegressor como ejemplo de modelo predictivo
modelo_ca = RandomForestRegressor(n_estimators=100, random_state=42)
modelo_ca.fit(X_train, y_train_ca)

# Predicciones y evaluación del modelo de CA
predicciones_ca_train = modelo_ca.predict(X_train)
predicciones_ca_test = modelo_ca.predict(X_test)

# Métricas para CA
mse_train_ca = mean_squared_error(y_train_ca, predicciones_ca_train)
mse_test_ca = mean_squared_error(y_test_ca, predicciones_ca_test)
r2_train_ca = r2_score(y_train_ca, predicciones_ca_train)
r2_test_ca = r2_score(y_test_ca, predicciones_ca_test)

print(f"\nEvaluación del modelo para CA:")
print(f"MSE en entrenamiento: {mse_train_ca:.2f}")
print(f"MSE en prueba: {mse_test_ca:.2f}")
print(f"R2 en entrenamiento: {r2_train_ca:.2f}")
print(f"R2 en prueba: {r2_test_ca:.2f}")

# Modelo para CR - Usaremos el mismo modelo RandomForestRegressor para CR
modelo_cr = RandomForestRegressor(n_estimators=100, random_state=42)
modelo_cr.fit(X_train, y_train_cr)

# Predicciones y evaluación del modelo de CR
predicciones_cr_train = modelo_cr.predict(X_train)
predicciones_cr_test = modelo_cr.predict(X_test)

# Métricas para CR
mse_train_cr = mean_squared_error(y_train_cr, predicciones_cr_train)
mse_test_cr = mean_squared_error(y_test_cr, predicciones_cr_test)
r2_train_cr = r2_score(y_train_cr, predicciones_cr_train)
r2_test_cr = r2_score(y_test_cr, predicciones_cr_test)

print(f"\nEvaluación del modelo para CR:")
print(f"MSE en entrenamiento: {mse_train_cr:.2f}")
print(f"MSE en prueba: {mse_test_cr:.2f}")
print(f"R2 en entrenamiento: {r2_train_cr:.2f}")
print(f"R2 en prueba: {r2_test_cr:.2f}")

# Importancia de las variables en el modelo de CA
importancia_variables_ca = modelo_ca.feature_importances_

# Mostrar la importancia de las variables
print("\nImportancia de las variables para CA:")
for i, col in enumerate(variables_predictoras):
    print(f"{col}: {importancia_variables_ca[i]:.4f}")

# Importancia de las variables en el modelo de CR
importancia_variables_cr = modelo_cr.feature_importances_

# Mostrar la importancia de las variables
print("\nImportancia de las variables para CR:")
for i, col in enumerate(variables_predictoras):
    print(f"{col}: {importancia_variables_cr[i]:.4f}")

# Visualización de la importancia de las variables
plt.figure(figsize=(12, 6))

# Importancia para CA
plt.subplot(1, 2, 1)
plt.barh(variables_predictoras, importancia_variables_ca)
plt.title("Importancia de las Variables para CA")
plt.xlabel("Importancia")

# Importancia para CR
plt.subplot(1, 2, 2)
plt.barh(variables_predictoras, importancia_variables_cr)
plt.title("Importancia de las Variables para CR")
plt.xlabel("Importancia")

plt.tight_layout()
plt.show()


Faltantes de CA: 134045 valores (41.74%)
Faltantes de CR: 134045 valores (41.74%)



Evaluación del modelo para CA:
MSE en entrenamiento: 0.49
MSE en prueba: 2.41
R2 en entrenamiento: 0.99
R2 en prueba: 0.95



Evaluación del modelo para CR:
MSE en entrenamiento: 0.26
MSE en prueba: 4.74
R2 en entrenamiento: 0.99
R2 en prueba: 0.68

Importancia de las variables para CA:
EspecieNormalizada: 0.0310
Estrato: 0.0075
CD: 0.0344
VSC: 0.0452
NPies: 0.0107
ABas: 0.7727
IAVC: 0.0094
VCC: 0.0232
VLE: 0.0659

Importancia de las variables para CR:
EspecieNormalizada: 0.1300
Estrato: 0.0103
CD: 0.0065
VSC: 0.0142
NPies: 0.0362
ABas: 0.3357
IAVC: 0.0236
VCC: 0.0170
VLE: 0.4266


/tmp/ipykernel_2004326/3861681437.py:113: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [44]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# Cargar los archivos CSV de IFN3 y IFN4
archivo_ifn3 = 'IFN_España/IFN3_sig/ifn3_sig_Parcelas_exs.csv'
archivo_ifn4 = 'IFN_España/IFN4_sig/ifn4_sig_Parcelas_exs.csv'

df_ifn3 = pd.read_csv(archivo_ifn3)
df_ifn4 = pd.read_csv(archivo_ifn4)

# Variables predictoras comunes
variables_predictoras = ['EspecieNormalizada', 'Estrato', 'CD', 'VSC', 'NPies', 'ABas', 'IAVC', 'VCC', 'VLE']

# Limpiar los datos de IFN4 eliminando filas con valores faltantes en CA y CR
df_ifn4_limpio = df_ifn4.dropna(subset=variables_predictoras + ['CA', 'CR'])

# Variables predictoras y objetivo para entrenar el modelo en IFN4
X_ifn4 = df_ifn4_limpio[variables_predictoras]
y_ca_ifn4 = df_ifn4_limpio['CA']
y_cr_ifn4 = df_ifn4_limpio['CR']

# Entrenar el modelo RandomForestRegressor con los datos de IFN4
modelo_ca_ifn4 = RandomForestRegressor(n_estimators=100, random_state=42)
modelo_ca_ifn4.fit(X_ifn4, y_ca_ifn4)

modelo_cr_ifn4 = RandomForestRegressor(n_estimators=100, random_state=42)
modelo_cr_ifn4.fit(X_ifn4, y_cr_ifn4)

# Tarea 1: Rellenar los faltantes de CA y CR en IFN4
# Seleccionamos las filas con valores faltantes en CA y CR
faltantes_ifn4 = df_ifn4[df_ifn4['CA'].isnull() | df_ifn4['CR'].isnull()]

# Variables predictoras para las filas con valores faltantes en IFN4
X_faltantes_ifn4 = faltantes_ifn4[variables_predictoras]

# Predicciones para las filas con valores faltantes
predicciones_ca_faltantes = modelo_ca_ifn4.predict(X_faltantes_ifn4)
predicciones_cr_faltantes = modelo_cr_ifn4.predict(X_faltantes_ifn4)

# Rellenar los valores faltantes de CA y CR con las predicciones
df_ifn4.loc[df_ifn4['CA'].isnull(), 'CA'] = predicciones_ca_faltantes
df_ifn4.loc[df_ifn4['CR'].isnull(), 'CR'] = predicciones_cr_faltantes

# Guardar el archivo IFN4 con los valores rellenados
df_ifn4.to_csv('IFN_España/IFN4_sig/ifn4_sig_Parcelas_exs.csv', index=False)
print("Los valores faltantes de CA y CR en IFN4 han sido rellenados correctamente.")

# Tarea 2: Crear las variables CA y CR en IFN3
# Variables predictoras para las predicciones en IFN3
X_ifn3 = df_ifn3[variables_predictoras]

# Predicciones para IFN3 usando el modelo entrenado con IFN4
predicciones_ca_ifn3 = modelo_ca_ifn4.predict(X_ifn3)
predicciones_cr_ifn3 = modelo_cr_ifn4.predict(X_ifn3)

# Añadir las predicciones como nuevas columnas en el DataFrame de IFN3
df_ifn3['CA'] = predicciones_ca_ifn3
df_ifn3['CR'] = predicciones_cr_ifn3

# Guardar el archivo IFN3 con las nuevas columnas CA y CR
df_ifn3.to_csv('IFN_España/IFN3_sig/ifn3_sig_Parcelas_exs.csv', index=False)
print("Las columnas CA y CR han sido añadidas a IFN3 correctamente.")


Los valores faltantes de CA y CR en IFN4 han sido rellenados correctamente.


Las columnas CA y CR han sido añadidas a IFN3 correctamente.


In [45]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# Cargar los archivos CSV de IFN2 y IFN4
archivo_ifn2 = 'IFN_España/IFN2/ifn2_IIFL03BD.csv'
archivo_ifn4 = 'IFN_España/IFN4_sig/ifn4_sig_Parcelas_exs.csv'

df_ifn2 = pd.read_csv(archivo_ifn2)
df_ifn4 = pd.read_csv(archivo_ifn4)

# Variables predictoras comunes
variables_predictoras = ['Especie', 'CD', 'VSC', 'NPies', 'ABas', 'IAVC', 'VCC', 'VLE']

# Limpiar los datos de IFN4 eliminando filas con valores faltantes en CA y CR
df_ifn4_limpio = df_ifn4.dropna(subset=variables_predictoras + ['CA', 'CR'])

# Variables predictoras y objetivo para entrenar el modelo en IFN4
X_ifn4 = df_ifn4_limpio[variables_predictoras]
y_ca_ifn4 = df_ifn4_limpio['CA']
y_cr_ifn4 = df_ifn4_limpio['CR']

# Entrenar el modelo RandomForestRegressor con los datos de IFN4
modelo_ca_ifn4 = RandomForestRegressor(n_estimators=100, random_state=42)
modelo_ca_ifn4.fit(X_ifn4, y_ca_ifn4)

modelo_cr_ifn4 = RandomForestRegressor(n_estimators=100, random_state=42)
modelo_cr_ifn4.fit(X_ifn4, y_cr_ifn4)

# Verificar que las columnas coinciden entre el modelo y los datos de IFN2
X_ifn2 = df_ifn2[variables_predictoras]

# Comprobar que no hay filas vacías o mal alineadas en los datos de IFN2
X_ifn2_cleaned = X_ifn2.dropna(subset=variables_predictoras)

# Predicciones para IFN2 usando el modelo entrenado con IFN4
predicciones_ca_ifn2 = modelo_ca_ifn4.predict(X_ifn2_cleaned)
predicciones_cr_ifn2 = modelo_cr_ifn4.predict(X_ifn2_cleaned)

# Añadir las predicciones como nuevas columnas en el DataFrame de IFN2 (solo para las filas correspondientes)
df_ifn2.loc[X_ifn2_cleaned.index, 'CA'] = predicciones_ca_ifn2
df_ifn2.loc[X_ifn2_cleaned.index, 'CR'] = predicciones_cr_ifn2

# Guardar el archivo IFN2 con las nuevas columnas CA y CR
df_ifn2.to_csv('IFN_España/IFN2/ifn2_IIFL03BD.csv', index=False)
print("Las columnas CA y CR han sido añadidas a IFN2 correctamente.")


Las columnas CA y CR han sido añadidas a IFN2 correctamente.


Quiero extraer las ubicaciones para ir descargando los datos 

In [46]:
import pandas as pd

# Rutas de los archivos
ifn4_pcdatosmap = 'IFN_España/IFN4_campo/ifn4_campo_PCDatosMap.csv'
ifn3_pcdatosmap = 'IFN_España/IFN3_campo/ifn3_campo_PCDatosMap.csv'
ifn2_datedst = 'IFN_España/IFN2/ifn2_DATEST.csv'

# Leer los archivos CSV
df_ifn4_pcdatosmap = pd.read_csv(ifn4_pcdatosmap)
df_ifn3_pcdatosmap = pd.read_csv(ifn3_pcdatosmap)
df_ifn2_datedst = pd.read_csv(ifn2_datedst)

# Filtrar las columnas necesarias de IFN4
df_ifn4_ubicaciones = df_ifn4_pcdatosmap[['Provincia', 'IDParcela', 'CoorX', 'CoorY', 'Huso', 'Presencia']].copy()

# Filtrar las filas con 'Presencia' == 1, 4, 6, 7 y las correspondientes coordenadas
df_ifn4_ubicaciones = df_ifn4_ubicaciones[df_ifn4_ubicaciones['Presencia'].isin([1, 4, 6, 7])]

# Filtrar las filas con 'Presencia' == 3, 5 de IFN3 y añadirlas
df_ifn3_ubicaciones = df_ifn3_pcdatosmap[df_ifn3_pcdatosmap['Presencia'].isin([3, 5])][['Provincia', 'IDParcela', 'CoorX', 'CoorY', 'Presencia']].copy()

# Filtrar las filas con 'Presencia' == 2 de IFN2 y añadirlas
df_ifn2_ubicaciones = df_ifn2_datedst[df_ifn2_datedst['Presencia'] == 2][['Provincia', 'IDParcela', 'CoorX', 'CoorY', 'Presencia']].copy()

# Concatenar los DataFrames de IFN4, IFN3 y IFN2
final_df = pd.concat([df_ifn4_ubicaciones, df_ifn3_ubicaciones, df_ifn2_ubicaciones], ignore_index=True)

# Para las entradas con 'Presencia' en {1, 4, 6, 7}, tomar las coordenadas de IFN4
final_df.loc[final_df['Presencia'].isin([1, 4, 6, 7]), 'CoorX'] = final_df.loc[final_df['Presencia'].isin([1, 4, 6, 7]), 'CoorX']
final_df.loc[final_df['Presencia'].isin([1, 4, 6, 7]), 'CoorY'] = final_df.loc[final_df['Presencia'].isin([1, 4, 6, 7]), 'CoorY']
final_df.loc[final_df['Presencia'].isin([1, 4, 6, 7]), 'Huso'] = final_df.loc[final_df['Presencia'].isin([1, 4, 6, 7]), 'Huso']

# Para las entradas con 'Presencia' en {3, 5}, tomar las coordenadas de IFN3
final_df.loc[final_df['Presencia'].isin([3, 5]), 'CoorX'] = final_df.loc[final_df['Presencia'].isin([3, 5]), 'CoorX']
final_df.loc[final_df['Presencia'].isin([3, 5]), 'CoorY'] = final_df.loc[final_df['Presencia'].isin([3, 5]), 'CoorY']

# Para las entradas con 'Presencia' == 2, tomar las coordenadas de IFN2
final_df.loc[final_df['Presencia'] == 2, 'CoorX'] = final_df.loc[final_df['Presencia'] == 2, 'CoorX']
final_df.loc[final_df['Presencia'] == 2, 'CoorY'] = final_df.loc[final_df['Presencia'] == 2, 'CoorY']

# Rellenar los valores faltantes de 'Huso' con la moda de cada 'Provincia'
huso_moda = final_df.groupby('Provincia')['Huso'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
final_df['Huso'] = final_df.apply(
    lambda row: row['Huso'] if pd.notnull(row['Huso']) else huso_moda.get(row['Provincia'], None), axis=1
)

# Eliminar IDParcela duplicados
final_df = final_df.drop_duplicates(subset=['IDParcela'])

# Guardar el archivo CSV final
ubicaciones_csv = 'IFN_España/ubicaciones.csv'
final_df.to_csv(ubicaciones_csv, index=False)

print(f"Archivo 'ubicaciones.csv' guardado correctamente en: {ubicaciones_csv}")


/tmp/ipykernel_2004326/2487258416.py:10: DtypeWarning: Columns (0: Hoja50) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ifn3_pcdatosmap = pd.read_csv(ifn3_pcdatosmap)


Archivo 'ubicaciones.csv' guardado correctamente en: IFN_España/ubicaciones.csv


diagnostico de nan en las coordenadas

In [47]:
import pandas as pd

# Rutas de los archivos
ifn4_pcdatosmap = 'IFN_España/IFN4_campo/ifn4_campo_PCDatosMap.csv'
ifn3_pcdatosmap = 'IFN_España/IFN3_campo/ifn3_campo_PCDatosMap.csv'
ifn2_datedst = r'IFN_España/IFN2/ifn2_DATEST.csv'
ubicaciones = 'IFN_España/ubicaciones.csv'
# Leer los archivos CSV
df_ifn4_pcdatosmap = pd.read_csv(ifn4_pcdatosmap)
df_ifn3_pcdatosmap = pd.read_csv(ifn3_pcdatosmap)
df_ifn2_datedst = pd.read_csv(ifn2_datedst)
df_ubicaciones = pd.read_csv(ubicaciones)
# Contar los valores NaN en las columnas 'CoorX' y 'CoorY' para cada archivo
nan_count_ifn4 = df_ifn4_pcdatosmap[['CoorX', 'CoorY']].isna().sum()
nan_count_ifn3 = df_ifn3_pcdatosmap[['CoorX', 'CoorY']].isna().sum()
nan_count_ifn2 = df_ifn2_datedst[['CoorX', 'CoorY']].isna().sum()
nan_count_ubicaciones = df_ubicaciones[['CoorX', 'CoorY']].isna().sum()
# Crear un DataFrame para guardar el diagnóstico
diagnostico_df = pd.DataFrame({
    'Archivo': ['ifn4_pcdatosmap', 'ifn3_pcdatosmap', 'ifn2_datedst','ubicaciones'],
    'CoorX_Nan': [nan_count_ifn4['CoorX'], nan_count_ifn3['CoorX'], nan_count_ifn2['CoorX'],nan_count_ubicaciones['CoorX']],
    'CoorY_Nan': [nan_count_ifn4['CoorY'], nan_count_ifn3['CoorY'], nan_count_ifn2['CoorY'],nan_count_ubicaciones['CoorY']]
})

# Mostrar los resultados en consola
print("Conteo de valores NaN en CoorX y CoorY de cada archivo:")
print(diagnostico_df)


Conteo de valores NaN en CoorX y CoorY de cada archivo:
           Archivo  CoorX_Nan  CoorY_Nan
0  ifn4_pcdatosmap          0          0
1  ifn3_pcdatosmap          0          0
2     ifn2_datedst          0          0
3      ubicaciones          0          0


/tmp/ipykernel_2004326/2150484686.py:10: DtypeWarning: Columns (0: Hoja50) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ifn3_pcdatosmap = pd.read_csv(ifn3_pcdatosmap)


tambien quiero los años de cada inventario para las descargas

elimino primero algunos valores no deseados del ifn3, en ifn2 ya los habia eliminado. En ifn4 son razonables

In [48]:
import pandas as pd

# Ruta del archivo
archivo_ubicaciones = 'IFN_España/IFN3_campo/ifn3_campo_PCParcelas.csv'

# Cargar el archivo CSV
df_ubicaciones = pd.read_csv(archivo_ubicaciones)
df_ubicaciones['Ano'] = pd.to_numeric(df_ubicaciones['Ano'], errors='coerce')

# Filtrar y eliminar las filas con los valores especificados en 'Ano'
valores_a_eliminar =  [0.0, 2051.0, 1980.0, 1986.0, 1980, 1992.0]

# Eliminar las filas en las que 'Ano_ifn3' tenga alguno de los valores especificados
df_ubicaciones_filtrado = df_ubicaciones[~df_ubicaciones['Ano'].isin(valores_a_eliminar)]

# Guardar el archivo filtrado
archivo_filtrado = 'IFN_España/IFN3_campo/ifn3_campo_PCParcelas.csv'
df_ubicaciones_filtrado.to_csv(archivo_filtrado, index=False)

print(f"Archivo filtrado guardado en: {archivo_filtrado}")


/tmp/ipykernel_2004326/620533546.py:7: DtypeWarning: Columns (0: Ano, 1: Nivel2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ubicaciones = pd.read_csv(archivo_ubicaciones)


Archivo filtrado guardado en: IFN_España/IFN3_campo/ifn3_campo_PCParcelas.csv


In [49]:
import pandas as pd

# Rutas de los archivos
ifn4_pcparcelas = 'IFN_España/IFN4_campo/ifn4_campo_PCParcelas.csv'
ifn3_pcparcelas = 'IFN_España/IFN3_campo/ifn3_campo_PCParcelas.csv'
ifn2_datedst = 'IFN_España/IFN2/ifn2_DATEST.csv'

# Leer los archivos CSV
df_ifn4_pcparcelas = pd.read_csv(ifn4_pcparcelas)
df_ifn3_pcparcelas = pd.read_csv(ifn3_pcparcelas)
df_ifn2_datedst = pd.read_csv(ifn2_datedst)

# Filtrar las columnas necesarias de cada archivo (sólo la columna 'IDParcela' y 'Ano')
df_ifn4_ano = df_ifn4_pcparcelas[['IDParcela', 'Ano']].copy()
df_ifn3_ano = df_ifn3_pcparcelas[['IDParcela', 'Ano']].copy()
df_ifn2_ano = df_ifn2_datedst[['IDParcela', 'Ano']].copy()

# Leer el archivo de ubicaciones creado previamente
ubicaciones_csv = 'IFN_España/ubicaciones.csv'
final_df = pd.read_csv(ubicaciones_csv)

# Función segura para asignar valores de Ano
def get_ano_for_id(row, df_ano, presencias, column='Ano'):
    if row['Presencia'] in presencias:
        matching_rows = df_ano[df_ano['IDParcela'] == row['IDParcela']]
        if not matching_rows.empty:
            return matching_rows[column].values[0]
    return None

# Añadir Ano_ifn4, Ano_ifn3 y Ano_ifn2 según las condiciones de 'Presencia'
final_df['Ano_ifn4'] = final_df.apply(
    lambda row: get_ano_for_id(row, df_ifn4_ano, [1, 4, 6, 7], 'Ano'), axis=1
)

final_df['Ano_ifn3'] = final_df.apply(
    lambda row: get_ano_for_id(row, df_ifn3_ano, [1, 3, 5, 7], 'Ano'), axis=1
)

final_df['Ano_ifn2'] = final_df.apply(
    lambda row: get_ano_for_id(row, df_ifn2_ano, [1, 2, 5, 6], 'Ano'), axis=1
)

# Guardar el archivo final con las nuevas columnas añadidas
final_df.to_csv(ubicaciones_csv, index=False)

print(f"Archivo 'ubicaciones.csv' con las columnas Ano_ifn4, Ano_ifn3, Ano_ifn2 guardado correctamente en: {ubicaciones_csv}")


/tmp/ipykernel_2004326/1379680023.py:10: DtypeWarning: Columns (0: Nivel2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ifn3_pcparcelas = pd.read_csv(ifn3_pcparcelas)


Archivo 'ubicaciones.csv' con las columnas Ano_ifn4, Ano_ifn3, Ano_ifn2 guardado correctamente en: IFN_España/ubicaciones.csv


In [50]:
import pandas as pd

# Ruta del archivo
archivo_ubicaciones = 'IFN_España/ubicaciones.csv'

# Cargar el archivo CSV
df_ubicaciones = pd.read_csv(archivo_ubicaciones)

# Contar los valores de las columnas 'Ano_ifn4', 'Ano_ifn3', 'Ano_ifn2'
conteo_ano_ifn4 = df_ubicaciones['Ano_ifn4'].value_counts()
conteo_ano_ifn3 = df_ubicaciones['Ano_ifn3'].value_counts()
conteo_ano_ifn2 = df_ubicaciones['Ano_ifn2'].value_counts()

# Mostrar los conteos
print("Conteo de valores para 'Ano_ifn4':")
print(conteo_ano_ifn4)

print("\nConteo de valores para 'Ano_ifn3':")
print(conteo_ano_ifn3)

print("\nConteo de valores para 'Ano_ifn2':")
print(conteo_ano_ifn2)

Conteo de valores para 'Ano_ifn4':
Ano_ifn4
2009.0    9331
2019.0    8492
2020.0    5911
2023.0    5038
2010.0    5008
2008.0    4084
2018.0    2927
2017.0    2863
2016.0    1863
2012.0    1629
2014.0    1576
2011.0    1424
2021.0    1328
2015.0    1223
2013.0     422
2005.0      57
2002.0       1
Name: count, dtype: int64

Conteo de valores para 'Ano_ifn3':
Ano_ifn3
2003.0    10341
2004.0     9622
2001.0     9144
2002.0     9121
2000.0     7140
1999.0     7057
2005.0     6967
2007.0     6812
1998.0     6399
2006.0     6180
1997.0     2479
2008.0      322
1989.0       46
Name: count, dtype: int64

Conteo de valores para 'Ano_ifn2':
Ano_ifn2
1986.0    2440
1992.0    1936
1988.0    1207
1990.0     774
1994.0     592
1991.0     573
1996.0     330
1995.0     126
1987.0       5
1980.0       2
1982.0       2
1989.0       2
1985.0       1
1999.0       1
1998.0       1
Name: count, dtype: int64


CORRECCIONES EN EL HUSO 
identifico los limites en longitud provinciales y luego tomo como parcelas mal proyectadas aquellas que estan fuera del rango de su provincia
me descagro un shapefile con los limites provinciales de españa

In [51]:
import pandas as pd

# Cargar el archivo CSV
df = pd.read_csv('IFN_España/ubicaciones.csv')

# Filtrar las filas con Huso = 0 o NaN
filtro_huso_0_o_nan = df['Huso'].isna() | (df['Huso'] == 0)

# Provincias a modificar según las reglas
modificaciones_huso = {
    4: 30,  # Almería
    11: 30,  # Cádiz
    14: 30,  # Córdoba
    18: 30,  # Granada
    21: 29,  # Huelva
    23: 30,  # Jaén
    29: 30,  # Málaga
    41:30, #cadiz
    50:30 #Zaragoza
}

# Aplicar las modificaciones de Huso en las filas correspondientes
for provincia, huso in modificaciones_huso.items():
    df.loc[(df['Provincia'] == provincia) & filtro_huso_0_o_nan, 'Huso'] = huso

# Verificar los cambios
print(df[df['Provincia'].isin(modificaciones_huso.keys()) & filtro_huso_0_o_nan][['Provincia', 'Huso']])

# Guardar el DataFrame modificado
df.to_csv('IFN_España/ubicaciones.csv', index=False)

print("Archivo con las modificaciones guardado en: data/raw/IFN_España/ubicaciones_modificadas.csv")


        Provincia  Huso
53900          18  30.0
53901          18  30.0
53902          18  30.0
53903          18  30.0
53904          18  30.0
...           ...   ...
100890         21  29.0
100891         21  29.0
100892         21  29.0
100893         21  29.0
100894         21  29.0

[6758 rows x 2 columns]


Archivo con las modificaciones guardado en: data/raw/IFN_España/ubicaciones_modificadas.csv


¿hay otras con el huso mal?

In [52]:
import pandas as pd

# Cargar el archivo CSV
df = pd.read_csv('IFN_España/ubicaciones.csv')

# Filtrar las filas donde 'CoorX' y 'CoorY' no son nulas y son diferentes de 0
filtro_coordenadas = (df['CoorX'].notna()) & (df['CoorX'] != 0) & (df['CoorY'].notna()) & (df['CoorY'] != 0)

# Filtrar las filas donde 'Huso' está en NaN o es igual a 0
filtro_huso = df['Huso'].isna() | (df['Huso'] == 0)

# Aplicar ambos filtros
parcelas_faltantes_huso = df[filtro_coordenadas & filtro_huso]

# Mostrar el número de parcelas que cumplen las condiciones
print(f"Se han encontrado {parcelas_faltantes_huso.shape[0]} parcelas con CoorX y CoorY no nulas y distintos de 0, y Huso en NaN o 0.")

# Si deseas ver las primeras filas
print(parcelas_faltantes_huso.head())



Se han encontrado 24942 parcelas con CoorX y CoorY no nulas y distintos de 0, y Huso en NaN o 0.
       Provincia IDParcela     CoorX      CoorY  Huso  Presencia  Ano_ifn4  \
56360         25     25_01  311000.0  4745000.0   NaN          3       NaN   
56361         25     25_02  312000.0  4745000.0   NaN          3       NaN   
56362         25     25_03  311000.0  4744000.0   NaN          3       NaN   
56363         25     25_04  312000.0  4744000.0   NaN          3       NaN   
56364         25     25_05  313000.0  4744000.0   NaN          3       NaN   

       Ano_ifn3  Ano_ifn2  
56360    2001.0       NaN  
56361    2001.0       NaN  
56362    2001.0       NaN  
56363    2001.0       NaN  
56364    2001.0       NaN  


identifico rangos de longitud

In [53]:
try:
    import geopandas as gpd
    import folium
    from shapely.geometry import LineString
    
    # Cargar el shapefile de límites provinciales
    provincias_gdf = gpd.read_file('IFN_España/shape/SHP_ETRS89/ll_provinciales_inspire_peninbal_etrs89/ll_provinciales_inspire_peninbal_etrs89.shp')
    
    # Lista de nombres de provincias
    provincias = [
        'Albacete', 'Alicante', 'Almería', 'Álava', 'Asturias', 'Ávila', 'Badajoz', 'Baleares',
        'Barcelona', 'Bizkaia', 'Burgos', 'Cáceres', 'Cádiz', 'Cantabria', 'Castellón', 'Ciudad Real',
        'Córdoba', 'A Coruña', 'Cuenca', 'Gipuzkoa', 'Girona', 'Granada', 'Guadalajara', 'Huelva',
        'Huesca', 'Jaén', 'León', 'Lleida', 'Lugo', 'Madrid', 'Málaga', 'Murcia', 'Navarra', 'Ourense',
        'Palencia', 'Las Palmas', 'Pontevedra', 'La Rioja', 'Salamanca', 'Santa Cruz de Tenerife', 
        'Segovia', 'Sevilla', 'Soria', 'Tarragona', 'Teruel', 'Toledo', 'Valencia', 'Valladolid', 
        'Zamora', 'Zaragoza'
    ]
    
    # Crear un mapa centrado en España
    m = folium.Map(location=[40.4637, -3.7492], zoom_start=6)
    
    # Diccionario para almacenar las longitudes (min, max) de cada provincia
    longitudes = {}
    
    # Iterar sobre cada provincia
    for provincia_nombre in provincias:
        # Filtrar las fronteras correspondientes a la provincia de interés
        provincias_fronteras = provincias_gdf[provincias_gdf['NAME_BOUND'].str.contains(provincia_nombre, case=False)]
        
        # Calcular las longitudes mínima y máxima para la provincia
        longitudes_provincia = []
        
        for _, row in provincias_fronteras.iterrows():
            coords = list(row['geometry'].coords)
            for point in coords:
                longitudes_provincia.append(point[0])  # Coordenadas X (longitud)
        
        if longitudes_provincia:  # Solo calcular min/max si hay coordenadas
            longitud_min = min(longitudes_provincia)
            longitud_max = max(longitudes_provincia)
            
            # Almacenar en el diccionario
            longitudes[provincia_nombre] = [provincia_nombre, longitud_min, longitud_max]
            
            # Dibujar las fronteras de la provincia en el mapa
            for _, provincia in provincias_fronteras.iterrows():
                coords = list(provincia['geometry'].coords)
                folium.PolyLine(locations=[(point[1], point[0]) for point in coords], color='blue', weight=2).add_to(m)
            
            # Dibujar la línea horizontal que marca el rango de longitud de la provincia
            folium.PolyLine(
                locations=[(40.4637, longitud_min), (40.4637, longitud_max)],  # Latitud fija (aproximadamente el centro de España)
                color='red',
                weight=4,
                opacity=1
            ).add_to(m)
        else:
            print(f"No se encontraron coordenadas válidas para la provincia {provincia_nombre}.")
    
    # Guardar el mapa como HTML
    mapa_path = 'IFN_España/mapa_fronteras_y_rango_longitud_todas_provincias.html'
    m.save(mapa_path)
    
    print(f"Mapa guardado en: {mapa_path}")
    print("\nDiccionario de longitudes (min, max) para cada provincia:")
    print(longitudes)
    
except Exception as e:
    print(f"⚠️  Celda 94 saltada (shapefile no disponible): {e}")


⚠️  Celda 94 saltada (shapefile no disponible): IFN_España/shape/SHP_ETRS89/ll_provinciales_inspire_peninbal_etrs89/ll_provinciales_inspire_peninbal_etrs89.shp: No such file or directory


Identifico los rangos de latitud

In [54]:
try:
    import geopandas as gpd
    import folium
    from shapely.geometry import LineString
    
    # Cargar el shapefile de límites provinciales
    provincias_gdf = gpd.read_file('IFN_España/shape/SHP_ETRS89/ll_provinciales_inspire_peninbal_etrs89/ll_provinciales_inspire_peninbal_etrs89.shp')
    
    # Lista de nombres de provincias
    provincias = [
        'Albacete', 'Alicante', 'Almería', 'Álava', 'Asturias', 'Ávila', 'Badajoz', 'Baleares',
        'Barcelona', 'Bizkaia', 'Burgos', 'Cáceres', 'Cádiz', 'Cantabria', 'Castellón', 'Ciudad Real',
        'Córdoba', 'A Coruña', 'Cuenca', 'Gipuzkoa', 'Girona', 'Granada', 'Guadalajara', 'Huelva',
        'Huesca', 'Jaén', 'León', 'Lleida', 'Lugo', 'Madrid', 'Málaga', 'Murcia', 'Navarra', 'Ourense',
        'Palencia', 'Las Palmas', 'Pontevedra', 'La Rioja', 'Salamanca', 'Santa Cruz de Tenerife', 
        'Segovia', 'Sevilla', 'Soria', 'Tarragona', 'Teruel', 'Toledo', 'Valencia', 'Valladolid', 
        'Zamora', 'Zaragoza'
    ]
    
    # Crear un mapa centrado en España
    m = folium.Map(location=[40.4637, -3.7492], zoom_start=6)
    
    # Diccionario para almacenar las latitudes (min, max) de cada provincia
    latitudes = {}
    
    # Iterar sobre cada provincia
    for provincia_nombre in provincias:
        # Filtrar las fronteras correspondientes a la provincia de interés
        provincias_fronteras = provincias_gdf[provincias_gdf['NAME_BOUND'].str.contains(provincia_nombre, case=False)]
        
        # Calcular las latitudes mínima y máxima para la provincia
        latitudes_provincia = []
        
        for _, row in provincias_fronteras.iterrows():
            coords = list(row['geometry'].coords)
            for point in coords:
                latitudes_provincia.append(point[1])  # Coordenadas Y (latitud)
        
        if latitudes_provincia:  # Solo calcular min/max si hay coordenadas
            latitud_min = min(latitudes_provincia)
            latitud_max = max(latitudes_provincia)
            
            # Almacenar en el diccionario
            latitudes[provincia_nombre] = [provincia_nombre, latitud_min, latitud_max]
            
            # Dibujar las fronteras de la provincia en el mapa
            for _, provincia in provincias_fronteras.iterrows():
                coords = list(provincia['geometry'].coords)
                folium.PolyLine(locations=[(point[1], point[0]) for point in coords], color='blue', weight=2).add_to(m)
            
            # Dibujar la línea horizontal que marca el rango de latitud de la provincia
            folium.PolyLine(
                locations=[(latitud_min, -3.7492), (latitud_max, -3.7492)],  # Longitud fija (aproximadamente el centro de España)
                color='red',
                weight=4,
                opacity=1
            ).add_to(m)
        else:
            print(f"No se encontraron coordenadas válidas para la provincia {provincia_nombre}.")
    
    
    print("\nDiccionario de latitudes (min, max) para cada provincia:")
    print(latitudes)
    
except Exception as e:
    print(f"⚠️  Celda 96 saltada (shapefile no disponible): {e}")


⚠️  Celda 96 saltada (shapefile no disponible): IFN_España/shape/SHP_ETRS89/ll_provinciales_inspire_peninbal_etrs89/ll_provinciales_inspire_peninbal_etrs89.shp: No such file or directory


longitudes = {
    '02': ['Albacete', -2.882888421, -0.915793253],
    '03': ['Alicante', -1.094161755, 0.234067363363889],
    '04': ['Almería', -3.14018737, -1.6301237430000002],
    '01': ['Álava', -3.2867669470000003, -2.232687147],
    '33': ['Asturias', -7.182488914, -4.5104534990000005],
    '05': ['Ávila', -5.737561939000001, -4.160234294],
    '06': ['Badajoz', -7.335668028000001, -4.6475773910000004],
    '08': ['Barcelona', 1.3603969310000001, 2.777584332],
    '48': ['Bizkaia', -3.450329097, -2.4126037030000003],
    '09': ['Burgos', -4.335327148, -2.5173969940000003],
    '10': ['Cáceres', -7.541331761, -4.952950512],
    '11': ['Cádiz', -6.472066351770252, -5.086769376],
    '39': ['Cantabria', -4.851778269, -3.1496347360000003],
    '12': ['Castellón', -0.846296899, 0.690313419798671],
    '13': ['Ciudad Real', -5.0469484200000005, -2.6378903030000003],
    '14': ['Córdoba', -5.585923993000001, -4.000821256],
    '15': ['A Coruña', -9.301515672, -7.661721236000001],
    '16': ['Cuenca', -3.170234546, -1.14254654],
    '20': ['Gipuzkoa', -2.602719147, -1.729343469],
    '17': ['Girona', 1.724297688, 3.33255381296874],
    '18': ['Granada', -4.328132946, -2.207789069],
    '19': ['Guadalajara', -3.541098185, -1.535637879],
    '21': ['Huelva', -7.523047057, -6.121558762],
    '22': ['Huesca', -0.9341684400000001, 0.771306673],
    '23': ['Jaén', -4.287640435, -2.4349517940000003],
    '24': ['León', -7.077068851000001, -4.7338814540000005],
    '25': ['Lleida', 0.32003648, 1.8550576950000002],
    '27': ['Lugo', -7.999732198, -6.814053008],
    '28': ['Madrid', -4.579076175, -3.052983314],
    '29': ['Málaga', -5.611776730000001, -3.7659671070000003],
    '30': ['Murcia', -2.344184266, -0.6479829725699346],
    '31': ['Navarra', -2.5000827, -0.724069693],
    '32': ['Ourense', -8.36634439, -6.733953272000001],
    '34': ['Palencia', -5.031894516, -3.889622382],
    '36': ['Pontevedra', -8.94945260299778, -7.861428011],
    '26': ['La Rioja', -3.1342713900000003, -1.678701495],
    '37': ['Salamanca', -6.931705367, -5.08984113],
    '40': ['Segovia', -4.7246528450000005, -3.206929036],
    '41': ['Sevilla', -6.538514692000001, -4.653350316],
    '42': ['Soria', -3.550438343, -1.775371604],
    '43': ['Tarragona', 0.15907026600000002, 1.653151317],
    '44': ['Teruel', -1.8057277390000002, 0.293500763],
    '45': ['Toledo', -5.4061836880000005, -2.908233289],
    '46': ['Valencia', -1.528811131, -0.02479343],
    '47': ['Valladolid', -5.520890757, -3.980442109],
    '49': ['Zamora', -7.034043109000001, -5.228873192],
    '50': ['Zaragoza', -2.173671268, 0.385694968]
}


latitudes=
{'02': ['Albacete', 38.021837689, 39.423076668], 
'03': ['Alicante', 37.843778910000005, 38.887391429000004], 
'04': ['Almería', 35.93764597790876, 37.916247788], 
'01': ['Álava', 42.472360844, 43.217187327000005], 
'33': ['Asturias', 42.8825428, 43.666532231987446], 
'05': ['Ávila', 40.081925931, 41.164249717000004], 
'06': ['Badajoz', 37.941026032, 39.451788828000005], 
'08': ['Barcelona', 41.192745294000005, 42.322960728000005], 
'48': ['Bizkaia', 42.981652872000005, 43.45724613149264], 
'09': ['Burgos', 41.450784169, 43.198747316], 
'10': ['Cáceres', 39.031485559000004, 40.486651389], 
'11': ['Cádiz', 36.000275007, 37.052445275000004], 
'39': ['Cantabria', 42.758049853, 43.513692999], 
'12': ['Castellón', 39.714697789000006, 40.788631201], 
'13': ['Ciudad Real', 38.342452539, 39.576761494], 
'14': ['Córdoba', 37.184070531, 38.729087381], 
'15': ['A Coruña', 42.45903353682553, 43.79237956787006], 
'16': ['Cuenca', 39.226480682, 40.658692371], 
'20': ['Gipuzkoa', 42.895245254, 43.395910838000006], 
'17': ['Girona', 41.650768515, 42.495574682000004], 
'18': ['Granada', 36.693519778, 38.084093161000006], 
'19': ['Guadalajara', 40.152455694000004, 41.327632215], 
'21': ['Huelva', 36.789718347000004, 38.208209793], 
'22': ['Huesca', 41.34780562, 42.924386322000004], 
'23': ['Jaén', 37.378278089000005, 38.532534265], 
'24': ['León', 42.029226138000006, 43.23820416], 
'25': ['Lleida', 41.274196758, 42.861449467], 
'27': ['Lugo', 42.325700246000004, 43.761644173136006], 
'28': ['Madrid', 39.884719334, 41.165844755], 
'29': ['Málaga', 36.310280594000005, 37.282199489], 
'30': ['Murcia', 37.373752541, 38.755084348000004], 
'31': ['Navarra', 41.90989372, 43.314843090000004], 
'32': ['Ourense', 41.807233058, 42.578599936], 
'34': ['Palencia', 41.758166346, 43.060988881], 
'36': ['Pontevedra', 41.86884894957103, 42.860157060000006], 
'26': ['La Rioja', 41.919033954, 42.644264701000004], 
'37': ['Salamanca', 40.238923473, 41.294215916000006], 
'40': ['Segovia', 40.632984784, 41.586082420000004], 
'41': ['Sevilla', 36.842191546, 38.197417261000005], 
'42': ['Soria', 41.056507809, 42.146706107], 
'43': ['Tarragona', 40.523073055000005, 41.582606314], 
'44': ['Teruel', 39.846778156, 41.354389427], 
'45': ['Toledo', 39.258416781, 40.318341548], 
'46': ['Valencia', 38.68655364, 40.211683158], 
'47': ['Valladolid', 41.094015309, 42.31184203], 
'49': ['Zamora', 41.1171729, 42.254192535], 
'50': ['Zaragoza', 40.936215684000004, 42.744052031]}

Identifico las parcelas fuera de lugar 

In [55]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium

# =======================
# 1. Cargar y preparar datos
# =======================
df = pd.read_csv('IFN_España/ubicaciones.csv')
df = df.dropna(subset=['CoorX', 'CoorY', 'Huso'])

# Asegurarse que Huso es entero y válido
df['Huso'] = df['Huso'].astype(int)
df = df[df['Huso'].isin([28, 29, 30, 31])]

# Asegurar tipos numéricos
df['CoorX'] = pd.to_numeric(df['CoorX'], errors='coerce')
df['CoorY'] = pd.to_numeric(df['CoorY'], errors='coerce')
df = df.dropna(subset=['CoorX', 'CoorY'])

# =======================
# 2. Crear geometrías por huso
# =======================
df['geometry'] = df.apply(lambda row: Point(row['CoorX'], row['CoorY']), axis=1)

gdf_28 = gpd.GeoDataFrame(df[df['Huso'] == 28], geometry='geometry', crs='EPSG:25828')
gdf_29 = gpd.GeoDataFrame(df[df['Huso'] == 29], geometry='geometry', crs='EPSG:25829')
gdf_30 = gpd.GeoDataFrame(df[df['Huso'] == 30], geometry='geometry', crs='EPSG:25830')
gdf_31 = gpd.GeoDataFrame(df[df['Huso'] == 31], geometry='geometry', crs='EPSG:25831')

# Reproyectar a WGS84
gdf = pd.concat([
    gdf_28.to_crs(epsg=4326),
    gdf_29.to_crs(epsg=4326),
    gdf_30.to_crs(epsg=4326),
    gdf_31.to_crs(epsg=4326)
])

# Añadir columnas de lat/lon y arreglar provincia
gdf['latitud'] = gdf.geometry.y
gdf['longitud'] = gdf.geometry.x
gdf['Provincia'] = gdf['Provincia'].astype(str).str.zfill(2)

# =======================
# 3. Diccionario de rangos de longitud
# =======================
longitudes = {
    '02': ['Albacete', -2.882888421, -0.915793253],
    '03': ['Alicante', -1.094161755, 0.234067363363889],
    '04': ['Almería', -3.14018737, -1.6301237430000002],
    '01': ['Álava', -3.2867669470000003, -2.232687147],
    '33': ['Asturias', -7.182488914, -4.5104534990000005],
    '05': ['Ávila', -5.737561939000001, -4.160234294],
    '06': ['Badajoz', -7.335668028000001, -4.6475773910000004],
    '08': ['Barcelona', 1.3603969310000001, 2.777584332],
    '48': ['Bizkaia', -3.450329097, -2.4126037030000003],
    '09': ['Burgos', -4.335327148, -2.5173969940000003],
    '10': ['Cáceres', -7.541331761, -4.952950512],
    '11': ['Cádiz', -6.472066351770252, -5.086769376],
    '39': ['Cantabria', -4.851778269, -3.1496347360000003],
    '12': ['Castellón', -0.846296899, 0.690313419798671],
    '13': ['Ciudad Real', -5.0469484200000005, -2.6378903030000003],
    '14': ['Córdoba', -5.585923993000001, -4.000821256],
    '15': ['A Coruña', -9.301515672, -7.661721236000001],
    '16': ['Cuenca', -3.170234546, -1.14254654],
    '20': ['Gipuzkoa', -2.602719147, -1.729343469],
    '17': ['Girona', 1.724297688, 3.33255381296874],
    '18': ['Granada', -4.328132946, -2.207789069],
    '19': ['Guadalajara', -3.541098185, -1.535637879],
    '21': ['Huelva', -7.523047057, -6.121558762],
    '22': ['Huesca', -0.9341684400000001, 0.771306673],
    '23': ['Jaén', -4.287640435, -2.4349517940000003],
    '24': ['León', -7.077068851000001, -4.7338814540000005],
    '25': ['Lleida', 0.32003648, 1.8550576950000002],
    '27': ['Lugo', -7.999732198, -6.814053008],
    '28': ['Madrid', -4.579076175, -3.052983314],
    '29': ['Málaga', -5.611776730000001, -3.7659671070000003],
    '30': ['Murcia', -2.344184266, -0.6479829725699346],
    '31': ['Navarra', -2.5000827, -0.724069693],
    '32': ['Ourense', -8.36634439, -6.733953272000001],
    '34': ['Palencia', -5.031894516, -3.889622382],
    '36': ['Pontevedra', -8.94945260299778, -7.861428011],
    '26': ['La Rioja', -3.1342713900000003, -1.678701495],
    '37': ['Salamanca', -6.931705367, -5.08984113],
    '40': ['Segovia', -4.7246528450000005, -3.206929036],
    '41': ['Sevilla', -6.538514692000001, -4.653350316],
    '42': ['Soria', -3.550438343, -1.775371604],
    '43': ['Tarragona', 0.15907026600000002, 1.653151317],
    '44': ['Teruel', -1.8057277390000002, 0.293500763],
    '45': ['Toledo', -5.4061836880000005, -2.908233289],
    '46': ['Valencia', -1.528811131, -0.02479343],
    '47': ['Valladolid', -5.520890757, -3.980442109],
    '49': ['Zamora', -7.034043109000001, -5.228873192],
    '50': ['Zaragoza', -2.173671268, 0.385694968]
}

# =======================
# 4. Diagnóstico y visualización
# =======================
m = folium.Map(location=[gdf['latitud'].mean(), gdf['longitud'].mean()], zoom_start=6)
parcelas_mal_proyectadas = []

for _, row in gdf.iterrows():
    cod_prov = row['Provincia']
   
    if cod_prov in longitudes:
        lon_min, lon_max = longitudes[cod_prov][1], longitudes[cod_prov][2]
        if not (lon_min <= row['longitud'] <= lon_max):
            parcelas_mal_proyectadas.append(row)
            folium.CircleMarker(
                location=[row['latitud'], row['longitud']],
                radius=5,
                color='red',
                fill=True,
                fill_color='red',
                fill_opacity=0.6,
                popup=f"Provincia: {row['Provincia']} | Huso: {row['Huso']}"
            ).add_to(m)

# =======================
# 5. Salidas
# =======================
print(f"\n🔎 Total de parcelas: {len(gdf)}")
print(f"❌ Parcelas mal proyectadas: {len(parcelas_mal_proyectadas)}\n")

# Guardar el archivo CSV de las parcelas mal proyectadas
parcelas_mal_proyectadas_df = pd.DataFrame(parcelas_mal_proyectadas)
parcelas_mal_proyectadas_df.to_csv('IFN_España/mal_proyectadas.csv', index=False)

# Guardar el mapa
m.save('IFN_España/mapa_parcelas_mal_proyectadas.html')

print("Archivo CSV y mapa generados con las parcelas mal proyectadas.")


🔎 Total de parcelas: 76691
❌ Parcelas mal proyectadas: 4046



Archivo CSV y mapa generados con las parcelas mal proyectadas.


¿que provincias son las que estan mal?

In [56]:
import pandas as pd

# Cargar el archivo CSV mal_proyectadas
df_mal_proyectadas = pd.read_csv('IFN_España/mal_proyectadas.csv')

# Obtener los valores únicos de la columna 'Provincia'
valores_unicos_provincia = df_mal_proyectadas['Provincia'].unique()

# Imprimir los valores únicos
print(valores_unicos_provincia)


[ 6 37 49 21  1 41 50 33 11 24 34 47  8]


Las parcelas mal proyectadas pertenecen a las fronteras entre Husos, corregimos el Huso 
[49:30, 10:30, 37:30,  3:31  6:29, 11:29, 12:31 24:29, 33:29, 41:29, 44 50:31, 18:29, 22:30]

In [57]:
import pandas as pd

# Cargar el archivo mal_proyectadas.csv
df_mal_proyectadas = pd.read_csv('IFN_España/mal_proyectadas.csv')

# Diccionario con las asignaciones de Huso por Provincia
huso_correcciones = {
    49: 30,  # Zamora: Huso 30
    10: 30,  # Cáceres: Huso 30
    37: 30,  # Salamanca: Huso 30
    3: 31,   # Alicante: Huso 31
    6: 29,   # Badajoz: Huso 29
    11: 29,  # Cádiz: Huso 29
    12: 31,  # Castellón: Huso 31
    24: 29,  # León: Huso 29
    33: 29,  # Asturias: Huso 29
    41: 29,  # Sevilla: Huso 29
    44: 31,  # Teruel: Huso 31
    50: 31,  # Zaragoza: Huso 31
    18: 29,  # Granada: Huso 29
    22: 30   # Huesca: Huso 30
}

# Añadir la columna 'HusoCorregido' y aplicar las correcciones según la provincia
df_mal_proyectadas['HusoCorregido'] = df_mal_proyectadas['Provincia'].map(huso_correcciones)

# Guardar el archivo actualizado
df_mal_proyectadas.to_csv('IFN_España/mal_proyectadas.csv', index=False)

# Imprimir el DataFrame con la nueva columna HusoCorregido
print(df_mal_proyectadas[['Provincia', 'Huso', 'HusoCorregido']].head())


   Provincia  Huso  HusoCorregido
0          6    29           29.0
1          6    29           29.0
2          6    29           29.0
3          6    29           29.0
4          6    29           29.0


Vemos si la correcion es efectiva

In [58]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium

# Cargar el archivo CSV con las coordenadas y HusoCorregido
df = pd.read_csv('IFN_España/mal_proyectadas.csv')

# Eliminar filas con valores faltantes en las coordenadas y HusoCorregido
df = df.dropna(subset=['CoorX', 'CoorY', 'HusoCorregido'])

# Filtrar por HusoCorregido 29 y 30
df = df[df['HusoCorregido'].isin([28,29, 30,31])]

# Asegurar tipos numéricos para las coordenadas
df['CoorX'] = pd.to_numeric(df['CoorX'], errors='coerce')
df['CoorY'] = pd.to_numeric(df['CoorY'], errors='coerce')

# Eliminar filas con valores faltantes en las coordenadas
df = df.dropna(subset=['CoorX', 'CoorY'])

# Crear geometría de coordenadas
df['geometry'] = df.apply(lambda row: Point(row['CoorX'], row['CoorY']), axis=1)

# Crear GeoDataFrames por HusoCorregido
gdf_28 = gpd.GeoDataFrame(df[df['HusoCorregido'] == 28], geometry='geometry', crs='EPSG:25828')  # HusoCorregido 28
gdf_29 = gpd.GeoDataFrame(df[df['HusoCorregido'] == 29], geometry='geometry', crs='EPSG:25829')  # HusoCorregido 29
gdf_30 = gpd.GeoDataFrame(df[df['HusoCorregido'] == 30], geometry='geometry', crs='EPSG:25830')  # HusoCorregido 30
gdf_31 = gpd.GeoDataFrame(df[df['HusoCorregido'] == 31], geometry='geometry', crs='EPSG:25831')  # HusoCorregido 31

# Reproyectar a WGS84 (EPSG:4326)
gdf_28 = gdf_28.to_crs(epsg=4326)
gdf_29 = gdf_29.to_crs(epsg=4326)
gdf_30 = gdf_30.to_crs(epsg=4326)
gdf_31 = gdf_31.to_crs(epsg=4326)

# Unir las parcelas reproyectadas
gdf = pd.concat([gdf_28,gdf_29, gdf_30,gdf_31])

# Extraer las coordenadas de latitud y longitud
df['latitud'] = gdf.geometry.y
df['longitud'] = gdf.geometry.x

# Crear un mapa de folium centrado en las coordenadas de las parcelas
m = folium.Map(location=[df['latitud'].mean(), df['longitud'].mean()], zoom_start=12)

# Crear un diccionario con colores para cada provincia 
colores_provincia = {
    1: 'blue',
    2: 'green',
    3: 'red',
    4: 'purple',
    5: 'orange',
    6: 'pink',
    7: 'brown',
    8: 'gray',
    9: 'yellow',
    10: 'cyan',
    11: 'magenta',
    12: 'lime',
    13: 'indigo',
    14: 'violet',
    15: 'teal',
    16: 'olive',
    17: 'maroon',
    18: 'navy',
    19: 'turquoise',
    20: 'coral',
    21: 'chocolate',
    22: 'gold',
    23: 'plum',
    24: 'slateblue',
    25: 'mediumseagreen',
    26: 'crimson',
    27: 'mediumslateblue',
    28: 'darkslategray',
    29: 'darkorange',
    30: 'forestgreen',
    31: 'darkviolet',
    32: 'khaki',
    33: 'silver',
    34: 'lightcoral',
    35: 'deepskyblue',
    36: 'mediumvioletred',
    37: 'chartreuse',
    38: 'palevioletred',
    39: 'aqua',
    40: 'dodgerblue',
    41: 'mediumspringgreen',
    42: 'lightgreen',
    43: 'seashell',
    44: 'lightblue',
    45: 'darkgreen',
    46: 'mediumorchid',
    47: 'gainsboro',
    48: 'snow',
    49: 'lavender',
    50: 'mistyrose'
}


# Añadir los puntos de cada parcela con un círculo coloreado según la provincia
for _, row in df.iterrows():
    color = colores_provincia.get(row['Provincia'], 'gray')  # Si la provincia no está en el diccionario, usa color gris
    folium.CircleMarker(
        location=[row['latitud'], row['longitud']],
        radius=5,  # Tamaño del punto
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.6,
        popup=f"Provincia: {row['Provincia']}, HusoCorregido: {row['HusoCorregido']}"
    ).add_to(m)

# Guardar el mapa como HTML
m.save('IFN_España/mapa_parcelas_corregidas.html')

print("Mapa guardado en: data/raw/IFN_España/mapa_parcelas_colores.html")


Mapa guardado en: data/raw/IFN_España/mapa_parcelas_colores.html


Ahora edito el archivo ubicaciones.csv 

In [59]:
import pandas as pd

# Cargar el archivo ubicaciones.csv
ubicaciones_path = 'IFN_España/ubicaciones.csv'
df_ubicaciones = pd.read_csv(ubicaciones_path)

# Cargar el archivo mal_proyectadas_corregidas.csv (puede no existir)
mal_proyectadas_path = 'IFN_España/mal_proyectadas_corregidas.csv'
import os
if not os.path.exists(mal_proyectadas_path):
    print("⚠️  mal_proyectadas_corregidas.csv no encontrado, saltando corrección de huso.")
    df_mal_proyectadas = pd.DataFrame(columns=['IDParcela', 'HusoCorregido'])
else:
    df_mal_proyectadas = pd.read_csv(mal_proyectadas_path)

# Asegurarse de que los IDs de Parcela están en formato cadena (sin necesidad de conversión a entero)
df_ubicaciones['IDParcela'] = df_ubicaciones['IDParcela'].astype(str)
df_mal_proyectadas['IDParcela'] = df_mal_proyectadas['IDParcela'].astype(str)

# Realizar la unión entre los dos DataFrames usando 'IDParcela'
df_ubicaciones = df_ubicaciones.merge(df_mal_proyectadas[['IDParcela', 'HusoCorregido']], 
                                       on='IDParcela', 
                                       how='left')

# Modificar la columna 'Huso' con los valores de 'HusoCorregido' donde estén presentes
df_ubicaciones['Huso'] = df_ubicaciones['HusoCorregido'].fillna(df_ubicaciones['Huso'])

# Eliminar la columna 'HusoCorregido' ya que ya no es necesaria
df_ubicaciones = df_ubicaciones.drop(columns=['HusoCorregido'])

# Guardar el DataFrame actualizado de nuevo en el archivo CSV
df_ubicaciones.to_csv('IFN_España/ubicaciones.csv', index=False)

print("Archivo 'ubicaciones.csv' actualizado con los valores de HusoCorregido.")


⚠️  mal_proyectadas_corregidas.csv no encontrado, saltando corrección de huso.


Archivo 'ubicaciones.csv' actualizado con los valores de HusoCorregido.


elimino las parcelas que estan fuera del rango de latitud (africanas) y de longitud 

In [60]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium

# =======================
# 1. Cargar y preparar datos
# =======================
df = pd.read_csv('IFN_España/ubicaciones.csv')
df = df.dropna(subset=['CoorX', 'CoorY', 'Huso'])

# Asegurarse que Huso es entero y válido
df['Huso'] = df['Huso'].astype(int)
df = df[df['Huso'].isin([28, 29, 30, 31])]

# Asegurar tipos numéricos
df['CoorX'] = pd.to_numeric(df['CoorX'], errors='coerce')
df['CoorY'] = pd.to_numeric(df['CoorY'], errors='coerce')
df = df.dropna(subset=['CoorX', 'CoorY'])

# =======================
# 2. Crear geometrías por huso
# =======================
df['geometry'] = df.apply(lambda row: Point(row['CoorX'], row['CoorY']), axis=1)

gdf_28 = gpd.GeoDataFrame(df[df['Huso'] == 28], geometry='geometry', crs='EPSG:25828')
gdf_29 = gpd.GeoDataFrame(df[df['Huso'] == 29], geometry='geometry', crs='EPSG:25829')
gdf_30 = gpd.GeoDataFrame(df[df['Huso'] == 30], geometry='geometry', crs='EPSG:25830')
gdf_31 = gpd.GeoDataFrame(df[df['Huso'] == 31], geometry='geometry', crs='EPSG:25831')

# Reproyectar a WGS84
gdf = pd.concat([
    gdf_28.to_crs(epsg=4326),
    gdf_29.to_crs(epsg=4326),
    gdf_30.to_crs(epsg=4326),
    gdf_31.to_crs(epsg=4326)
])

# Añadir columnas de lat/lon y arreglar provincia
gdf['latitud'] = gdf.geometry.y
gdf['longitud'] = gdf.geometry.x
gdf['Provincia'] = gdf['Provincia'].astype(str).str.zfill(2)

# =======================
# 3. Diccionario de rangos de latitud
# =======================
latitudes = {'02': ['Albacete', 38.021837689, 39.423076668], 
'03': ['Alicante', 37.843778910000005, 38.887391429000004], 
'04': ['Almería', 35.93764597790876, 37.916247788], 
'01': ['Álava', 42.472360844, 43.217187327000005], 
'33': ['Asturias', 42.8825428, 43.666532231987446], 
'05': ['Ávila', 40.081925931, 41.164249717000004], 
'06': ['Badajoz', 37.941026032, 39.451788828000005], 
'08': ['Barcelona', 41.192745294000005, 42.322960728000005], 
'48': ['Bizkaia', 42.981652872000005, 43.45724613149264], 
'09': ['Burgos', 41.450784169, 43.198747316], 
'10': ['Cáceres', 39.031485559000004, 40.486651389], 
'11': ['Cádiz', 36.000275007, 37.052445275000004], 
'39': ['Cantabria', 42.758049853, 43.513692999], 
'12': ['Castellón', 39.714697789000006, 40.788631201], 
'13': ['Ciudad Real', 38.342452539, 39.576761494], 
'14': ['Córdoba', 37.184070531, 38.729087381], 
'15': ['A Coruña', 42.45903353682553, 43.79237956787006], 
'16': ['Cuenca', 39.226480682, 40.658692371], 
'20': ['Gipuzkoa', 42.895245254, 43.395910838000006], 
'17': ['Girona', 41.650768515, 42.495574682000004], 
'18': ['Granada', 36.693519778, 38.084093161000006], 
'19': ['Guadalajara', 40.152455694000004, 41.327632215], 
'21': ['Huelva', 36.789718347000004, 38.208209793], 
'22': ['Huesca', 41.34780562, 42.924386322000004], 
'23': ['Jaén', 37.378278089000005, 38.532534265], 
'24': ['León', 42.029226138000006, 43.23820416], 
'25': ['Lleida', 41.274196758, 42.861449467], 
'27': ['Lugo', 42.325700246000004, 43.761644173136006], 
'28': ['Madrid', 39.884719334, 41.165844755], 
'29': ['Málaga', 36.310280594000005, 37.282199489], 
'30': ['Murcia', 37.373752541, 38.755084348000004], 
'31': ['Navarra', 41.90989372, 43.314843090000004], 
'32': ['Ourense', 41.807233058, 42.578599936], 
'34': ['Palencia', 41.758166346, 43.060988881], 
'36': ['Pontevedra', 41.86884894957103, 42.860157060000006], 
'26': ['La Rioja', 41.919033954, 42.644264701000004], 
'37': ['Salamanca', 40.238923473, 41.294215916000006], 
'40': ['Segovia', 40.632984784, 41.586082420000004], 
'41': ['Sevilla', 36.842191546, 38.197417261000005], 
'42': ['Soria', 41.056507809, 42.146706107], 
'43': ['Tarragona', 40.523073055000005, 41.582606314], 
'44': ['Teruel', 39.846778156, 41.354389427], 
'45': ['Toledo', 39.258416781, 40.318341548], 
'46': ['Valencia', 38.68655364, 40.211683158], 
'47': ['Valladolid', 41.094015309, 42.31184203], 
'49': ['Zamora', 41.1171729, 42.254192535], 
'50': ['Zaragoza', 40.936215684000004, 42.744052031]}

# =======================
# 4. Diagnóstico y visualización
# =======================
m = folium.Map(location=[gdf['latitud'].mean(), gdf['longitud'].mean()], zoom_start=6)
parcelas_mal_proyectadas = []

for _, row in gdf.iterrows():
    cod_prov = row['Provincia']
   
    if cod_prov in latitudes:
        lat_min, lat_max = latitudes[cod_prov][1], latitudes[cod_prov][2]
        if not (lat_min <= row['latitud'] <= lat_max):  # Comprobar si la latitud está fuera del rango
            parcelas_mal_proyectadas.append(row)
            folium.CircleMarker(
                location=[row['latitud'], row['longitud']],
                radius=5,
                color='red',
                fill=True,
                fill_color='red',
                fill_opacity=0.6,
                popup=f"Provincia: {row['Provincia']} | Huso: {row['Huso']}"
            ).add_to(m)

# =======================
# 5. Salidas
# =======================
print(f"\n🔎 Total de parcelas: {len(gdf)}")
print(f"❌ Parcelas mal proyectadas: {len(parcelas_mal_proyectadas)}\n")

# Guardar el archivo CSV de las parcelas mal proyectadas
parcelas_mal_proyectadas_df = pd.DataFrame(parcelas_mal_proyectadas)
parcelas_mal_proyectadas_df.to_csv('IFN_España/mal_proyectadas_latitud.csv', index=False)


🔎 Total de parcelas: 76691
❌ Parcelas mal proyectadas: 3111



In [61]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium

# =======================
# 1. Cargar y preparar datos
# =======================
df = pd.read_csv('IFN_España/ubicaciones.csv')
df = df.dropna(subset=['CoorX', 'CoorY', 'Huso'])

# Asegurarse que Huso es entero y válido
df['Huso'] = df['Huso'].astype(int)
df = df[df['Huso'].isin([28, 29, 30, 31])]

# Asegurar tipos numéricos
df['CoorX'] = pd.to_numeric(df['CoorX'], errors='coerce')
df['CoorY'] = pd.to_numeric(df['CoorY'], errors='coerce')
df = df.dropna(subset=['CoorX', 'CoorY'])

# =======================
# 2. Crear geometrías por huso
# =======================
df['geometry'] = df.apply(lambda row: Point(row['CoorX'], row['CoorY']), axis=1)

gdf_28 = gpd.GeoDataFrame(df[df['Huso'] == 28], geometry='geometry', crs='EPSG:25828')
gdf_29 = gpd.GeoDataFrame(df[df['Huso'] == 29], geometry='geometry', crs='EPSG:25829')
gdf_30 = gpd.GeoDataFrame(df[df['Huso'] == 30], geometry='geometry', crs='EPSG:25830')
gdf_31 = gpd.GeoDataFrame(df[df['Huso'] == 31], geometry='geometry', crs='EPSG:25831')

# Reproyectar a WGS84
gdf = pd.concat([
    gdf_28.to_crs(epsg=4326),
    gdf_29.to_crs(epsg=4326),
    gdf_30.to_crs(epsg=4326),
    gdf_31.to_crs(epsg=4326)
])

# Añadir columnas de lat/lon y arreglar provincia
gdf['latitud'] = gdf.geometry.y
gdf['longitud'] = gdf.geometry.x
gdf['Provincia'] = gdf['Provincia'].astype(str).str.zfill(2)

# =======================
# 3. Diccionario de rangos de longitud
# =======================
longitudes = {
    '02': ['Albacete', -2.882888421, -0.915793253],
    '03': ['Alicante', -1.094161755, 0.234067363363889],
    '04': ['Almería', -3.14018737, -1.6301237430000002],
    '01': ['Álava', -3.2867669470000003, -2.232687147],
    '33': ['Asturias', -7.182488914, -4.5104534990000005],
    '05': ['Ávila', -5.737561939000001, -4.160234294],
    '06': ['Badajoz', -7.335668028000001, -4.6475773910000004],
    '08': ['Barcelona', 1.3603969310000001, 2.777584332],
    '48': ['Bizkaia', -3.450329097, -2.4126037030000003],
    '09': ['Burgos', -4.335327148, -2.5173969940000003],
    '10': ['Cáceres', -7.541331761, -4.952950512],
    '11': ['Cádiz', -6.472066351770252, -5.086769376],
    '39': ['Cantabria', -4.851778269, -3.1496347360000003],
    '12': ['Castellón', -0.846296899, 0.690313419798671],
    '13': ['Ciudad Real', -5.0469484200000005, -2.6378903030000003],
    '14': ['Córdoba', -5.585923993000001, -4.000821256],
    '15': ['A Coruña', -9.301515672, -7.661721236000001],
    '16': ['Cuenca', -3.170234546, -1.14254654],
    '20': ['Gipuzkoa', -2.602719147, -1.729343469],
    '17': ['Girona', 1.724297688, 3.33255381296874],
    '18': ['Granada', -4.328132946, -2.207789069],
    '19': ['Guadalajara', -3.541098185, -1.535637879],
    '21': ['Huelva', -7.523047057, -6.121558762],
    '22': ['Huesca', -0.9341684400000001, 0.771306673],
    '23': ['Jaén', -4.287640435, -2.4349517940000003],
    '24': ['León', -7.077068851000001, -4.7338814540000005],
    '25': ['Lleida', 0.32003648, 1.8550576950000002],
    '27': ['Lugo', -7.999732198, -6.814053008],
    '28': ['Madrid', -4.579076175, -3.052983314],
    '29': ['Málaga', -5.611776730000001, -3.7659671070000003],
    '30': ['Murcia', -2.344184266, -0.6479829725699346],
    '31': ['Navarra', -2.5000827, -0.724069693],
    '32': ['Ourense', -8.36634439, -6.733953272000001],
    '34': ['Palencia', -5.031894516, -3.889622382],
    '36': ['Pontevedra', -8.94945260299778, -7.861428011],
    '26': ['La Rioja', -3.1342713900000003, -1.678701495],
    '37': ['Salamanca', -6.931705367, -5.08984113],
    '40': ['Segovia', -4.7246528450000005, -3.206929036],
    '41': ['Sevilla', -6.538514692000001, -4.653350316],
    '42': ['Soria', -3.550438343, -1.775371604],
    '43': ['Tarragona', 0.15907026600000002, 1.653151317],
    '44': ['Teruel', -1.8057277390000002, 0.293500763],
    '45': ['Toledo', -5.4061836880000005, -2.908233289],
    '46': ['Valencia', -1.528811131, -0.02479343],
    '47': ['Valladolid', -5.520890757, -3.980442109],
    '49': ['Zamora', -7.034043109000001, -5.228873192],
    '50': ['Zaragoza', -2.173671268, 0.385694968]
}

# =======================
# 4. Diagnóstico y visualización
# =======================
m = folium.Map(location=[gdf['latitud'].mean(), gdf['longitud'].mean()], zoom_start=6)
parcelas_mal_proyectadas = []

for _, row in gdf.iterrows():
    cod_prov = row['Provincia']
   
    if cod_prov in longitudes:
        lon_min, lon_max = longitudes[cod_prov][1], longitudes[cod_prov][2]
        if not (lon_min <= row['longitud'] <= lon_max):
            parcelas_mal_proyectadas.append(row)
            folium.CircleMarker(
                location=[row['latitud'], row['longitud']],
                radius=5,
                color='red',
                fill=True,
                fill_color='red',
                fill_opacity=0.6,
                popup=f"Provincia: {row['Provincia']} | Huso: {row['Huso']}"
            ).add_to(m)

# =======================
# 5. Salidas
# =======================
print(f"\n🔎 Total de parcelas: {len(gdf)}")
print(f"❌ Parcelas mal proyectadas: {len(parcelas_mal_proyectadas)}\n")

# Guardar el archivo CSV de las parcelas mal proyectadas
parcelas_mal_proyectadas_df = pd.DataFrame(parcelas_mal_proyectadas)
parcelas_mal_proyectadas_df.to_csv('IFN_España/mal_proyectadas_final.csv', index=False)


🔎 Total de parcelas: 76691
❌ Parcelas mal proyectadas: 4046



In [62]:
import pandas as pd
import os

# Cargar los archivos CSV
ubicaciones_df = pd.read_csv('IFN_España/ubicaciones.csv')
ubicaciones_df['IDParcela'] = ubicaciones_df['IDParcela'].astype(str)

# Cargar archivos opcionales de mal proyectadas
mal_ids_list = []
for mal_path in ['IFN_España/mal_proyectadas_latitud.csv', 'IFN_España/mal_proyectadas_final.csv']:
    if os.path.exists(mal_path):
        df_tmp = pd.read_csv(mal_path)
        df_tmp['IDParcela'] = df_tmp['IDParcela'].astype(str)
        mal_ids_list.append(df_tmp['IDParcela'])
    else:
        print(f"⚠️  {mal_path} no encontrado, saltando.")

# Eliminar las filas con IDParcela mal proyectadas (si hay)
if mal_ids_list:
    mal_proyectadas_ids = pd.concat(mal_ids_list)
    ubicaciones_df_cleaned = ubicaciones_df[~ubicaciones_df['IDParcela'].isin(mal_proyectadas_ids)]
else:
    ubicaciones_df_cleaned = ubicaciones_df.copy()
    print("ℹ️  No se eliminaron entradas (no hay archivos de mal proyectadas).")

# Guardar el archivo limpio
ubicaciones_df_cleaned.to_csv('IFN_España/ubicaciones.csv', index=False)
print(f"Se han eliminado {len(ubicaciones_df) - len(ubicaciones_df_cleaned)} entradas.")
print("Archivo limpio guardado como 'ubicaciones.csv'.")


Se han eliminado 6262 entradas.
Archivo limpio guardado como 'ubicaciones.csv'.


Finalmente elimino del resto de archivos de los inventarios todas las parcelas que no esten en ubicaciones.csv

In [63]:
import pandas as pd

# Cargar el archivo de ubicaciones
ubicaciones_df = pd.read_csv('IFN_España/ubicaciones.csv')

# Obtener la lista de IDParcela presentes en el archivo de ubicaciones
id_parcelas_ubicaciones = ubicaciones_df['IDParcela'].astype(str).tolist()

# Lista de archivos a procesar
archivos = [
    'IFN_España/IFN2/ifn2_DATEST.csv',
    'IFN_España/IFN2/ifn2_IIFL00BD.csv',
    'IFN_España/IFN2/ifn2_IIFL03BD.csv',
    'IFN_España/IFN2/ifn2_PIESME.csv',
    'IFN_España/IFN3_campo/ifn3_campo_Brotes.csv',
    'IFN_España/IFN3_campo/ifn3_campo_PCDatosMap.csv',
    'IFN_España/IFN3_campo/ifn3_campo_PCEspParc.csv',
    'IFN_España/IFN3_campo/ifn3_campo_PCParcelas.csv',
    'IFN_España/IFN3_campo/ifn3_campo_PCTablaEsp.csv',
    'IFN_España/IFN3_sig/ifn3_sig_CambioEspecie.csv',
    'IFN_España/IFN3_sig/ifn3_sig_Mayores_exs.csv',
    'IFN_España/IFN3_sig/ifn3_sig_Parcelas_exs.csv',
    'IFN_España/IFN4_campo/ifn4_campo_Brotes.csv',
    'IFN_España/IFN4_campo/ifn4_campo_PCDatosMap.csv',
    'IFN_España/IFN4_campo/ifn4_campo_PCEspParc.csv',
    'IFN_España/IFN4_campo/ifn4_campo_PCParcelas.csv',
    'IFN_España/IFN4_sig/ifn4_sig_CambioEspecie.csv',
    'IFN_España/IFN4_sig/ifn4_sig_Mayores_exs.csv',
    'IFN_España/IFN4_sig/ifn4_sig_Parcelas_exs.csv'
]

# Procesar cada archivo
for archivo in archivos:
    # Cargar el archivo CSV
    df = pd.read_csv(archivo)
    
    # Verificar si el archivo contiene la columna 'IDParcela'
    if 'IDParcela' in df.columns:
        # Filtrar las filas que contienen IDParcela en el archivo y que están en 'ubicaciones.csv'
        df_filtrado = df[df['IDParcela'].astype(str).isin(id_parcelas_ubicaciones)]
        
        # Guardar el archivo filtrado
        df_filtrado.to_csv(archivo, index=False)
        
        print(f"Archivo {archivo} procesado y guardado como {archivo}")
    else:
        print(f"El archivo {archivo} no contiene la columna 'IDParcela' y no ha sido modificado.")

print("Proceso completado.")


Archivo IFN_España/IFN2/ifn2_DATEST.csv procesado y guardado como IFN_España/IFN2/ifn2_DATEST.csv


Archivo IFN_España/IFN2/ifn2_IIFL00BD.csv procesado y guardado como IFN_España/IFN2/ifn2_IIFL00BD.csv


Archivo IFN_España/IFN2/ifn2_IIFL03BD.csv procesado y guardado como IFN_España/IFN2/ifn2_IIFL03BD.csv


Archivo IFN_España/IFN2/ifn2_PIESME.csv procesado y guardado como IFN_España/IFN2/ifn2_PIESME.csv


/tmp/ipykernel_2004326/2561683656.py:35: DtypeWarning: Columns (0: Especie, 1: EspecieNormalizada) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo)


Archivo IFN_España/IFN3_campo/ifn3_campo_Brotes.csv procesado y guardado como IFN_España/IFN3_campo/ifn3_campo_Brotes.csv


... (9 outputs left out)

Archivo IFN_España/IFN4_campo/ifn4_campo_PCEspParc.csv procesado y guardado como IFN_España/IFN4_campo/ifn4_campo_PCEspParc.csv


Archivo IFN_España/IFN4_campo/ifn4_campo_PCParcelas.csv procesado y guardado como IFN_España/IFN4_campo/ifn4_campo_PCParcelas.csv
El archivo IFN_España/IFN4_sig/ifn4_sig_CambioEspecie.csv no contiene la columna 'IDParcela' y no ha sido modificado.


Archivo IFN_España/IFN4_sig/ifn4_sig_Mayores_exs.csv procesado y guardado como IFN_España/IFN4_sig/ifn4_sig_Mayores_exs.csv


Archivo IFN_España/IFN4_sig/ifn4_sig_Parcelas_exs.csv procesado y guardado como IFN_España/IFN4_sig/ifn4_sig_Parcelas_exs.csv
Proceso completado.


Modifico las ubicaciones de las variables

In [64]:
import pandas as pd

# Cargar el archivo de ubicaciones
ubicaciones_df = pd.read_csv('IFN_España/ubicaciones.csv')

# Cargar los archivos a editar
archivo1 = 'IFN_España/IFN2/ifn2_DATEST.csv'
archivo2 = 'IFN_España/IFN3_campo/ifn3_campo_PCDatosMap.csv'

# Cargar los datos de los archivos
df1 = pd.read_csv(archivo1)
df2 = pd.read_csv(archivo2)

# Crear un diccionario de CoorX, CoorY, Huso usando IDParcela como clave primaria
ubicaciones_dict = ubicaciones_df.set_index('IDParcela')[['CoorX', 'CoorY', 'Huso']].to_dict('index')

# Función para actualizar CoorX, CoorY y Huso en los DataFrames
def actualizar_coordenadas_y_huso(df, ubicaciones_dict):
    # Iterar sobre las filas del DataFrame
    for idx, row in df.iterrows():
        id_parcela = row['IDParcela']
        
        # Verificar si la IDParcela está en el diccionario de ubicaciones
        if id_parcela in ubicaciones_dict:
            # Obtener los valores de CoorX, CoorY y Huso
            coor = ubicaciones_dict[id_parcela]
            
            # Sustituir CoorX, CoorY y añadir Huso
            df.at[idx, 'CoorX'] = coor['CoorX']
            df.at[idx, 'CoorY'] = coor['CoorY']
            df.at[idx, 'Huso'] = coor['Huso']
    
    return df

# Actualizar los archivos
df1_actualizado = actualizar_coordenadas_y_huso(df1, ubicaciones_dict)
df2_actualizado = actualizar_coordenadas_y_huso(df2, ubicaciones_dict)

# Guardar los archivos actualizados
df1_actualizado.to_csv('IFN_España/IFN2/ifn2_DATEST.csv', index=False)
df2_actualizado.to_csv('IFN_España/IFN3_campo/ifn3_campo_PCDatosMap.csv', index=False)

print("Los archivos han sido actualizados y guardados.")


/tmp/ipykernel_2004326/2669197280.py:12: DtypeWarning: Columns (0: Hoja50) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv(archivo2)


Los archivos han sido actualizados y guardados.


He renombrado los archivos para no liarme

In [65]:
import os
import pandas as pd

# Carpetas a recorrer
carpetas = [
    'IFN_España/IFN2',
    'IFN_España/IFN3_campo',
    'IFN_España/IFN3_sig',
    'IFN_España/IFN4_campo',
    'IFN_España/IFN4_sig'
]

# Recorrer cada carpeta
for carpeta in carpetas:
    print(f"Carpeta: {carpeta}")
    
    # Listar archivos en la carpeta
    for archivo in os.listdir(carpeta):
        # Verificar si el archivo es un .csv
        if archivo.endswith('.csv'):
            ruta_archivo = os.path.join(carpeta, archivo)
            
            # Cargar el CSV
            df = pd.read_csv(ruta_archivo)
            
            # Imprimir el nombre del archivo y las columnas
            print(f"\nArchivo: {archivo}")
            print(f"Columnas: {df.columns.tolist()}")
    
    print("-" * 50)


Carpeta: IFN_España/IFN2

Archivo: ifn2_IIFL04BD.csv
Columnas: ['VSC', 'G', 'Especie', 'Estrato', 'IAVC', 'VCC', 'COD_LINEA', 'CANT_P_MA', 'VLE', 'LIT_LINEA', 'Provincia']

Archivo: ifn2_PIESME.csv
Columnas: ['Provincia', 'Estadillo', 'NUMORDEN', 'Especie', 'NUMERO', 'ALTUMED', 'REGENA', 'IDParcela']



Archivo: ifn2_IIFL03BD.csv
Columnas: ['Provincia', 'Estadillo', 'Especie', 'CD', 'NPies', 'VCC', 'VSC', 'VLE', 'IAVC', 'ABas', 'IDParcela', 'CA', 'CR']



Archivo: ifn2_IIFL00BD.csv
Columnas: ['Provincia', 'Estadillo', 'OrdenIf2', 'Rumbo', 'Distanci', 'Especie', 'Dn1', 'Dn2', 'Ht', 'G', 'VCC', 'VSC', 'IAVC', 'VLE', 'IDParcela']


/tmp/ipykernel_2004326/3788470662.py:24: DtypeWarning: Columns (0: PARAMESP, 1: DISTANCI, 2: ALTURA) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(ruta_archivo)
/tmp/ipykernel_2004326/3788470662.py:24: DtypeWarning: Columns (0: ALTURFF, 1: DIAMCOPA2, 2: ALTUPV, 3: DIAMCOPA1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(ruta_archivo)



Archivo: ifn2_PIESMA.csv
Columnas: ['PARAMESP', 'Provincia', 'Estadillo', 'OrdenIf2', 'Especie', 'NUMORDEN', 'DIAMETRO1', 'FORMA', 'DISTANCI', 'ALTURA', 'Rumbo', 'DIAMETRO2', 'CALIDAD', 'TIPO', 'Provincia.1', 'IDParcela']

Archivo: ifn2_TIPOSX.csv
Columnas: ['Provincia', 'CRECIMI2', 'Estadillo', 'CORTEZA1', 'ALTURFF', 'CORTEZA2', 'NUMORDEN', 'DIAMCOPA2', 'CRECIMI1', 'ALTUPV', 'DIAMFF', 'DIAMETR4', 'TIPO', 'DIAMCOPA1', 'Provincia.1', 'IDParcela']

Archivo: ifn2_parcela_y_especie.csv
Columnas: ['Provincia', 'Estadillo', 'Estrato', 'CoorX', 'CoorY', 'Ano', 'INE', 'FRACCION1', 'FRACCION2', 'Especie1', 'Ocupa1', 'Estado1', 'Especie2', 'Ocupa2', 'Estado2', 'Especie3', 'Ocupa3', 'Estado3', 'ESPECIB1', 'MASAB1', 'ORIGEB1', 'EDADB1', 'ESPECIB2', 'MASAB2', 'ORIGEB2', 'EDADB2', 'ESPECIB3', 'MASAB3', 'ORIGEB3', 'EDADB3', 'CLASUELO', 'ESPESOR', 'CLACOBER', 'CUBIERTA', 'MErosiva', 'DisEsp', 'ComEsp', 'ORIENTA1', 'ORIENTA2', 'MAXPEND1', 'MAXPEND2', 'IDParcela', 'Presencia', 'Huso', 'Radio', 'Superfi


Archivo: ifn2_MATORR.csv
Columnas: ['Provincia', 'ALTUMED', 'Estadillo', 'Especie', 'NUMORDEN', 'FRACCAB', 'Provincia.1', 'IDParcela']

Archivo: ifn2_DATEST.csv
Columnas: ['Provincia', 'Estadillo', 'Estrato', 'CoorX', 'CoorY', 'Ano', 'INE', 'FRACCION1', 'FRACCION2', 'Especie1', 'Ocupa1', 'Estado1', 'Especie2', 'Ocupa2', 'Estado2', 'Especie3', 'Ocupa3', 'Estado3', 'ESPECIB1', 'MASAB1', 'ORIGEB1', 'EDADB1', 'ESPECIB2', 'MASAB2', 'ORIGEB2', 'EDADB2', 'ESPECIB3', 'MASAB3', 'ORIGEB3', 'EDADB3', 'CLASUELO', 'ESPESOR', 'CLACOBER', 'CUBIERTA', 'MErosiva', 'DisEsp', 'ComEsp', 'ORIENTA1', 'ORIENTA2', 'MAXPEND1', 'MAXPEND2', 'IDParcela', 'Presencia', 'Huso']
--------------------------------------------------
Carpeta: IFN_España/IFN3_campo

Archivo: ifn3_campo_PCDetTabla.csv
Columnas: ['CodTabla', 'DenValor', 'Valor', 'Provincia']

Archivo: ifn3_campo_PCRegenera.csv
Columnas: ['Provincia', 'Estadillo', 'Especie', 'Tipo', 'CatDes', 'Densidad', 'NumPies', 'Hm', 'IDParcela', 'EspecieNormalizada']

A

... (16 outputs left out)


Archivo: ifn4_sig_Mayores_exs.csv
Columnas: ['Provincia', 'Estadillo', 'Especie', 'EspecieOriginal', 'Estrato', 'nArbol', 'OrdenIf3', 'Rumbo', 'Distanci', 'CD', 'Ht', 'Dn1', 'Dn2', 'G', 'IAVC', 'VCC', 'VSC', 'VLE', 'CA', 'CR', 'IDParcela', 'EspecieNormalizada']

Archivo: ifn4_sig_EspDominante.csv
Columnas: ['Proceso', 'EspDom', 'IdEspDom', 'Provincia']



Archivo: ifn4_sig_Parcelas_exs.csv
Columnas: ['Provincia', 'Estadillo', 'Especie', 'Estrato', 'CD', 'VSC', 'NPies', 'ABas', 'IAVC', 'VCC', 'VLE', 'CA', 'CR', 'IDParcela', 'EspecieNormalizada']

Archivo: ifn4_sig_CambioEspecieReg.csv
Columnas: ['Proceso', 'EspCambiada', 'EspOriginal', 'NumPies', 'Provincia']

Archivo: ifn4_sig_Estratos_exs.csv
Columnas: ['Proceso', 'Estrato', 'Especie', 'CD', 'NPies', 'ABas', 'IAVC', 'VCC', 'VSC', 'VLE', 'Provincia', 'EspecieNormalizada']

Archivo: ifn4_sig_Parcpoly.csv
Columnas: ['Estrato', 'Cla', 'ForArb_Polygon', 'Altitud Absoluta', 'SubClase', 'Polygon', 'Exposicion', 'Proceso', 'Provincia', 'Pendiente Rango', 'Pendiente Absoluta', 'Altitud Rango', 'ForArb', 'Orientacion', 'Parcela']

Archivo: ifn4_sig_Tarifas_IFN.csv
Columnas: ['PARAMET', 'RR', 'CPARAM', 'CC', 'ESTR', 'BB', 'APLIC', 'SISTGEOG', 'PROVINCIA', 'AA', 'ESPECIE', 'TT', 'DD', 'NUM', 'CALIDAD', 'PP', 'SS', 'R2', 'KK', 'FORMA', 'QQ', 'R2A', 'MODELO', 'PARESP', 'PROCESO', 'CLASE', 'Provinci


Archivo: ifn4_sig_Poligon.csv
Columnas: ['N5', 'TIPOESTR', 'TipoBosque', 'ForMat', 'ForSubpiso', 'HmMat', 'SUPERFICIE', 'Proceso', 'POLYGON', 'Contenido', 'ModComb', 'Provincia', 'N2', 'RegBio', 'Esp1', 'FccHer', 'O3', 'FccArb', 'N6', 'E2', 'N1', 'Esp3', 'N3', 'E3', 'CompEsp', 'ForHer', 'Estrato', 'E1', 'ETes', 'O1', 'ForMat2', 'ForArb', 'Esp2', 'FccMat', 'O2']

Archivo: ifn4_sig_CambioEspecie.csv
Columnas: ['Provincia', 'EspCambiada', 'EspOriginal', 'NumPies']
--------------------------------------------------


/tmp/ipykernel_2004326/3788470662.py:24: DtypeWarning: Columns (0: N5, 1: TipoBosque, 2: CompEsp) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(ruta_archivo)


Añado variables Radio y Superficie para cada inventario. 

Calculo el radio como la distancia mas grande arbol/centro de cada parcela. Relleno los faltantes con la media. Superficie cirucilar en ha

In [66]:
#IFN2
import pandas as pd
import numpy as np

# Cargar los archivos
df_parcela = pd.read_csv('IFN_España/IFN2/ifn2_DATEST.csv')
df_pies_mayores = pd.read_csv('IFN_España/IFN2/ifn2_IIFL00BD.csv')  # tiene IDParcela y Distanci

# Calcular Radio como el máximo de 'Distanci' agrupado por IDParcela
df_radio = df_pies_mayores.groupby('IDParcela')['Distanci'].max().reset_index()

# Unir el cálculo de Radio con df_parcela usando 'IDParcela'
df_parcela = pd.merge(df_parcela, df_radio, on='IDParcela', how='left')

# Rellenar los valores faltantes de Radio con la media
df_parcela['Distanci'].fillna(df_parcela['Distanci'].mean(), inplace=True)

# Calcular Superficie usando la fórmula del área de un círculo (pi * radio^2), convirtiendo de m² a ha
df_parcela['Superficie'] = np.pi * (df_parcela['Distanci'] ** 2) / 1_000_000  # m² a ha

# Renombrar la columna de Distanci a Radio
df_parcela.rename(columns={'Distanci': 'Radio'}, inplace=True)

# Guardar el resultado (incluye Especie1/2/3, DisEsp, ComEsp, Radio, Superficie)
df_parcela.to_csv('IFN_España/IFN2/ifn2_parcela_y_especie.csv', index=False)
print("Archivo ifn2_parcela_y_especie.csv creado correctamente.")


Archivo ifn2_parcela_y_especie.csv creado correctamente.


/tmp/ipykernel_2004326/2738711905.py:16: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df_parcela['Distanci'].fillna(df_parcela['Distanci'].mean(), inplace=True)


In [67]:
#IFN3
import pandas as pd
import numpy as np

# Cargar los archivos
df_parcela = pd.read_csv('IFN_España/IFN3_campo/ifn3_campo_PCParcelas.csv')
df_pies_mayores = pd.read_csv('IFN_España/IFN3_sig/ifn3_sig_Mayores_exs.csv')

# Calcular Radio como el máximo de 'Distanci' agrupado por IDParcela
df_radio = df_pies_mayores.groupby('IDParcela')['Distanci'].max().reset_index()

# Unir el cálculo de Radio con df_parcela usando 'IDParcela'
df_parcela = pd.merge(df_parcela, df_radio, on='IDParcela', how='left')

# Rellenar los valores faltantes de Radio con la media
df_parcela['Distanci'].fillna(df_parcela['Distanci'].mean(), inplace=True)

# Calcular Superficie usando la fórmula del área de un círculo (pi * radio^2), convirtiendo de m² a ha
df_parcela['Superficie'] = np.pi * (df_parcela['Distanci'] ** 2) / 1_000_000  # m² a ha

# Renombrar la columna de Distanci a Radio
df_parcela.rename(columns={'Distanci': 'Radio'}, inplace=True)

# Guardar el resultado en un nuevo archivo CSV
df_parcela.to_csv('IFN_España/IFN3_campo/ifn3_campo_parcela.csv', index=False)
print("Archivo ifn3_campo_parcela.csv creado correctamente.")


/tmp/ipykernel_2004326/1200830149.py:6: DtypeWarning: Columns (0: Nivel2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_parcela = pd.read_csv('IFN_España/IFN3_campo/ifn3_campo_PCParcelas.csv')


/tmp/ipykernel_2004326/1200830149.py:16: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df_parcela['Distanci'].fillna(df_parcela['Distanci'].mean(), inplace=True)


Archivo ifn3_campo_parcela.csv creado correctamente.


In [68]:
#IFN4
import pandas as pd
import numpy as np

# Cargar los archivos
df_parcela = pd.read_csv('IFN_España/IFN4_campo/ifn4_campo_PCParcelas.csv')
df_pies_mayores = pd.read_csv('IFN_España/IFN4_sig/ifn4_sig_Mayores_exs.csv')

# Calcular Radio como el máximo de 'Distanci' agrupado por IDParcela
df_radio = df_pies_mayores.groupby('IDParcela')['Distanci'].max().reset_index()

# Unir el cálculo de Radio con df_parcela usando 'IDParcela'
df_parcela = pd.merge(df_parcela, df_radio, on='IDParcela', how='left')

# Rellenar los valores faltantes de Radio con la media
df_parcela['Distanci'].fillna(df_parcela['Distanci'].mean(), inplace=True)

# Calcular Superficie usando la fórmula del área de un círculo (pi * radio^2), convirtiendo de m² a ha
df_parcela['Superficie'] = np.pi * (df_parcela['Distanci'] ** 2) / 1_000_000  # m² a ha

# Renombrar la columna de Distanci a Radio
df_parcela.rename(columns={'Distanci': 'Radio'}, inplace=True)

# Guardar el resultado en un nuevo archivo CSV
df_parcela.to_csv('IFN_España/IFN4_campo/ifn4_campo_parcela.csv', index=False)
print("Archivo ifn4_campo_parcela.csv creado correctamente.")


/tmp/ipykernel_2004326/36561859.py:16: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df_parcela['Distanci'].fillna(df_parcela['Distanci'].mean(), inplace=True)


Archivo ifn4_campo_parcela.csv creado correctamente.
